# 03 — Stage C: bridge calibration, single-sample networks and eNESH (Sections 3.4–3.6)

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### Bridge check -- do the 60 independent Stage-A TCGA tumors have matching

In [ ]:
"""
Bridge check -- do the 60 independent Stage-A TCGA tumors have matching
GDC STAR RNA-seq gene expression files?

This does NOT touch the 144 survival patients or any survival labels.
Answers exactly one question: is Option 4 (paired bridge calibration)
actually feasible given real data availability.
"""

import requests
import pandas as pd
from pathlib import Path

DOWNLOADS = BASE
STAGE_A_TUMOR_FILE = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY" / "STAGE_A_TCGA_GBM_INDEPENDENT_TUMORS.csv"

OUTFILE = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY" / "BRIDGE_CHECK_GDC_AVAILABILITY.csv"

# ------------------------------------------------------------
# 1. Load the 60 Stage-A bridge candidate patients
# ------------------------------------------------------------
stage_a = pd.read_csv(STAGE_A_TUMOR_FILE)
print(f"Stage-A independent tumors: {len(stage_a)}")
print(f"Columns: {stage_a.columns.tolist()}")

bridge_patient_ids = sorted(stage_a["Patient ID"].dropna().astype(str).unique())
print(f"Unique bridge candidate patients: {len(bridge_patient_ids)}")
print(f"Examples: {bridge_patient_ids[:5]}")

# ------------------------------------------------------------
# 2. Query GDC for matching STAR-Counts RNA-seq gene expression files
# ------------------------------------------------------------
url = "https://api.gdc.cancer.gov/files"
payload = {
    "filters": {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.submitter_id", "value": bridge_patient_ids}},
            {"op": "in", "content": {"field": "data_type", "value": ["Gene Expression Quantification"]}},
            {"op": "in", "content": {"field": "experimental_strategy", "value": ["RNA-Seq"]}},
        ],
    },
    "fields": "file_id,file_name,cases.submitter_id,analysis.workflow_type,state",
    "format": "JSON",
    "size": "1000",
}

print("\nQuerying GDC API...")
r = requests.post(url, json=payload, timeout=60)
r.raise_for_status()
hits = r.json()["data"]["hits"]
print(f"GDC returned {len(hits)} matching files (all data types/workflows before filtering)")

rows = []
for h in hits:
    cases = h.get("cases", [{}])
    patient_id = cases[0].get("submitter_id") if cases else None
    rows.append({
        "file_id": h["file_id"],
        "file_name": h.get("file_name"),
        "patient_id": patient_id,
        "workflow_type": h.get("analysis", {}).get("workflow_type"),
        "state": h.get("state"),
    })

gdc_files = pd.DataFrame(rows)
print(f"\nWorkflow types found:")
print(gdc_files["workflow_type"].value_counts())

# restrict specifically to STAR - Counts, matching what the 144 patients use
star_files = gdc_files[gdc_files["workflow_type"] == "STAR - Counts"].copy()
print(f"\nSTAR - Counts files: {len(star_files)}")

matched_patients = sorted(set(star_files["patient_id"]) & set(bridge_patient_ids))
missing_patients = sorted(set(bridge_patient_ids) - set(star_files["patient_id"]))

print(f"\n{'='*60}")
print(f"BRIDGE FEASIBILITY RESULT")
print(f"{'='*60}")
print(f"Bridge candidates with a matching GDC STAR-Counts file: {len(matched_patients)}/{len(bridge_patient_ids)}")
print(f"Missing: {len(missing_patients)}")
if missing_patients:
    print(f"Missing patient IDs: {missing_patients}")

# duplicate check -- multiple STAR files per patient?
dup_check = star_files["patient_id"].value_counts()
n_dup = (dup_check > 1).sum()
if n_dup:
    print(f"\nNOTE: {n_dup} bridge patients have >1 STAR-Counts file "
          f"(multiple RNA aliquots, same situation as the original 167->144 "
          f"dedup) -- will need the same selection logic before calibration.")

star_files.to_csv(OUTFILE, index=False)
print(f"\nSaved -> {OUTFILE}")

print(f"\n{'='*60}")
if len(matched_patients) >= 45:
    print(f"RESULT: {len(matched_patients)} bridge pairs is a workable n for")
    print(f"gene-wise calibration with shrinkage. Proceed to Option 4.")
elif len(matched_patients) >= 20:
    print(f"RESULT: {len(matched_patients)} bridge pairs is workable but thin --")
    print(f"shrinkage toward (a=0, b=1) for low-evidence genes becomes important,")
    print(f"not optional. Proceed with caution.")
else:
    print(f"RESULT: {len(matched_patients)} bridge pairs is too few for reliable")
    print(f"gene-wise calibration. Reconsider Option 1-3, or look for additional")
    print(f"bridge candidates beyond the Stage-A 60.")

### BRIDGE SAMPLE-LEVEL PAIRING CHECK

In [ ]:
"""
BRIDGE SAMPLE-LEVEL PAIRING CHECK
=================================

Goal:
Verify whether the 60 independent Stage-A TCGA-GBM tumors have the
SAME biological sample represented in both:

    recount3
    and
    GDC STAR-Counts

This step DOES NOT:
- fit calibration
- touch survival labels
- touch the 144 survival cohort

It only determines which bridge patients are valid true paired samples.

Output:
    exact_sample_match
    patient_only_match
    unresolved

IMPORTANT:
Patient-level agreement alone is NOT enough for calibration.
We require agreement at least through TCGA sample type, and vial letter
when both resources expose it.
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path

import pandas as pd
import requests


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_A_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_A_DISCOVERY"
)

STAGE_A_TUMOR_FILE = (
    STAGE_A_DIR
    / "STAGE_A_TCGA_GBM_INDEPENDENT_TUMORS.csv"
)

OUTFILE = (
    STAGE_A_DIR
    / "BRIDGE_SAMPLE_LEVEL_PAIRING.csv"
)


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)


def checkpoint(condition, message):

    status = "PASS" if condition else "FAIL"

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise AssertionError(
            f"Checkpoint failed: {message}"
        )


def tcga_sample_level_match(
    recount_id,
    gdc_sample_id
):
    """
    Match TCGA barcodes at biological SAMPLE level.

    Accept:

        TCGA-02-2483-01A-01R-....
        TCGA-02-2483-01A

    Accept if one barcode is more detailed than the other,
    provided they agree through the available sample-level fields.

    Reject:

        TCGA-02-2483
        TCGA-02-2483-01A

    because the first barcode contains patient-level information only.

    Reject:

        TCGA-02-2483-01A
        TCGA-02-2483-01B

    because the vial differs.

    Reject:

        TCGA-02-2483-01A
        TCGA-02-2483-02A

    because sample type differs.
    """

    if pd.isna(recount_id):
        return False

    if pd.isna(gdc_sample_id):
        return False


    recount_id = str(
        recount_id
    ).strip()

    gdc_sample_id = str(
        gdc_sample_id
    ).strip()


    if recount_id == "":
        return False

    if gdc_sample_id == "":
        return False


    r_parts = recount_id.split("-")
    g_parts = gdc_sample_id.split("-")


    # ----------------------------------------------------------
    # Need at least:
    #
    # TCGA - XX - XXXX - 01...
    #
    # Otherwise recount3 ID is only patient-level and cannot
    # establish an exact biological sample.
    # ----------------------------------------------------------

    if len(r_parts) < 4:
        return False

    if len(g_parts) < 4:
        return False


    # ----------------------------------------------------------
    # Patient barcode must agree:
    #
    # TCGA-XX-XXXX
    # ----------------------------------------------------------

    if r_parts[:3] != g_parts[:3]:
        return False


    # ----------------------------------------------------------
    # Sample-type digits
    #
    # 01 = primary tumor
    # 02 = recurrent tumor etc.
    #
    # If code is 01A, compare 01 first.
    # ----------------------------------------------------------

    r_sample_field = r_parts[3]
    g_sample_field = g_parts[3]


    if len(r_sample_field) < 2:
        return False

    if len(g_sample_field) < 2:
        return False


    r_sample_type = r_sample_field[:2]
    g_sample_type = g_sample_field[:2]


    if r_sample_type != g_sample_type:
        return False


    # ----------------------------------------------------------
    # Vial letter
    #
    # If BOTH sides expose vial information,
    # require agreement.
    #
    # Example:
    # 01A vs 01B -> reject
    #
    # If one side exposes only 01 while the other exposes 01A,
    # accept because they agree at their finest common granularity.
    # ----------------------------------------------------------

    r_has_vial = (
        len(r_sample_field) >= 3
        and r_sample_field[2].isalpha()
    )

    g_has_vial = (
        len(g_sample_field) >= 3
        and g_sample_field[2].isalpha()
    )


    if r_has_vial and g_has_vial:

        if (
            r_sample_field[2].upper()
            !=
            g_sample_field[2].upper()
        ):
            return False


    return True


# ======================================================================
# STEP 1 — LOAD RECOUNT3 BRIDGE SAMPLE IDS
# ======================================================================

section(
    "STEP 1 — recount3 sample ID format"
)


bridge_tumors = pd.read_csv(
    STAGE_A_TUMOR_FILE,
    low_memory=False
)


required_columns = [
    "Patient ID",
    "Recount sample ID"
]


missing_columns = [
    c
    for c in required_columns
    if c not in bridge_tumors.columns
]


if missing_columns:

    raise ValueError(
        "Missing required columns in Stage-A tumor file:\n"
        f"{missing_columns}\n\n"
        f"Available columns:\n"
        f"{bridge_tumors.columns.tolist()}"
    )


print(
    "Stage-A rows:",
    len(bridge_tumors)
)


print(
    "Unique Stage-A patients:",
    bridge_tumors["Patient ID"].nunique()
)


checkpoint(
    bridge_tumors["Patient ID"].nunique() == 60,
    "Stage-A bridge cohort contains exactly 60 patients"
)


# ------------------------------------------------------------
# Ensure one recount3 sample ID per patient
# ------------------------------------------------------------

recount_counts = (
    bridge_tumors
    .groupby("Patient ID")["Recount sample ID"]
    .nunique()
)


bad_recount_pairs = recount_counts[
    recount_counts != 1
]


checkpoint(
    len(bad_recount_pairs) == 0,
    "each bridge patient has exactly one frozen recount3 sample ID"
)


recount3_ids = (
    bridge_tumors
    .drop_duplicates(
        subset="Patient ID"
    )
    .set_index("Patient ID")["Recount sample ID"]
)


print(
    "\nFirst 10 recount3 sample IDs:"
)


for pid, rid in list(
    recount3_ids.items()
)[:10]:

    print(
        f"  {pid}"
        f" -> {rid!r}"
        f"  length={len(str(rid))}"
    )


length_distribution = (
    recount3_ids
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)


print(
    "\nrecount3 sample ID length distribution:"
)


print(
    length_distribution.to_dict()
)


# ------------------------------------------------------------
# Verify that IDs contain at least sample-level granularity
# ------------------------------------------------------------

sample_level_flags = (
    recount3_ids
    .astype(str)
    .apply(
        lambda x:
        len(x.split("-")) >= 4
    )
)


print(
    "\nrecount3 IDs with sample-level granularity:",
    int(sample_level_flags.sum()),
    "/",
    len(sample_level_flags)
)


if not sample_level_flags.all():

    print(
        "\nWARNING:"
    )

    print(
        "Some recount3 IDs do not expose sample-level granularity."
    )

    print(
        "Those patients cannot be called exact sample matches."
    )


# ======================================================================
# STEP 2 — QUERY GDC SAMPLE + ALIQUOT BARCODES
# ======================================================================

section(
    "STEP 2 — Query GDC sample/aliquot identifiers"
)


bridge_patient_ids = sorted(
    recount3_ids
    .index
    .astype(str)
    .unique()
)


url = (
    "https://api.gdc.cancer.gov/files"
)


payload = {

    "filters": {

        "op": "and",

        "content": [

            {
                "op": "in",
                "content": {
                    "field": "cases.submitter_id",
                    "value": bridge_patient_ids
                }
            },

            {
                "op": "in",
                "content": {
                    "field": "data_type",
                    "value": [
                        "Gene Expression Quantification"
                    ]
                }
            },

            {
                "op": "in",
                "content": {
                    "field": "experimental_strategy",
                    "value": [
                        "RNA-Seq"
                    ]
                }
            }
        ]
    },

    "fields": (
        "file_id,"
        "file_name,"
        "cases.submitter_id,"
        "cases.samples.submitter_id,"
        "cases.samples.sample_type,"
        "cases.samples.portions.submitter_id,"
        "cases.samples.portions.analytes.submitter_id,"
        "cases.samples.portions.analytes.aliquots.submitter_id,"
        "analysis.workflow_type,"
        "state"
    ),

    "format": "JSON",

    "size": "1000"
}


print(
    "Querying GDC API..."
)


response = requests.post(
    url,
    json=payload,
    timeout=120
)


response.raise_for_status()


hits = (
    response
    .json()["data"]["hits"]
)


print(
    "Total files returned:",
    len(hits)
)


# ======================================================================
# PARSE GDC RESPONSE
# ======================================================================

rows = []


for hit in hits:

    workflow_type = (
        hit
        .get(
            "analysis",
            {}
        )
        .get(
            "workflow_type"
        )
    )


    # only GDC STAR - Counts
    if workflow_type != "STAR - Counts":
        continue


    cases = hit.get(
        "cases",
        []
    )


    for case in cases:

        patient_id = (
            case.get(
                "submitter_id"
            )
        )


        samples = case.get(
            "samples",
            []
        )


        # In the unlikely event sample metadata are absent,
        # still record the file for transparency.
        if not samples:

            rows.append(
                {
                    "file_id":
                        hit.get("file_id"),

                    "file_name":
                        hit.get("file_name"),

                    "patient_id":
                        patient_id,

                    "gdc_sample_barcode":
                        None,

                    "gdc_sample_type":
                        None,

                    "gdc_portion_barcodes":
                        None,

                    "gdc_analyte_barcodes":
                        None,

                    "gdc_aliquot_barcodes":
                        None
                }
            )

            continue


        for sample in samples:

            sample_barcode = (
                sample.get(
                    "submitter_id"
                )
            )


            sample_type = (
                sample.get(
                    "sample_type"
                )
            )


            portion_barcodes = []

            analyte_barcodes = []

            aliquot_barcodes = []


            portions = sample.get(
                "portions",
                []
            )


            for portion in portions:

                portion_id = (
                    portion.get(
                        "submitter_id"
                    )
                )


                if portion_id:

                    portion_barcodes.append(
                        portion_id
                    )


                analytes = portion.get(
                    "analytes",
                    []
                )


                for analyte in analytes:

                    analyte_id = (
                        analyte.get(
                            "submitter_id"
                        )
                    )


                    if analyte_id:

                        analyte_barcodes.append(
                            analyte_id
                        )


                    aliquots = analyte.get(
                        "aliquots",
                        []
                    )


                    for aliquot in aliquots:

                        aliquot_id = (
                            aliquot.get(
                                "submitter_id"
                            )
                        )


                        if aliquot_id:

                            aliquot_barcodes.append(
                                aliquot_id
                            )


            rows.append(
                {
                    "file_id":
                        hit.get("file_id"),

                    "file_name":
                        hit.get("file_name"),

                    "patient_id":
                        patient_id,

                    "gdc_sample_barcode":
                        sample_barcode,

                    "gdc_sample_type":
                        sample_type,

                    "gdc_portion_barcodes":
                        ";".join(
                            sorted(
                                set(
                                    portion_barcodes
                                )
                            )
                        )
                        if portion_barcodes
                        else None,

                    "gdc_analyte_barcodes":
                        ";".join(
                            sorted(
                                set(
                                    analyte_barcodes
                                )
                            )
                        )
                        if analyte_barcodes
                        else None,

                    "gdc_aliquot_barcodes":
                        ";".join(
                            sorted(
                                set(
                                    aliquot_barcodes
                                )
                            )
                        )
                        if aliquot_barcodes
                        else None
                }
            )


gdc_samples = pd.DataFrame(
    rows
)


print(
    "\nSTAR-Counts sample records:",
    len(gdc_samples)
)


print(
    "Unique GDC patients represented:",
    gdc_samples[
        "patient_id"
    ].nunique()
)


checkpoint(
    gdc_samples[
        "patient_id"
    ].nunique() == 60,
    "all 60 bridge patients appear in the GDC STAR-Counts query"
)


print(
    "\nFirst 10 GDC sample records:"
)


preview_columns = [
    "patient_id",
    "gdc_sample_barcode",
    "gdc_sample_type",
    "file_name"
]


print(
    gdc_samples[
        preview_columns
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ======================================================================
# STEP 3 — SAMPLE-LEVEL MATCHING
# ======================================================================

section(
    "STEP 3 — Compare recount3 vs GDC sample barcodes"
)


results = []


for patient_id, recount3_sample in (
    recount3_ids.items()
):


    candidates = (
        gdc_samples[
            gdc_samples[
                "patient_id"
            ]
            ==
            patient_id
        ]
        .copy()
    )


    # ----------------------------------------------------------
    # No GDC candidate
    # ----------------------------------------------------------

    if len(candidates) == 0:

        results.append(
            {
                "patient_id":
                    patient_id,

                "recount3_sample":
                    recount3_sample,

                "status":
                    "unresolved",

                "n_gdc_candidate_records":
                    0,

                "n_exact_sample_records":
                    0,

                "matched_file_ids":
                    None,

                "matched_file_names":
                    None,

                "matched_gdc_sample_barcodes":
                    None,

                "matched_aliquot_barcodes":
                    None
            }
        )

        continue


    # ----------------------------------------------------------
    # Exact biological sample match
    # ----------------------------------------------------------

    exact_mask = (
        candidates[
            "gdc_sample_barcode"
        ]
        .apply(
            lambda gdc_id:
            tcga_sample_level_match(
                recount3_sample,
                gdc_id
            )
        )
    )


    exact_matches = (
        candidates[
            exact_mask
        ]
        .copy()
    )


    if len(exact_matches) > 0:

        results.append(
            {
                "patient_id":
                    patient_id,

                "recount3_sample":
                    recount3_sample,

                "status":
                    "exact_sample_match",

                "n_gdc_candidate_records":
                    len(candidates),

                "n_exact_sample_records":
                    len(exact_matches),

                "matched_file_ids":
                    ";".join(
                        sorted(
                            exact_matches[
                                "file_id"
                            ]
                            .dropna()
                            .astype(str)
                            .unique()
                        )
                    ),

                "matched_file_names":
                    ";".join(
                        sorted(
                            exact_matches[
                                "file_name"
                            ]
                            .dropna()
                            .astype(str)
                            .unique()
                        )
                    ),

                "matched_gdc_sample_barcodes":
                    ";".join(
                        sorted(
                            exact_matches[
                                "gdc_sample_barcode"
                            ]
                            .dropna()
                            .astype(str)
                            .unique()
                        )
                    ),

                "matched_aliquot_barcodes":
                    ";".join(
                        sorted(
                            exact_matches[
                                "gdc_aliquot_barcodes"
                            ]
                            .dropna()
                            .astype(str)
                            .unique()
                        )
                    )
                    if (
                        exact_matches[
                            "gdc_aliquot_barcodes"
                        ]
                        .notna()
                        .any()
                    )
                    else None
            }
        )


    # ----------------------------------------------------------
    # Patient exists in GDC, but sample barcode does not match
    # ----------------------------------------------------------

    else:

        results.append(
            {
                "patient_id":
                    patient_id,

                "recount3_sample":
                    recount3_sample,

                "status":
                    "patient_only_match",

                "n_gdc_candidate_records":
                    len(candidates),

                "n_exact_sample_records":
                    0,

                "matched_file_ids":
                    None,

                "matched_file_names":
                    None,

                "matched_gdc_sample_barcodes":
                    None,

                "matched_aliquot_barcodes":
                    None
            }
        )


# ======================================================================
# BUILD RESULT TABLE
# ======================================================================

pairing_df = pd.DataFrame(
    results
)


checkpoint(
    len(pairing_df) == 60,
    "one pairing result exists for each of the 60 bridge patients"
)


checkpoint(
    pairing_df[
        "patient_id"
    ].is_unique,
    "every bridge patient appears exactly once in the final pairing table"
)


# ======================================================================
# SAVE
# ======================================================================

pairing_df.to_csv(
    OUTFILE,
    index=False
)


# ======================================================================
# FINAL SUMMARY
# ======================================================================

section(
    "SAMPLE-LEVEL PAIRING RESULT"
)


status_counts = (
    pairing_df[
        "status"
    ]
    .value_counts()
)


for status in [
    "exact_sample_match",
    "patient_only_match",
    "unresolved"
]:

    print(
        f"{status}:",
        int(
            status_counts.get(
                status,
                0
            )
        )
    )


n_exact = int(
    status_counts.get(
        "exact_sample_match",
        0
    )
)


n_patient_only = int(
    status_counts.get(
        "patient_only_match",
        0
    )
)


n_unresolved = int(
    status_counts.get(
        "unresolved",
        0
    )
)


print(
    "\nExact sample match rate:",
    f"{n_exact}/60",
    f"({100*n_exact/60:.1f}%)"
)


# ======================================================================
# Exact sample pairs with more than one GDC file
# ======================================================================

multi_exact = (
    pairing_df[
        (
            pairing_df[
                "status"
            ]
            ==
            "exact_sample_match"
        )
        &
        (
            pairing_df[
                "n_exact_sample_records"
            ]
            >
            1
        )
    ]
)


print(
    "\nExact sample matches with >1 GDC file/record:",
    len(multi_exact)
)


if len(multi_exact) > 0:

    print(
        "\nThese require QC tie-breaking "
        "only among files corresponding "
        "to the SAME biological sample:"
    )


    print(
        multi_exact[
            [
                "patient_id",
                "recount3_sample",
                "n_exact_sample_records",
                "matched_gdc_sample_barcodes",
                "matched_file_ids"
            ]
        ]
        .to_string(
            index=False
        )
    )


# ======================================================================
# DECISION
# ======================================================================

section(
    "BRIDGE CALIBRATION DECISION"
)


if n_exact >= 50:

    print(
        f"RESULT: {n_exact}/60 exact biological sample matches."
    )

    print(
        "This is a strong paired bridge cohort."
    )

    print(
        "Proceed to calibration using ONLY these exact sample pairs."
    )


elif n_exact >= 30:

    print(
        f"RESULT: {n_exact}/60 exact biological sample matches."
    )

    print(
        "This is workable, but calibration must use exactly "
        f"these {n_exact} pairs."
    )

    print(
        "Do NOT add patient-only matches simply to increase sample size."
    )


else:

    print(
        f"RESULT: only {n_exact}/60 exact biological sample matches."
    )

    print(
        "The paired-bridge strategy is substantially weakened."
    )

    print(
        "Do not fit the calibration yet; reconsider the harmonization strategy."
    )


print(
    "\nSaved:"
)

print(
    OUTFILE
)

### STAGE C0 PART 1 — EXACT-PAIR BRIDGE CALIBRATION MATRICES

In [ ]:
"""
STAGE C0 PART 1 — EXACT-PAIR BRIDGE CALIBRATION MATRICES
=======================================================

Goal
----
Build TWO ALIGNED expression matrices for the SAME 60 exact biological
TCGA-GBM samples:

    1) GDC STAR-Counts side
    2) recount3 / Monorail side

Both are converted to:
    log2(CPM + 1)

These matrices are ONLY for platform calibration.

They are NOT:
- survival modeling data
- biomarker evaluation data
- the 144 survival cohort

Key safeguards
--------------
1. Use ONLY patients classified as exact_sample_match.
2. For multi-file GDC patients, choose only among files belonging to
   the exact biological sample.
3. recount3 sample barcode is resolved through recount3 metadata to the
   REAL gene_sums expression column.
4. recount3 annotation/comment lines are skipped correctly.
5. star.average_mapped_length is selected explicitly.
6. Ensembl version suffixes are stripped before gene intersection.
7. Both sides are normalized to log2(CPM+1).

CHANGE FROM PREVIOUS VERSION: Step 2 (GDC download) is now batched,
retried, and validated (gzip magic bytes + Content-Length check) BEFORE
attempting extraction, instead of downloading all 87 files in one shot
and finding out only when tarfile/gzip crashes with a bare EOFError.
Everything else is unchanged from the version that already ran cleanly
through Step 1.
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path
import gzip
import io
import re
import tarfile

import numpy as np
import pandas as pd
import requests


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_A_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_A_DISCOVERY"
)

OUTDIR = (
    STAGE_A_DIR
    / "STAGE_C0_BRIDGE_CALIBRATION"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True
)


PAIRING_FILE = (
    STAGE_A_DIR
    / "BRIDGE_SAMPLE_LEVEL_PAIRING.csv"
)

STAGE_A_TUMOR_FILE = (
    STAGE_A_DIR
    / "STAGE_A_TCGA_GBM_INDEPENDENT_TUMORS.csv"
)


# Safe CSV caches
RECOUNT3_META_CACHE = (
    OUTDIR
    / "tcga_gbm_recount3_metadata.csv"
)

RECOUNT3_QC_CACHE = (
    OUTDIR
    / "tcga_gbm_recount3_qc.csv"
)

RECOUNT3_GENE_SUMS_CACHE = (
    OUTDIR
    / "tcga_gbm_gene_sums.csv"
)


BASE = (
    "http://duffel.rail.bio/"
    "recount3/human/data_sources"
)

TCGA_PROJECT = "GBM"

# Step 2 download settings (new)
BATCH_SIZE = 15
MAX_RETRIES = 3


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def checkpoint(condition, message):

    status = "PASS" if condition else "FAIL"

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise AssertionError(
            f"Checkpoint failed: {message}"
        )


def project_shard(project):

    return project[-2:]


def strip_version(x):

    return (
        str(x)
        .split(".")[0]
        .strip()
    )


def clean_string(x):

    if pd.isna(x):
        return None

    x = str(x).strip()

    if x == "":
        return None

    if x.lower() == "nan":
        return None

    return x


# ======================================================================
# RECOUNT3 PARSER
# skips ##annotation=... comment lines
# ======================================================================

def load_recount3_gz_bytes(raw_bytes):

    text = gzip.decompress(
        raw_bytes
    ).decode(
        "utf-8"
    )

    lines = text.splitlines(
        keepends=True
    )

    data_start = None

    for i, line in enumerate(lines):

        if not line.startswith("#"):

            data_start = i
            break


    if data_start is None:

        raise RuntimeError(
            "Could not find real recount3 table header."
        )


    return pd.read_csv(
        io.StringIO(
            "".join(
                lines[data_start:]
            )
        ),
        sep="\t",
        low_memory=False
    )


def fetch_recount3_table(url, timeout=300):

    print("Downloading:")
    print(url)

    r = requests.get(
        url,
        timeout=timeout
    )

    r.raise_for_status()

    return load_recount3_gz_bytes(
        r.content
    )


# ======================================================================
# SELECT CORRECT AVERAGE MAPPED LENGTH COLUMN
# ======================================================================

def select_avg_mapped_length_column(columns):

    preferred = [
        "recount_qc.star.average_mapped_length",
        "star.average_mapped_length"
    ]


    for col in preferred:

        if col in columns:

            return col


    candidates = [
        c for c in columns
        if (
            "average_mapped_length"
            in c.lower()
            and
            "length2"
            not in c.lower()
        )
    ]


    if len(candidates) == 1:

        return candidates[0]


    raise RuntimeError(
        "Could not uniquely identify "
        "star.average_mapped_length.\n"
        f"Candidates: {candidates}"
    )


# ======================================================================
# GDC STAR FILE PARSER
# ======================================================================

def parse_star_count_file(path):

    opener = (
        gzip.open
        if str(path).endswith(".gz")
        else open
    )


    with opener(
        path,
        "rt"
    ) as f:

        lines = f.readlines()


    data_start = None

    for i, line in enumerate(lines):

        first_field = (
            line
            .split("\t")[0]
            .strip()
        )


        if re.match(
            r"^ENSG\d+",
            first_field
        ):

            data_start = i
            break


    if data_start is None:

        raise ValueError(
            f"No ENSG gene rows found in:\n{path}"
        )


    df = pd.read_csv(
        io.StringIO(
            "".join(
                lines[data_start:]
            )
        ),
        sep="\t",
        header=None,
        names=[
            "gene_id",
            "gene_name",
            "gene_type",
            "unstranded",
            "stranded_first",
            "stranded_second",
            "tpm_unstranded",
            "fpkm_unstranded",
            "fpkm_uq_unstranded"
        ]
    )


    return df


# ======================================================================
# LOG2 CPM
# ======================================================================

def log2_cpm(matrix):

    matrix = matrix.apply(
        pd.to_numeric,
        errors="coerce"
    )


    checkpoint(
        not matrix.isna().all().all(),
        "count matrix contains usable numeric values"
    )


    lib_size = matrix.sum(
        axis=0
    )


    checkpoint(
        (lib_size > 0).all(),
        "all samples have positive library size"
    )


    cpm = (
        matrix
        .div(
            lib_size,
            axis=1
        )
        *
        1e6
    )


    return np.log2(
        cpm + 1
    )


# ======================================================================
# STEP 2 DOWNLOAD HELPERS (new -- batched, retried, validated)
# ======================================================================

def validate_gdc_response(content, content_length_header, n_ids_in_batch):
    """Check BEFORE attempting extraction, not after it crashes."""
    if n_ids_in_batch == 1:
        # GDC returns the RAW file directly (not a tar) for a single ID
        return True, "single-file response, not a tar"
    if len(content) < 2 or content[:2] != b"\x1f\x8b":
        return False, (f"Response is not gzip data (first 20 bytes: {content[:20]!r}) "
                        f"-- likely an error page or JSON from GDC, not the archive")
    if content_length_header is not None and len(content) != int(content_length_header):
        return False, (f"Truncated download: got {len(content)} bytes, "
                        f"server said Content-Length={content_length_header}")
    return True, "OK"


def download_batch_with_retry(batch, batch_num, n_batches, download_dir):
    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  Batch {batch_num}/{n_batches} ({len(batch)} files), attempt {attempt}/{MAX_RETRIES}...")
        try:
            resp = requests.post(
                "https://api.gdc.cancer.gov/data",
                json={"ids": batch},
                headers={"Content-Type": "application/json"},
                timeout=300,
            )
            resp.raise_for_status()
            content = resp.content
            content_length = resp.headers.get("Content-Length")

            ok, msg = validate_gdc_response(content, content_length, len(batch))
            if not ok:
                raise IOError(msg)

            if len(batch) == 1:
                fname = resp.headers.get("Content-Disposition", "")
                out_name = fname.split("filename=")[-1].strip('"; ') if "filename=" in fname else f"{batch[0]}.tsv"
                out_path = download_dir / batch[0]
                out_path.mkdir(exist_ok=True)
                with open(out_path / out_name, "wb") as f:
                    f.write(content)
            else:
                batch_tar_path = download_dir / f"batch_{batch_num}.tar.gz"
                with open(batch_tar_path, "wb") as f:
                    f.write(content)
                with tarfile.open(batch_tar_path, "r:gz") as tar:
                    tar.extractall(download_dir)

            print(f"    OK -- {len(content)/1e6:.2f} MB")
            return True

        except Exception as e:
            print(f"    Attempt {attempt} failed: {e}")
            if attempt == MAX_RETRIES:
                print(f"    Batch {batch_num} FAILED after {MAX_RETRIES} attempts.")
                print(f"    file_ids in this failed batch: {batch}")
                return False
    return False


# ======================================================================
# STEP 1 — LOAD EXACT PAIRING
# ======================================================================

section(
    "STEP 1 — Load exact sample-level bridge pairs"
)


pairing = pd.read_csv(
    PAIRING_FILE,
    low_memory=False
)


exact = (
    pairing[
        pairing["status"]
        ==
        "exact_sample_match"
    ]
    .copy()
)


checkpoint(
    len(exact) == 60,
    f"exactly 60 exact sample matches found ({len(exact)})"
)


# ------------------------------------------------------------
# IMPORTANT:
# pairing output stores matched_file_ids like:
#
# id1;id2
#
# not a Python list.
# ------------------------------------------------------------

if "matched_file_ids" not in exact.columns:

    raise RuntimeError(
        "Expected column 'matched_file_ids' "
        "not found in pairing file.\n"
        f"Columns: {exact.columns.tolist()}"
    )


def parse_file_ids(x):

    if pd.isna(x):

        return []

    return [
        item.strip()
        for item in str(x).split(";")
        if item.strip()
    ]


exact[
    "matched_file_id_list"
] = (
    exact[
        "matched_file_ids"
    ]
    .apply(
        parse_file_ids
    )
)


exact[
    "n_matched_files"
] = (
    exact[
        "matched_file_id_list"
    ]
    .apply(
        len
    )
)


checkpoint(
    (
        exact[
            "n_matched_files"
        ]
        >= 1
    ).all(),
    "every exact bridge sample has at least one eligible GDC file"
)


n_multi = int(
    (
        exact[
            "n_matched_files"
        ]
        >
        1
    ).sum()
)


print(
    "Single exact-match file:",
    int(
        (
            exact[
                "n_matched_files"
            ]
            ==
            1
        ).sum()
    )
)


print(
    "Multiple exact-match files:",
    n_multi
)


checkpoint(
    n_multi == 27,
    f"27 expected multi-file samples, found {n_multi}"
)


# ======================================================================
# STEP 2 — DOWNLOAD ONLY EXACT-MATCH GDC FILES (batched + validated)
# ======================================================================

section(
    "STEP 2 — Download exact-match GDC STAR files"
)


all_candidate_file_ids = sorted(
    {
        fid
        for file_list
        in exact[
            "matched_file_id_list"
        ]
        for fid in file_list
    }
)


print(
    "Exact-match candidate files:",
    len(all_candidate_file_ids)
)


download_dir = (
    OUTDIR
    / "gdc_exact_match_files"
)

download_dir.mkdir(
    exist_ok=True
)


batches = [
    all_candidate_file_ids[i:i + BATCH_SIZE]
    for i in range(0, len(all_candidate_file_ids), BATCH_SIZE)
]

print(f"Downloading in {len(batches)} batches of up to {BATCH_SIZE} files each")

failed_batches = []
for i, batch in enumerate(batches, 1):
    success = download_batch_with_retry(batch, i, len(batches), download_dir)
    if not success:
        failed_batches.append(batch)

if failed_batches:
    failed_ids = [fid for b in failed_batches for fid in b]
    print(f"\n{len(failed_ids)}/{len(all_candidate_file_ids)} files failed after retries:")
    print(failed_ids)
    raise RuntimeError(
        "Some GDC files could not be downloaded after retries -- see list above. "
        "Investigate the specific failed file_ids before proceeding."
    )


extracted_files = (
    list(
        download_dir.rglob(
            "*.tsv"
        )
    )
    +
    list(
        download_dir.rglob(
            "*.tsv.gz"
        )
    )
)


print(
    "Extracted files:",
    len(extracted_files)
)


file_id_to_path = {}


for path in extracted_files:

    path_text = str(
        path
    )


    for fid in all_candidate_file_ids:

        if fid in path_text:

            file_id_to_path[
                fid
            ] = path
            break


checkpoint(
    len(file_id_to_path)
    ==
    len(all_candidate_file_ids),
    (
        "every eligible GDC file ID mapped to "
        f"an extracted file "
        f"({len(file_id_to_path)}/"
        f"{len(all_candidate_file_ids)})"
    )
)


# ======================================================================
# STEP 3 — RESOLVE MULTIPLE GDC FILES WITHIN EXACT SAMPLE ONLY
# ======================================================================

section(
    "STEP 3 — Resolve duplicate GDC files within exact biological sample"
)


selected_files = {}

quality_rows = []


for _, row in exact.iterrows():

    patient_id = (
        row[
            "patient_id"
        ]
    )


    candidate_ids = (
        row[
            "matched_file_id_list"
        ]
    )


    candidate_quality = []


    for fid in candidate_ids:

        path = (
            file_id_to_path[
                fid
            ]
        )


        df = parse_star_count_file(
            path
        )


        protein_coding = (
            df[
                df[
                    "gene_type"
                ]
                ==
                "protein_coding"
            ]
        )


        n_detected = int(
            (
                protein_coding[
                    "unstranded"
                ]
                >
                0
            ).sum()
        )


        total_assigned = float(
            protein_coding[
                "unstranded"
            ].sum()
        )


        candidate_quality.append(
            (
                fid,
                n_detected,
                total_assigned
            )
        )


        quality_rows.append(
            {
                "patient_id":
                    patient_id,

                "file_id":
                    fid,

                "n_protein_coding_detected":
                    n_detected,

                "total_assigned_counts":
                    total_assigned,

                "eligible_exact_sample_files":
                    len(
                        candidate_ids
                    )
            }
        )


    candidate_quality = sorted(
        candidate_quality,
        key=lambda x: (
            -x[1],
            -x[2]
        )
    )


    best_fid = (
        candidate_quality[0][0]
    )


    selected_files[
        patient_id
    ] = best_fid


quality_df = pd.DataFrame(
    quality_rows
)


quality_df.to_csv(
    OUTDIR
    / "STAGE_C0_GDC_FILE_QUALITY_EXACT_PAIRS.csv",
    index=False
)


checkpoint(
    len(selected_files) == 60,
    "exactly one selected GDC file for each of the 60 bridge samples"
)


checkpoint(
    len(
        set(
            selected_files.values()
        )
    )
    ==
    60,
    "selected GDC file IDs are unique across bridge samples"
)


# ======================================================================
# STEP 4 — BUILD GDC RAW COUNT MATRIX
# ======================================================================

section(
    "STEP 4 — Build GDC raw count matrix"
)


gdc_columns = {}


for patient_id, fid in selected_files.items():

    df = parse_star_count_file(
        file_id_to_path[
            fid
        ]
    )


    df[
        "ensembl_id"
    ] = (
        df[
            "gene_id"
        ]
        .apply(
            strip_version
        )
    )


    series = (
        df
        .set_index(
            "ensembl_id"
        )[
            "unstranded"
        ]
    )


    # deterministic duplicate handling
    series = (
        series[
            ~series
            .index
            .duplicated(
                keep="first"
            )
        ]
    )


    gdc_columns[
        patient_id
    ] = series


gdc_matrix = pd.DataFrame(
    gdc_columns
)


print(
    "GDC matrix:",
    gdc_matrix.shape[0],
    "genes x",
    gdc_matrix.shape[1],
    "patients"
)


checkpoint(
    gdc_matrix.shape[1] == 60,
    "GDC matrix contains exactly 60 bridge patients"
)


# ======================================================================
# STEP 5 — LOAD RECOUNT3 METADATA
# ======================================================================

section(
    "STEP 5 — Load recount3 TCGA-GBM metadata"
)


shard = project_shard(
    TCGA_PROJECT
)


metadata_url = (
    f"{BASE}/tcga/metadata/"
    f"{shard}/{TCGA_PROJECT}/"
    f"tcga.tcga.{TCGA_PROJECT}.MD.gz"
)


if RECOUNT3_META_CACHE.exists():

    print(
        "Using cached metadata:"
    )

    print(
        RECOUNT3_META_CACHE
    )


    recount_meta = pd.read_csv(
        RECOUNT3_META_CACHE,
        low_memory=False
    )


else:

    recount_meta = (
        fetch_recount3_table(
            metadata_url
        )
    )


    recount_meta.to_csv(
        RECOUNT3_META_CACHE,
        index=False
    )


print(
    "recount3 metadata:",
    recount_meta.shape
)


# ======================================================================
# STEP 6 — LOAD RECOUNT3 QC
# ======================================================================

section(
    "STEP 6 — Load recount3 QC"
)


qc_url = (
    f"{BASE}/tcga/metadata/"
    f"{shard}/{TCGA_PROJECT}/"
    f"tcga.recount_qc.{TCGA_PROJECT}.MD.gz"
)


if RECOUNT3_QC_CACHE.exists():

    print(
        "Using cached QC:"
    )

    print(
        RECOUNT3_QC_CACHE
    )


    recount_qc = pd.read_csv(
        RECOUNT3_QC_CACHE,
        low_memory=False
    )


else:

    recount_qc = (
        fetch_recount3_table(
            qc_url
        )
    )


    recount_qc.to_csv(
        RECOUNT3_QC_CACHE,
        index=False
    )


length_col = (
    select_avg_mapped_length_column(
        recount_qc.columns
    )
)


print(
    "Average mapped length column:",
    length_col
)


# ======================================================================
# STEP 7 — LOAD RECOUNT3 gene_sums
# ======================================================================

section(
    "STEP 7 — Load recount3 gene_sums"
)


gene_sums_url = (
    f"{BASE}/tcga/gene_sums/"
    f"{shard}/{TCGA_PROJECT}/"
    f"tcga.gene_sums.{TCGA_PROJECT}.G026.gz"
)


if RECOUNT3_GENE_SUMS_CACHE.exists():

    print(
        "Using cached gene_sums:"
    )

    print(
        RECOUNT3_GENE_SUMS_CACHE
    )


    gene_sums = pd.read_csv(
        RECOUNT3_GENE_SUMS_CACHE,
        low_memory=False
    )


else:

    gene_sums = (
        fetch_recount3_table(
            gene_sums_url
        )
    )


    gene_sums.to_csv(
        RECOUNT3_GENE_SUMS_CACHE,
        index=False
    )


print(
    "gene_sums raw shape:",
    gene_sums.shape
)


checkpoint(
    gene_sums.shape[0] > 1000,
    "gene_sums contains a plausible number of genes"
)


gene_col = (
    gene_sums.columns[0]
)


gene_sums = (
    gene_sums
    .set_index(
        gene_col
    )
)


# ======================================================================
# STEP 8 — RESOLVE RECOUNT3 EXTERNAL SAMPLE ID TO REAL gene_sums COLUMN
# ======================================================================

section(
    "STEP 8 — Map frozen recount3 barcodes to actual expression columns"
)


bridge_tumors = pd.read_csv(
    STAGE_A_TUMOR_FILE,
    low_memory=False
)


patient_to_recount_external = (
    bridge_tumors
    .set_index(
        "Patient ID"
    )[
        "Recount sample ID"
    ]
    .to_dict()
)


possible_meta_id_columns = [
    c
    for c in [
        "external_id",
        "rail_id",
        "tcga_barcode"
    ]
    if c in recount_meta.columns
]


checkpoint(
    len(
        possible_meta_id_columns
    )
    >
    0,
    "recount3 metadata contains usable identifier columns"
)


expression_columns = set(
    gene_sums.columns
)


resolved_recount_columns = {}


mapping_audit = []


for patient_id in selected_files.keys():

    external_sample = (
        patient_to_recount_external.get(
            patient_id
        )
    )


    checkpoint(
        external_sample is not None,
        f"{patient_id} has a frozen recount3 external sample ID"
    )


    meta_mask = pd.Series(
        False,
        index=recount_meta.index
    )


    for col in possible_meta_id_columns:

        values = (
            recount_meta[
                col
            ]
            .astype(str)
            .str.strip()
        )


        meta_mask |= (
            values
            ==
            str(
                external_sample
            ).strip()
        )


    matched_rows = (
        recount_meta[
            meta_mask
        ]
    )


    if len(
        matched_rows
    ) == 0:

        raise RuntimeError(
            f"No recount3 metadata row found for "
            f"{patient_id} / {external_sample}"
        )


    candidates = []


    for row_index, meta_row in matched_rows.iterrows():

        for col in possible_meta_id_columns:

            value = clean_string(
                meta_row.get(
                    col
                )
            )


            if (
                value is not None
                and
                value in expression_columns
            ):

                candidates.append(
                    (
                        value,
                        col,
                        row_index
                    )
                )


    # remove duplicate expression column candidates
    unique_candidates = {}


    for item in candidates:

        unique_candidates[
            item[0]
        ] = item


    candidates = list(
        unique_candidates.values()
    )


    if len(
        candidates
    ) != 1:

        raise RuntimeError(
            f"{patient_id}: expected exactly one recount3 "
            f"gene_sums column, found {len(candidates)}.\n"
            f"Candidates: {candidates}"
        )


    expression_col, matched_using, row_index = (
        candidates[0]
    )


    resolved_recount_columns[
        patient_id
    ] = expression_col


    mapping_audit.append(
        {
            "patient_id":
                patient_id,

            "recount_external_sample":
                external_sample,

            "expression_column":
                expression_col,

            "matched_using":
                matched_using,

            "metadata_row":
                row_index
        }
    )


checkpoint(
    len(
        resolved_recount_columns
    )
    ==
    60,
    "all 60 bridge patients resolved to an actual recount3 gene_sums column"
)


mapping_audit_df = pd.DataFrame(
    mapping_audit
)


mapping_audit_df.to_csv(
    OUTDIR
    / "STAGE_C0_RECOUNT3_SAMPLE_MAPPING_AUDIT.csv",
    index=False
)


# ======================================================================
# STEP 9 — MAP RECOUNT3 SAMPLE TO AVERAGE MAPPED READ LENGTH
# ======================================================================

section(
    "STEP 9 — Resolve recount3 average mapped read lengths"
)


possible_qc_ids = [
    c
    for c in [
        "external_id",
        "rail_id",
        "tcga_barcode"
    ]
    if c in recount_qc.columns
]


checkpoint(
    len(
        possible_qc_ids
    )
    >
    0,
    "recount3 QC contains usable identifier fields"
)


sample_lengths = {}


for patient_id, expression_col in (
    resolved_recount_columns.items()
):


    identifiers = {
        expression_col,
        patient_to_recount_external[
            patient_id
        ]
    }


    mask = pd.Series(
        False,
        index=recount_qc.index
    )


    for col in possible_qc_ids:

        values = (
            recount_qc[
                col
            ]
            .astype(str)
            .str.strip()
        )


        mask |= (
            values.isin(
                identifiers
            )
        )


    matches = (
        recount_qc[
            mask
        ]
    )


    if len(
        matches
    ) == 0:

        raise RuntimeError(
            f"No recount3 QC row found for "
            f"{patient_id}"
        )


    lengths = (
        pd.to_numeric(
            matches[
                length_col
            ],
            errors="coerce"
        )
        .dropna()
        .unique()
    )


    if len(
        lengths
    ) != 1:

        raise RuntimeError(
            f"{patient_id}: expected one mapped read "
            f"length, found {lengths.tolist()}"
        )


    sample_lengths[
        expression_col
    ] = float(
        lengths[0]
    )


checkpoint(
    len(
        sample_lengths
    )
    ==
    60,
    "mapped read length obtained for all 60 bridge samples"
)


# ======================================================================
# STEP 10 — BUILD RECOUNT3 COUNT MATRIX
# ======================================================================

section(
    "STEP 10 — Build recount3 estimated read-count matrix"
)


ordered_patients = list(
    selected_files.keys()
)


ordered_recount_columns = [
    resolved_recount_columns[
        pid
    ]
    for pid in ordered_patients
]


recount3_sub = (
    gene_sums[
        ordered_recount_columns
    ]
    .copy()
)


length_series = pd.Series(
    sample_lengths
)


length_series = (
    length_series.loc[
        ordered_recount_columns
    ]
)


recount3_counts = (
    recount3_sub
    .div(
        length_series,
        axis=1
    )
    .round()
)


recount3_counts.columns = (
    ordered_patients
)


recount3_counts.index = [
    strip_version(
        g
    )
    for g in recount3_counts.index
]


print(
    "recount3 matrix:",
    recount3_counts.shape[0],
    "genes x",
    recount3_counts.shape[1],
    "patients"
)


checkpoint(
    recount3_counts.shape[1] == 60,
    "recount3 matrix contains exactly 60 bridge patients"
)


checkpoint(
    list(
        recount3_counts.columns
    )
    ==
    list(
        gdc_matrix.columns
    ),
    "GDC and recount3 patient columns have identical order"
)


# ======================================================================
# STEP 11 — HARMONIZE BASE ENSEMBL IDs
# ======================================================================

section(
    "STEP 11 — Harmonize GDC v36 and recount3 v26 genes"
)


# remove duplicate Ensembl IDs deterministically
gdc_matrix = (
    gdc_matrix[
        ~gdc_matrix
        .index
        .duplicated(
            keep="first"
        )
    ]
)


recount3_counts = (
    recount3_counts[
        ~recount3_counts
        .index
        .duplicated(
            keep="first"
        )
    ]
)


common_genes = sorted(
    set(
        gdc_matrix.index
    )
    &
    set(
        recount3_counts.index
    )
)


print(
    "GDC genes:",
    gdc_matrix.index.nunique()
)


print(
    "recount3 genes:",
    recount3_counts.index.nunique()
)


print(
    "Common base Ensembl IDs:",
    len(
        common_genes
    )
)


checkpoint(
    len(
        common_genes
    )
    >
    1000,
    f"substantial common gene universe exists ({len(common_genes)})"
)


gdc_matrix = (
    gdc_matrix.loc[
        common_genes
    ]
)


recount3_counts = (
    recount3_counts.loc[
        common_genes
    ]
)


# ======================================================================
# STEP 12 — LOG2(CPM+1) ON BOTH SIDES
# ======================================================================

section(
    "STEP 12 — Normalize both bridge matrices to log2(CPM+1)"
)


gdc_log_cpm = (
    log2_cpm(
        gdc_matrix
    )
)


recount3_log_cpm = (
    log2_cpm(
        recount3_counts
    )
)


checkpoint(
    gdc_log_cpm.shape
    ==
    recount3_log_cpm.shape,
    f"bridge matrices have identical shape {gdc_log_cpm.shape}"
)


checkpoint(
    list(
        gdc_log_cpm.index
    )
    ==
    list(
        recount3_log_cpm.index
    ),
    "bridge matrices have identical gene order"
)


checkpoint(
    list(
        gdc_log_cpm.columns
    )
    ==
    list(
        recount3_log_cpm.columns
    ),
    "bridge matrices have identical patient order"
)


# ======================================================================
# STEP 13 — BASIC PAIRWISE AGREEMENT DIAGNOSTIC
# ======================================================================

section(
    "STEP 13 — Pre-calibration paired agreement diagnostic"
)


sample_correlations = []


for patient_id in ordered_patients:

    x = (
        gdc_log_cpm[
            patient_id
        ]
    )


    y = (
        recount3_log_cpm[
            patient_id
        ]
    )


    valid = (
        x.notna()
        &
        y.notna()
    )


    if valid.sum() < 100:

        corr = np.nan

    else:

        corr = (
            x[
                valid
            ]
            .corr(
                y[
                    valid
                ],
                method="spearman"
            )
        )


    sample_correlations.append(
        {
            "patient_id":
                patient_id,

            "precalibration_spearman":
                corr
        }
    )


agreement_df = pd.DataFrame(
    sample_correlations
)


print(
    agreement_df[
        "precalibration_spearman"
    ].describe()
)


agreement_df.to_csv(
    OUTDIR
    / "STAGE_C0_PRECALIBRATION_SAMPLE_CORRELATIONS.csv",
    index=False
)


# ======================================================================
# STEP 14 — SAVE ALIGNED MATRICES
# ======================================================================

section(
    "STEP 14 — Save exact-pair bridge matrices"
)


GDC_OUT = (
    OUTDIR
    / "STAGE_C0_GDC_LOG_CPM_60_EXACT_BRIDGE.csv"
)


RECOUNT_OUT = (
    OUTDIR
    / "STAGE_C0_RECOUNT3_LOG_CPM_60_EXACT_BRIDGE.csv"
)


gdc_log_cpm.to_csv(
    GDC_OUT
)


recount3_log_cpm.to_csv(
    RECOUNT_OUT
)


selected_file_table = pd.DataFrame(
    [
        {
            "patient_id":
                pid,

            "selected_gdc_file_id":
                fid,

            "recount3_expression_column":
                resolved_recount_columns[
                    pid
                ],

            "recount3_external_sample":
                patient_to_recount_external[
                    pid
                ]
        }
        for pid, fid in selected_files.items()
    ]
)


selected_file_table.to_csv(
    OUTDIR
    / "STAGE_C0_FINAL_BRIDGE_PAIR_TABLE.csv",
    index=False
)


print(
    "Saved:"
)

print(
    GDC_OUT
)

print(
    RECOUNT_OUT
)


print(
    "\nFinal dimensions:"
)

print(
    gdc_log_cpm.shape[0],
    "genes x",
    gdc_log_cpm.shape[1],
    "bridge samples"
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "STAGE C0 PART 1 COMPLETE"
)


print(
    "Two exact-pair bridge matrices are ready."
)


print(
    "Calibration currency: log2(CPM+1)"
)


print(
    "Bridge patients: 60 exact biological sample pairs"
)


print(
    "\nNEXT STEP:"
)


print(
    "Stage C0 Part 2:"
)


print(
    "Fit GDC -> recount3 calibration using bridge-training pairs,"
)


print(
    "evaluate on held-out bridge pairs,"
)


print(
    "and only if calibration improves agreement, apply it to the 144 patients."
)

### STAGE C0 — COMPLETE BRIDGE DIAGNOSTIC

In [ ]:
# ======================================================================
# STAGE C0 — COMPLETE BRIDGE DIAGNOSTIC
# FINAL CORRECTED VERSION
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu


# ======================================================================
# 0 — PATHS
# ======================================================================

DOWNLOADS = BASE

CAL_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_A_DISCOVERY"
    / "STAGE_C0_BRIDGE_CALIBRATION"
)


GDC_FILE = (
    CAL_DIR
    / "STAGE_C0_GDC_LOG_CPM_60_EXACT_BRIDGE.csv"
)


RECOUNT3_FILE = (
    CAL_DIR
    / "STAGE_C0_RECOUNT3_LOG_CPM_60_EXACT_BRIDGE.csv"
)


QUALITY_FILE = (
    CAL_DIR
    / "STAGE_C0_GDC_FILE_QUALITY_EXACT_PAIRS.csv"
)


AGREEMENT_OUTPUT = (
    CAL_DIR
    / "STAGE_C0_PRECALIBRATION_AGREEMENT.csv"
)


DIAGNOSTIC_OUTPUT = (
    CAL_DIR
    / "STAGE_C0_BRIDGE_DIAGNOSTIC.csv"
)


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def checkpoint(condition, message):

    status = "PASS" if condition else "FAIL"

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: {message}"
        )


# ======================================================================
# 1 — LOAD MATRICES
# ======================================================================

section(
    "1 — LOAD BRIDGE MATRICES"
)


print("GDC file:")
print(GDC_FILE)

print("\nrecount3 file:")
print(RECOUNT3_FILE)

print("\nQuality file:")
print(QUALITY_FILE)


checkpoint(
    GDC_FILE.exists(),
    "GDC bridge matrix exists"
)


checkpoint(
    RECOUNT3_FILE.exists(),
    "recount3 bridge matrix exists"
)


checkpoint(
    QUALITY_FILE.exists(),
    "GDC quality table exists"
)


gdc = pd.read_csv(
    GDC_FILE,
    index_col=0
)


recount3 = pd.read_csv(
    RECOUNT3_FILE,
    index_col=0
)


quality = pd.read_csv(
    QUALITY_FILE
)


print(
    "\nGDC matrix:",
    gdc.shape
)


print(
    "recount3 matrix:",
    recount3.shape
)


print(
    "Quality rows:",
    len(quality)
)


# ======================================================================
# 2 — ALIGNMENT CHECKS
# ======================================================================

section(
    "2 — ALIGNMENT CHECK"
)


checkpoint(
    gdc.shape == recount3.shape,
    f"same matrix shape: {gdc.shape}"
)


checkpoint(
    list(gdc.index)
    ==
    list(recount3.index),
    "same gene order"
)


checkpoint(
    list(gdc.columns)
    ==
    list(recount3.columns),
    "same patient order"
)


checkpoint(
    gdc.shape[1] == 60,
    "exactly 60 bridge patients"
)


print(
    "\nGenes:",
    gdc.shape[0]
)

print(
    "Patients:",
    gdc.shape[1]
)


# ======================================================================
# 3 — REBUILD PRE-CALIBRATION AGREEMENT
# ======================================================================

section(
    "3 — ALL-GENE PRE-CALIBRATION AGREEMENT"
)


agreement_rows = []


for patient_id in gdc.columns:

    x = pd.to_numeric(
        gdc[patient_id],
        errors="coerce"
    )

    y = pd.to_numeric(
        recount3[patient_id],
        errors="coerce"
    )


    valid = (
        x.notna()
        &
        y.notna()
    )


    if valid.sum() < 100:

        pearson = np.nan
        spearman = np.nan
        rmse = np.nan

    else:

        pearson = (
            x[valid]
            .corr(
                y[valid],
                method="pearson"
            )
        )


        spearman = (
            x[valid]
            .corr(
                y[valid],
                method="spearman"
            )
        )


        rmse = float(
            np.sqrt(
                np.mean(
                    (
                        x[valid].values
                        -
                        y[valid].values
                    )
                    ** 2
                )
            )
        )


    agreement_rows.append(
        {
            "patient_id":
                patient_id,

            "pearson_before":
                pearson,

            "spearman_before":
                spearman,

            "rmse_before":
                rmse
        }
    )


precal = pd.DataFrame(
    agreement_rows
)


print(
    "\nSpearman:"
)

print(
    precal[
        "spearman_before"
    ].describe()
)


print(
    "\nPearson:"
)

print(
    precal[
        "pearson_before"
    ].describe()
)


print(
    "\nRMSE:"
)

print(
    precal[
        "rmse_before"
    ].describe()
)


# save reconstructed agreement
precal.to_csv(
    AGREEMENT_OUTPUT,
    index=False
)


print(
    "\nSaved:"
)

print(
    AGREEMENT_OUTPUT
)


# ======================================================================
# 4 — WELL-EXPRESSED GENE CORRELATIONS
# ======================================================================

section(
    "4 — CORRELATION ON WELL-EXPRESSED GENES"
)


mean_expression = (
    gdc.mean(axis=1)
    +
    recount3.mean(axis=1)
) / 2


restricted_results = {}


for top_n in [
    5000,
    10000,
    20000
]:

    actual_n = min(
        top_n,
        len(mean_expression)
    )


    top_genes = (
        mean_expression
        .nlargest(
            actual_n
        )
        .index
    )


    correlations = {}


    for patient_id in gdc.columns:

        x = (
            gdc.loc[
                top_genes,
                patient_id
            ]
        )


        y = (
            recount3.loc[
                top_genes,
                patient_id
            ]
        )


        correlations[
            patient_id
        ] = (
            x.corr(
                y,
                method="spearman"
            )
        )


    values = pd.Series(
        correlations,
        name=f"spearman_top_{actual_n}"
    )


    restricted_results[
        actual_n
    ] = values


    print(
        f"\nTop {actual_n:,} genes:"
    )

    print(
        f"  mean   = {values.mean():.4f}"
    )

    print(
        f"  median = {values.median():.4f}"
    )

    print(
        f"  std    = {values.std():.4f}"
    )

    print(
        f"  min    = {values.min():.4f}"
    )

    print(
        f"  Q1     = "
        f"{values.quantile(0.25):.4f}"
    )

    print(
        f"  Q3     = "
        f"{values.quantile(0.75):.4f}"
    )

    print(
        f"  max    = {values.max():.4f}"
    )


print(
    "\nOriginal all-gene Spearman:"
)

print(
    f"  mean   = "
    f"{precal['spearman_before'].mean():.4f}"
)

print(
    f"  median = "
    f"{precal['spearman_before'].median():.4f}"
)


# ======================================================================
# 5 — SELECT ACTUAL QC RECORD PER PATIENT
# ======================================================================

section(
    "5 — SELECTED GDC FILE QUALITY"
)


print(
    "Quality columns:"
)

print(
    quality.columns.tolist()
)


# REAL COLUMN NAMES IN YOUR FILE

DETECTED_COL = (
    "n_protein_coding_detected"
)

COUNTS_COL = (
    "total_assigned_counts"
)


required_quality_columns = [
    "patient_id",
    DETECTED_COL,
    COUNTS_COL
]


missing_quality_columns = [

    col
    for col
    in required_quality_columns

    if col
    not in quality.columns
]


if missing_quality_columns:

    raise RuntimeError(
        "Missing required quality columns:\n"
        f"{missing_quality_columns}\n\n"
        f"Available columns:\n"
        f"{quality.columns.tolist()}"
    )


# ------------------------------------------------------------
# Same selection rule as bridge construction:
#
# 1. highest detected protein-coding genes
# 2. highest total assigned counts
# ------------------------------------------------------------

selected_quality = (
    quality
    .sort_values(
        [
            "patient_id",
            DETECTED_COL,
            COUNTS_COL
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset="patient_id",
        keep="first"
    )
    .copy()
)


checkpoint(
    selected_quality[
        "patient_id"
    ].nunique()
    ==
    60,
    "one selected QC record per bridge patient"
)


print(
    "\nSelected QC rows:",
    len(selected_quality)
)


# ======================================================================
# 6 — LOW vs HIGH AGREEMENT GROUPS
# ======================================================================

section(
    "6 — LOW vs HIGH AGREEMENT GROUPS"
)


q1 = (
    precal[
        "spearman_before"
    ]
    .quantile(
        0.25
    )
)


q3 = (
    precal[
        "spearman_before"
    ]
    .quantile(
        0.75
    )
)


low_agreement = (
    precal[
        precal[
            "spearman_before"
        ]
        <=
        q1
    ]
    .copy()
)


high_agreement = (
    precal[
        precal[
            "spearman_before"
        ]
        >=
        q3
    ]
    .copy()
)


print(
    f"Bottom quartile threshold: {q1:.4f}"
)


print(
    f"Top quartile threshold: {q3:.4f}"
)


print(
    "Low-agreement patients:",
    len(low_agreement)
)


print(
    "High-agreement patients:",
    len(high_agreement)
)


# ======================================================================
# 7 — MERGE QUALITY METRICS
# ======================================================================

low = (
    low_agreement
    .merge(
        selected_quality[
            [
                "patient_id",
                DETECTED_COL,
                COUNTS_COL
            ]
        ],
        on="patient_id",
        how="left"
    )
)


high = (
    high_agreement
    .merge(
        selected_quality[
            [
                "patient_id",
                DETECTED_COL,
                COUNTS_COL
            ]
        ],
        on="patient_id",
        how="left"
    )
)


checkpoint(
    low[
        DETECTED_COL
    ].notna().all(),
    "all low-agreement patients have QC values"
)


checkpoint(
    high[
        DETECTED_COL
    ].notna().all(),
    "all high-agreement patients have QC values"
)


# ======================================================================
# 8 — SHOW BOTTOM QUARTILE
# ======================================================================

section(
    "7 — BOTTOM-QUARTILE PATIENTS"
)


print(
    low[
        [
            "patient_id",
            "pearson_before",
            "spearman_before",
            "rmse_before",
            DETECTED_COL,
            COUNTS_COL
        ]
    ]
    .sort_values(
        "spearman_before"
    )
    .to_string(
        index=False
    )
)


# ======================================================================
# 9 — QC SUMMARY
# ======================================================================

section(
    "8 — GDC QC: LOW vs HIGH AGREEMENT"
)


print(
    "\nHIGH-agreement group"
)

print(
    "N:",
    len(high)
)


print(
    "\nDetected protein-coding genes:"
)

print(
    high[
        DETECTED_COL
    ].describe()
)


print(
    "\nAssigned counts:"
)

print(
    high[
        COUNTS_COL
    ].describe()
)


print(
    "\nLOW-agreement group"
)

print(
    "N:",
    len(low)
)


print(
    "\nDetected protein-coding genes:"
)

print(
    low[
        DETECTED_COL
    ].describe()
)


print(
    "\nAssigned counts:"
)

print(
    low[
        COUNTS_COL
    ].describe()
)


# ======================================================================
# 10 — FORMAL QC TESTS
# ======================================================================

section(
    "9 — LOW vs HIGH QC STATISTICAL TESTS"
)


test_results = []


for variable in [
    DETECTED_COL,
    COUNTS_COL
]:

    low_values = (
        low[
            variable
        ]
        .dropna()
    )


    high_values = (
        high[
            variable
        ]
        .dropna()
    )


    test = mannwhitneyu(
        low_values,
        high_values,
        alternative="two-sided"
    )


    print(
        f"\n{variable}"
    )


    print(
        f"LOW median  = "
        f"{low_values.median():,.2f}"
    )


    print(
        f"HIGH median = "
        f"{high_values.median():,.2f}"
    )


    print(
        f"U = {test.statistic:.2f}"
    )


    print(
        f"p = {test.pvalue:.6g}"
    )


    test_results.append(
        {
            "variable":
                variable,

            "low_median":
                low_values.median(),

            "high_median":
                high_values.median(),

            "mannwhitney_U":
                test.statistic,

            "p_value":
                test.pvalue
        }
    )


# ======================================================================
# 11 — SAVE FULL DIAGNOSTIC
# ======================================================================

section(
    "10 — SAVE COMPLETE DIAGNOSTIC"
)


patient_diagnostic = (
    precal
    .merge(
        selected_quality[
            [
                "patient_id",
                DETECTED_COL,
                COUNTS_COL
            ]
        ],
        on="patient_id",
        how="left"
    )
)


for top_n, values in (
    restricted_results.items()
):

    patient_diagnostic[
        f"spearman_top_{top_n}"
    ] = (
        patient_diagnostic[
            "patient_id"
        ]
        .map(
            values
        )
    )


patient_diagnostic.to_csv(
    DIAGNOSTIC_OUTPUT,
    index=False
)


qc_test_output = (
    CAL_DIR
    / "STAGE_C0_LOW_HIGH_QC_TESTS.csv"
)


pd.DataFrame(
    test_results
).to_csv(
    qc_test_output,
    index=False
)


print(
    "Saved:"
)

print(
    DIAGNOSTIC_OUTPUT
)

print(
    qc_test_output
)


# ======================================================================
# 12 — FINAL SUMMARY
# ======================================================================

section(
    "DIAGNOSTIC COMPLETE"
)


print(
    "All-gene pre-calibration:"
)


print(
    f"  Spearman mean   = "
    f"{precal['spearman_before'].mean():.4f}"
)


print(
    f"  Spearman median = "
    f"{precal['spearman_before'].median():.4f}"
)


for top_n, values in (
    restricted_results.items()
):

    print(
        f"\nTop {top_n:,} genes:"
    )

    print(
        f"  mean   = {values.mean():.4f}"
    )

    print(
        f"  median = {values.median():.4f}"
    )


print(
    "\nInterpretation rule:"
)


print(
    "If low- and high-agreement groups show no "
    "significant objective QC difference, keep all 60 "
    "exact bridge pairs for calibration."
)


print(
    "If low-agreement samples show clearly worse "
    "objective QC, define a QC rule first and only then "
    "exclude those samples."
)


print(
    "\nNEXT:"
)


print(
    "Stage C0 Part 2 — held-out GDC -> recount3 calibration."
)

### Which genes drive TCGA-06-0750's top-5k Spearman collapse (0.784 -> 0.413)?

In [ ]:
"""
Which genes drive TCGA-06-0750's top-5k Spearman collapse (0.784 -> 0.413)?
Distinguishes "a few problem genes" (annotation/paralog artifact, worth
naming) from "diffuse noise" (more consistent with a general sample issue).
"""

import numpy as np
import pandas as pd
from pathlib import Path

DOWNLOADS = BASE
CAL_DIR = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY" / "STAGE_C0_BRIDGE_CALIBRATION"

gdc = pd.read_csv(CAL_DIR / "STAGE_C0_GDC_LOG_CPM_60_EXACT_BRIDGE.csv", index_col=0)
recount3 = pd.read_csv(CAL_DIR / "STAGE_C0_RECOUNT3_LOG_CPM_60_EXACT_BRIDGE.csv", index_col=0)

PATIENT = "TCGA-06-0750"
TOP_N = 5000

mean_expr = (gdc.mean(axis=1) + recount3.mean(axis=1)) / 2
top_genes = mean_expr.nlargest(TOP_N).index

g = gdc.loc[top_genes, PATIENT]
r = recount3.loc[top_genes, PATIENT]
diff = (g - r).abs().sort_values(ascending=False)

top10_share = diff.head(10).sum() / diff.sum()
top30_share = diff.head(30).sum() / diff.sum()

print(f"Patient: {PATIENT}, top-{TOP_N} genes")
print(f"Top-10 outlier genes account for {top10_share*100:.1f}% of total |difference|")
print(f"Top-30 outlier genes account for {top30_share*100:.1f}% of total |difference|")
print(f"\n(Concentrated ~60-70%+ in top 10 -> specific problem genes, worth naming.")
print(f" Diffuse ~20-30% -> more consistent with a general sample issue.)\n")

print("Top 30 most divergent genes (Ensembl ID, GDC log2CPM, recount3 log2CPM, |diff|):")
top30 = pd.DataFrame({
    "gdc_log2cpm": g.loc[diff.head(30).index],
    "recount3_log2cpm": r.loc[diff.head(30).index],
    "abs_diff": diff.head(30),
})
print(top30.to_string())

print("\n" + "=" * 70)
print("For comparison: same analysis on a well-behaved patient")
print("=" * 70)
# pick whichever high-agreement patient you have -- placeholder here,
# swap in a real one from your top-quartile list
COMPARISON_PATIENT = None  # e.g. "TCGA-XX-XXXX" from your high-agreement group
if COMPARISON_PATIENT and COMPARISON_PATIENT in gdc.columns:
    g2 = gdc.loc[top_genes, COMPARISON_PATIENT]
    r2 = recount3.loc[top_genes, COMPARISON_PATIENT]
    diff2 = (g2 - r2).abs().sort_values(ascending=False)
    print(f"{COMPARISON_PATIENT}: top-10 share = {diff2.head(10).sum()/diff2.sum()*100:.1f}%")
else:
    print("Set COMPARISON_PATIENT to a high-agreement patient ID to compare against.")

print("\nManual check worth doing on the top divergent gene IDs above: cross-check")
print("their gene symbols against known multi-copy/paralog-prone loci (PAR genes,")
print("histone gene family, or other genes with multiple near-identical Ensembl")
print("records) -- this exact pattern (a small number of specific genes causing a")
print("large discrepancy) matched a real identifier-collision problem earlier in")
print("this project (SLC25A6, CD99, histone genes) when comparing GDC against a")
print("different reference source.")

### Tests whether TCGA-06-0750's collapse is explained by a GDC-vs-recount3

In [ ]:
"""
Tests whether TCGA-06-0750's collapse is explained by a GDC-vs-recount3
disagreement in mitochondrial read fraction specifically -- and whether
this is a one-patient problem or affects the whole bridge cohort to
varying degrees (which would argue for excluding MT genes from BOTH the
CPM denominator and the calibration gene set, not excluding patients).
"""

import numpy as np
import pandas as pd
from pathlib import Path

DOWNLOADS = BASE
CAL_DIR = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY" / "STAGE_C0_BRIDGE_CALIBRATION"

gdc_log = pd.read_csv(CAL_DIR / "STAGE_C0_GDC_LOG_CPM_60_EXACT_BRIDGE.csv", index_col=0)
recount3_log = pd.read_csv(CAL_DIR / "STAGE_C0_RECOUNT3_LOG_CPM_60_EXACT_BRIDGE.csv", index_col=0)

# Canonical mitochondrial protein-coding + rRNA gene Ensembl IDs.
# NOTE: these are from general knowledge, not a fresh database lookup for
# THIS analysis -- spot-check a couple against Ensembl/BioMart if you want
# full certainty before relying on this list for the paper.
MT_GENES = {
    "MT-ND1": "ENSG00000198888", "MT-ND2": "ENSG00000198763",
    "MT-CO1": "ENSG00000198804", "MT-CO2": "ENSG00000198712",
    "MT-ATP8": "ENSG00000228253", "MT-ATP6": "ENSG00000198899",
    "MT-CO3": "ENSG00000198938", "MT-ND3": "ENSG00000198840",
    "MT-ND4L": "ENSG00000212907", "MT-ND4": "ENSG00000198886",
    "MT-ND5": "ENSG00000198786", "MT-ND6": "ENSG00000198695",
    "MT-CYB": "ENSG00000198727", "MT-RNR1": "ENSG00000211459",
    "MT-RNR2": "ENSG00000210082",
}

mt_ids_present = [eid for eid in MT_GENES.values() if eid in gdc_log.index]
print(f"Canonical MT genes found in the bridge matrices: {len(mt_ids_present)}/{len(MT_GENES)}")
missing = [name for name, eid in MT_GENES.items() if eid not in gdc_log.index]
if missing:
    print(f"Not found (check ID spelling or filtering upstream): {missing}")

# Approximate CPM back from log2(CPM+1): CPM = 2^log2cpm - 1
gdc_cpm = 2 ** gdc_log - 1
recount3_cpm = 2 ** recount3_log - 1

# MT genes' share of total CPM (proxy for MT read fraction, since CPM
# values sum to ~1e6 across the full gene set by construction)
gdc_mt_fraction = gdc_cpm.loc[mt_ids_present].sum() / gdc_cpm.sum()
recount3_mt_fraction = recount3_cpm.loc[mt_ids_present].sum() / recount3_cpm.sum()

mt_summary = pd.DataFrame({
    "gdc_mt_fraction": gdc_mt_fraction,
    "recount3_mt_fraction": recount3_mt_fraction,
})
mt_summary["mt_fraction_discrepancy"] = (
    mt_summary["gdc_mt_fraction"] - mt_summary["recount3_mt_fraction"]
).abs()

mt_summary = mt_summary.sort_values("mt_fraction_discrepancy", ascending=False)
print("\nTop 10 patients by GDC-vs-recount3 MT-fraction discrepancy:")
print(mt_summary.head(10).to_string())

print(f"\nTCGA-06-0750's rank by MT-fraction discrepancy: "
      f"{list(mt_summary.index).index('TCGA-06-0750') + 1} out of {len(mt_summary)}")

print("\nCorrelation across ALL 60 patients: does MT-fraction discrepancy")
print("predict overall pipeline disagreement?")
# reload the precalibration correlations for the correlation check
precal = pd.read_csv(CAL_DIR / "STAGE_C0_PRECALIBRATION_AGREEMENT.csv")
merged = precal.merge(mt_summary, left_on="patient_id", right_index=True)
corr = merged["mt_fraction_discrepancy"].corr(merged["spearman_before"], method="spearman")
print(f"Spearman(MT-fraction discrepancy, overall pre-calibration agreement) = {corr:.3f}")
print("(A meaningfully negative correlation here would confirm: patients where")
print("GDC and recount3 disagree more about MT content also show worse overall")
print("agreement -- supporting the shared-denominator distortion hypothesis.)")

### MT-exclusion fix -- reuses everything already cached from Part 1.

In [ ]:
"""
MT-exclusion fix -- reuses everything already cached from Part 1.
No re-download. Loads STAGE_C0_FINAL_BRIDGE_PAIR_TABLE.csv (already has
the resolved file/sample assignments) and the cached recount3 files.
"""

import gzip
import io
import re
import numpy as np
import pandas as pd
from pathlib import Path

DOWNLOADS = BASE
STAGE_A_DIR = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY"
OUTDIR = STAGE_A_DIR / "STAGE_C0_BRIDGE_CALIBRATION"
GDC_DIR = OUTDIR / "gdc_exact_match_files"

pair_table = pd.read_csv(OUTDIR / "STAGE_C0_FINAL_BRIDGE_PAIR_TABLE.csv")

MT_GENES = {
    "ENSG00000198888","ENSG00000198763","ENSG00000198804","ENSG00000198712",
    "ENSG00000228253","ENSG00000198899","ENSG00000198938","ENSG00000198840",
    "ENSG00000212907","ENSG00000198886","ENSG00000198786","ENSG00000198695",
    "ENSG00000198727","ENSG00000211459","ENSG00000210082",
}

def strip_version(x):
    return str(x).split(".")[0].strip()

def parse_star_count_file(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as f:
        lines = f.readlines()
    data_start = next(i for i, l in enumerate(lines) if re.match(r"^ENSG\d+", l.split("\t")[0].strip()))
    return pd.read_csv(io.StringIO("".join(lines[data_start:])), sep="\t", header=None,
                        names=["gene_id","gene_name","gene_type","unstranded",
                               "stranded_first","stranded_second","tpm_unstranded",
                               "fpkm_unstranded","fpkm_uq_unstranded"])

def load_gz_skip_annotation(path):
    with open(path, "rb") as f:
        raw = f.read()
    return pd.read_csv(io.BytesIO(raw), low_memory=False)  # already cached as plain CSV

# ---- rebuild GDC raw matrix from cached extracted files ----
extracted = list(GDC_DIR.rglob("*.tsv")) + list(GDC_DIR.rglob("*.tsv.gz"))
file_id_to_path = {}
for _, row in pair_table.iterrows():
    fid = row["selected_gdc_file_id"]
    match = next((p for p in extracted if fid in str(p)), None)
    if match is None:
        raise FileNotFoundError(f"Cached GDC file for {fid} not found in {GDC_DIR} -- was it deleted?")
    file_id_to_path[fid] = match

gdc_columns = {}
for _, row in pair_table.iterrows():
    df = parse_star_count_file(file_id_to_path[row["selected_gdc_file_id"]])
    df["ensembl_id"] = df["gene_id"].apply(strip_version)
    series = df.set_index("ensembl_id")["unstranded"]
    gdc_columns[row["patient_id"]] = series[~series.index.duplicated(keep="first")]
gdc_matrix = pd.DataFrame(gdc_columns)
print(f"GDC matrix rebuilt from cache: {gdc_matrix.shape}")

# ---- rebuild recount3 raw matrix from cached gene_sums + QC ----
gene_sums = pd.read_csv(OUTDIR / "tcga_gbm_gene_sums.csv", low_memory=False).set_index("gene_id" if "gene_id" in pd.read_csv(OUTDIR / "tcga_gbm_gene_sums.csv", nrows=0).columns else pd.read_csv(OUTDIR / "tcga_gbm_gene_sums.csv", nrows=0).columns[0])
qc = pd.read_csv(OUTDIR / "tcga_gbm_recount3_qc.csv", low_memory=False)
len_col = next(c for c in qc.columns if c.lower().endswith("star.average_mapped_length"))
id_col = next((c for c in qc.columns if "external_id" in c.lower()), qc.columns[0])
avg_len = qc.set_index(id_col)[len_col]

recount3_cols = pair_table["recount3_expression_column"].tolist()
sub = gene_sums[recount3_cols]
lens = avg_len.reindex(recount3_cols)
recount3_counts = sub.div(lens, axis=1).round()
recount3_counts.columns = pair_table["patient_id"].tolist()
recount3_counts.index = [strip_version(g) for g in recount3_counts.index]
recount3_counts = recount3_counts[~recount3_counts.index.duplicated(keep="first")]
print(f"recount3 matrix rebuilt from cache: {recount3_counts.shape}")

# ---- harmonize genes ----
common = sorted(set(gdc_matrix.index) & set(recount3_counts.index))
gdc_matrix = gdc_matrix.loc[common]
recount3_counts = recount3_counts.loc[common]
mt_present = [g for g in MT_GENES if g in common]
print(f"MT genes present in matrix: {len(mt_present)}/15")

# ---- CPM, MT excluded from denominator only ----
def cpm_log2_exclude_mt(mat, mt_genes):
    non_mt = mat.index.difference(mt_genes)
    lib_size = mat.loc[non_mt].sum(axis=0)
    return np.log2(mat.div(lib_size, axis=1) * 1e6 + 1)

gdc_log_cpm = cpm_log2_exclude_mt(gdc_matrix, mt_present)
recount3_log_cpm = cpm_log2_exclude_mt(recount3_counts, mt_present)

# ---- quick before/after check, inline, no separate turn needed ----
def cpm_log2_plain(mat):
    return np.log2(mat.div(mat.sum(axis=0), axis=1) * 1e6 + 1)

gdc_old = cpm_log2_plain(gdc_matrix)
recount3_old = cpm_log2_plain(recount3_counts)
non_mt_idx = [g for g in common if g not in mt_present]

corr_before = [gdc_old[p].loc[non_mt_idx].corr(recount3_old[p].loc[non_mt_idx], method="spearman") for p in gdc_matrix.columns]
corr_after = [gdc_log_cpm[p].loc[non_mt_idx].corr(recount3_log_cpm[p].loc[non_mt_idx], method="spearman") for p in gdc_matrix.columns]

print(f"\nMedian non-MT Spearman BEFORE MT-exclusion: {np.median(corr_before):.4f}")
print(f"Median non-MT Spearman AFTER MT-exclusion:  {np.median(corr_after):.4f}")

gdc_log_cpm.to_csv(OUTDIR / "STAGE_C0_GDC_LOG_CPM_60_MTFIXED.csv")
recount3_log_cpm.to_csv(OUTDIR / "STAGE_C0_RECOUNT3_LOG_CPM_60_MTFIXED.csv")
print(f"\nSaved corrected matrices -> STAGE_C0_*_MTFIXED.csv")
print("MT genes and the calibration gene set for Part 2 should exclude these",
      f"{len(mt_present)} genes -- kept in the matrix for reference, not for calibration fitting.")

### STAGE C0 PART 2 — FINAL HELD-OUT GDC -> recount3 CALIBRATION VALIDATION

In [ ]:
"""
STAGE C0 PART 2 — FINAL HELD-OUT GDC -> recount3 CALIBRATION VALIDATION
======================================================================

Purpose
-------
Test whether GDC expression can be transformed toward the recount3 scale
using the 60 exact paired TCGA bridge samples.

IMPORTANT:
- No survival labels are used.
- The 144 survival patients are NOT used here.
- Calibration is learned ONLY from training bridge patients.
- Performance is measured ONLY on held-out bridge patients.
- MT genes are excluded.
- Repeated CV predictions are aggregated per patient BEFORE statistical tests.

Calibration model
-----------------
Gene-wise location/scale transformation:

    z = (x_GDC - mu_GDC) / sd_GDC

    x_calibrated =
        mu_recount3 + z * sd_recount3

BUT the scale ratio is shrunk toward 1 to avoid unstable corrections
for low-variance genes.

Evaluation
----------
Three gene universes:

1. all_non_MT
2. Stage-B network genes
3. DEG anchors

For every held-out patient:
    RMSE before / after
    Pearson before / after
    Spearman before / after

After repeated CV:
    average repeated held-out results per patient
    -> exactly 60 patient-level observations

Statistics:
    paired Wilcoxon
    bootstrap 95% CI
    percentage of patients improved

Final decision emphasizes:
    Stage-B network genes
    DEG anchors
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedKFold

from scipy.stats import (
    wilcoxon,
)


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_A_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_A_DISCOVERY"
)

CAL_DIR = (
    STAGE_A_DIR
    / "STAGE_C0_BRIDGE_CALIBRATION"
)

STAGE_B_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
)

STRING_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
)


GDC_FILE = (
    CAL_DIR
    / "STAGE_C0_GDC_LOG_CPM_60_MTFIXED.csv"
)

RECOUNT3_FILE = (
    CAL_DIR
    / "STAGE_C0_RECOUNT3_LOG_CPM_60_MTFIXED.csv"
)

STAGE_B_NODES_FILE = (
    STAGE_B_DIR
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)

FULL_MAPPING_FILE = (
    STRING_DIR
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)


# ======================================================================
# OUTPUTS
# ======================================================================

RAW_CV_OUT = (
    CAL_DIR
    / "STAGE_C0_PART2_RAW_HELDOUT_RESULTS.csv"
)

PATIENT_CV_OUT = (
    CAL_DIR
    / "STAGE_C0_PART2_PATIENT_LEVEL_RESULTS.csv"
)

SUMMARY_OUT = (
    CAL_DIR
    / "STAGE_C0_PART2_FINAL_SUMMARY.csv"
)

FINAL_PARAMS_OUT = (
    CAL_DIR
    / "STAGE_C0_FROZEN_CALIBRATION_PARAMS.csv"
)


# ======================================================================
# SETTINGS
# ======================================================================

N_SPLITS = 5
N_REPEATS = 10

RANDOM_SEED = 42


# ------------------------------------------------------------
# Calibration stability parameters
# ------------------------------------------------------------

# Genes with extremely small GDC variation should not be allowed
# to generate huge z-scores.
MIN_SD = 0.15

# Shrink scale ratio toward 1.
#
# Larger = more conservative.
# With ~48 training samples per fold:
#
# lambda = n / (n + SHRINKAGE_STRENGTH)
#
# so with 48 and strength=20:
# lambda ~ 0.706
#
SHRINKAGE_STRENGTH = 20.0


# ------------------------------------------------------------
# Bootstrap
# ------------------------------------------------------------

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 123


# ------------------------------------------------------------
# Practical interpretation thresholds
#
# These are NOT used alone as a magical hard rule.
# They help classify the evidence at the end.
# ------------------------------------------------------------

MIN_PATIENT_IMPROVEMENT_FRACTION = 0.60

MIN_RMSE_IMPROVEMENT_PCT = 5.0


# ======================================================================
# MT GENES
# ======================================================================

MT_GENES = {
    "ENSG00000198888",
    "ENSG00000198763",
    "ENSG00000198804",
    "ENSG00000198712",
    "ENSG00000228253",
    "ENSG00000198899",
    "ENSG00000198938",
    "ENSG00000198840",
    "ENSG00000212907",
    "ENSG00000198886",
    "ENSG00000198786",
    "ENSG00000198695",
    "ENSG00000198727",
    "ENSG00000211459",
    "ENSG00000210082",
}


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(condition, message):

    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: {message}"
        )


def strip_version(value):

    return (
        str(value)
        .split(".")[0]
        .strip()
    )


# ======================================================================
# SAFE CORRELATION
# ======================================================================

def safe_corr(x, y, method="pearson"):

    x = pd.Series(
        np.asarray(x, dtype=float)
    )

    y = pd.Series(
        np.asarray(y, dtype=float)
    )

    valid = (
        x.notna()
        &
        y.notna()
    )

    x = x[
        valid
    ]

    y = y[
        valid
    ]

    if len(x) < 20:
        return np.nan

    if x.std() == 0:
        return np.nan

    if y.std() == 0:
        return np.nan

    return float(
        x.corr(
            y,
            method=method
        )
    )


def calculate_rmse(x, y):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    valid = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    if valid.sum() == 0:
        return np.nan

    return float(
        np.sqrt(
            np.mean(
                (
                    x[valid]
                    -
                    y[valid]
                )
                ** 2
            )
        )
    )


# ======================================================================
# BOOTSTRAP MEDIAN CI
# ======================================================================

def bootstrap_median_ci(
    values,
    n_bootstrap=5000,
    seed=123
):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(
            values
        )
    ]

    if len(values) == 0:

        return (
            np.nan,
            np.nan
        )


    rng = np.random.default_rng(
        seed
    )

    medians = []


    for _ in range(
        n_bootstrap
    ):

        sample = rng.choice(
            values,
            size=len(values),
            replace=True
        )

        medians.append(
            np.median(
                sample
            )
        )


    return (
        float(
            np.percentile(
                medians,
                2.5
            )
        ),
        float(
            np.percentile(
                medians,
                97.5
            )
        )
    )


# ======================================================================
# CALIBRATION FIT
# ======================================================================

def fit_calibration(
    gdc_train,
    recount_train
):
    """
    Fit calibration using TRAIN samples only.

    Returns:
        mu_gdc
        mu_recount
        stable GDC SD
        shrunk scale ratio

    Scale ratio:
        raw_ratio = sd_recount / sd_gdc

    Then shrink:
        ratio_shrunk =
            1 + lambda * (raw_ratio - 1)

    where:
        lambda = n_train / (n_train + SHRINKAGE_STRENGTH)

    So uncertain scale corrections are pulled toward identity (=1).
    """

    n_train = (
        gdc_train.shape[1]
    )


    mu_gdc = (
        gdc_train.mean(
            axis=1
        )
    )

    mu_recount = (
        recount_train.mean(
            axis=1
        )
    )


    sd_gdc_raw = (
        gdc_train.std(
            axis=1,
            ddof=1
        )
    )

    sd_recount_raw = (
        recount_train.std(
            axis=1,
            ddof=1
        )
    )


    # ----------------------------------------------------------
    # Stabilize tiny variances
    # ----------------------------------------------------------

    sd_gdc = (
        sd_gdc_raw
        .clip(
            lower=MIN_SD
        )
    )

    sd_recount = (
        sd_recount_raw
        .clip(
            lower=MIN_SD
        )
    )


    raw_ratio = (
        sd_recount
        /
        sd_gdc
    )


    # ----------------------------------------------------------
    # Extra protection from extreme ratios
    # ----------------------------------------------------------

    raw_ratio = (
        raw_ratio.clip(
            lower=0.25,
            upper=4.0
        )
    )


    # ----------------------------------------------------------
    # Actual shrinkage toward identity
    # ----------------------------------------------------------

    shrink_lambda = (
        n_train
        /
        (
            n_train
            +
            SHRINKAGE_STRENGTH
        )
    )


    shrunk_ratio = (
        1.0
        +
        shrink_lambda
        *
        (
            raw_ratio
            -
            1.0
        )
    )


    params = pd.DataFrame(
        {
            "mu_gdc":
                mu_gdc,

            "sd_gdc_raw":
                sd_gdc_raw,

            "sd_gdc_used":
                sd_gdc,

            "mu_recount3":
                mu_recount,

            "sd_recount3_raw":
                sd_recount_raw,

            "sd_recount3_used":
                sd_recount,

            "raw_scale_ratio":
                raw_ratio,

            "shrunk_scale_ratio":
                shrunk_ratio,

            "shrink_lambda":
                shrink_lambda
        }
    )


    return params


# ======================================================================
# CALIBRATION APPLICATION
# ======================================================================

def apply_calibration(
    gdc_new,
    params
):

    genes = (
        gdc_new.index
        .intersection(
            params.index
        )
    )


    x = (
        gdc_new.loc[
            genes
        ]
    )


    p = (
        params.loc[
            genes
        ]
    )


    z = (
        x
        .sub(
            p[
                "mu_gdc"
            ],
            axis=0
        )
        .div(
            p[
                "sd_gdc_used"
            ],
            axis=0
        )
    )


    calibrated = (
        z
        .mul(
            p[
                "sd_gdc_used"
            ]
            *
            p[
                "shrunk_scale_ratio"
            ],
            axis=0
        )
        .add(
            p[
                "mu_recount3"
            ],
            axis=0
        )
    )


    return calibrated


# ======================================================================
# PATIENT-LEVEL EVALUATION
# ======================================================================

def evaluate_patient(
    before,
    after,
    actual,
    genes,
    patient_id,
    subset_name,
    repeat,
    fold
):

    genes = [
        gene
        for gene in genes
        if (
            gene in before.index
            and
            gene in after.index
            and
            gene in actual.index
        )
    ]


    if len(genes) < 50:

        raise RuntimeError(
            f"Too few genes for evaluation: "
            f"{subset_name} = {len(genes)}"
        )


    x_before = (
        before.loc[
            genes,
            patient_id
        ]
        .values
    )


    x_after = (
        after.loc[
            genes,
            patient_id
        ]
        .values
    )


    y = (
        actual.loc[
            genes,
            patient_id
        ]
        .values
    )


    rmse_before = calculate_rmse(
        x_before,
        y
    )

    rmse_after = calculate_rmse(
        x_after,
        y
    )


    pearson_before = safe_corr(
        x_before,
        y,
        method="pearson"
    )

    pearson_after = safe_corr(
        x_after,
        y,
        method="pearson"
    )


    spearman_before = safe_corr(
        x_before,
        y,
        method="spearman"
    )

    spearman_after = safe_corr(
        x_after,
        y,
        method="spearman"
    )


    return {
        "repeat":
            repeat,

        "fold":
            fold,

        "patient_id":
            patient_id,

        "gene_subset":
            subset_name,

        "n_genes":
            len(genes),

        "rmse_before":
            rmse_before,

        "rmse_after":
            rmse_after,

        "delta_rmse":
            rmse_after
            -
            rmse_before,

        "rmse_improvement_pct":
            100
            *
            (
                rmse_before
                -
                rmse_after
            )
            /
            rmse_before,

        "pearson_before":
            pearson_before,

        "pearson_after":
            pearson_after,

        "delta_pearson":
            pearson_after
            -
            pearson_before,

        "spearman_before":
            spearman_before,

        "spearman_after":
            spearman_after,

        "delta_spearman":
            spearman_after
            -
            spearman_before
    }


# ======================================================================
# STEP 1 — LOAD MATRICES
# ======================================================================

section(
    "STEP 1 — LOAD MT-FIXED BRIDGE MATRICES"
)


checkpoint(
    GDC_FILE.exists(),
    "GDC MT-fixed matrix exists"
)

checkpoint(
    RECOUNT3_FILE.exists(),
    "recount3 MT-fixed matrix exists"
)


gdc = pd.read_csv(
    GDC_FILE,
    index_col=0
)

recount3 = pd.read_csv(
    RECOUNT3_FILE,
    index_col=0
)


checkpoint(
    gdc.shape
    ==
    recount3.shape,
    f"matrix shapes match: {gdc.shape}"
)


checkpoint(
    list(gdc.index)
    ==
    list(recount3.index),
    "gene order matches"
)


checkpoint(
    list(gdc.columns)
    ==
    list(recount3.columns),
    "patient order matches"
)


checkpoint(
    gdc.shape[1]
    ==
    60,
    "60 exact bridge patients"
)


patients = list(
    gdc.columns
)


calibration_genes = [
    gene
    for gene
    in gdc.index
    if gene
    not in MT_GENES
]


print(
    "Total genes:",
    len(
        gdc.index
    )
)

print(
    "Calibration genes after MT exclusion:",
    len(
        calibration_genes
    )
)


checkpoint(
    len(
        calibration_genes
    )
    >
    50000,
    "large calibration gene universe retained"
)


# ======================================================================
# STEP 2 — LOAD STAGE-B SUBSETS
# ======================================================================

section(
    "STEP 2 — LOAD STAGE-B NETWORK / DEG ANCHORS"
)


checkpoint(
    STAGE_B_NODES_FILE.exists(),
    "Stage-B node file exists"
)

checkpoint(
    FULL_MAPPING_FILE.exists(),
    "full STRING mapping file exists"
)


stage_b_nodes = pd.read_csv(
    STAGE_B_NODES_FILE,
    low_memory=False
)

mapping = pd.read_csv(
    FULL_MAPPING_FILE,
    low_memory=False
)


checkpoint(
    "string_protein_id"
    in stage_b_nodes.columns,
    "Stage-B node table has STRING IDs"
)


checkpoint(
    "Is_DEG_Anchor"
    in stage_b_nodes.columns,
    "Stage-B node table has anchor flag"
)


checkpoint(
    "string_protein_id"
    in mapping.columns,
    "mapping contains STRING IDs"
)


# ------------------------------------------------------------
# Determine Ensembl column robustly
# ------------------------------------------------------------

ensembl_candidates = [
    c
    for c
    in mapping.columns
    if "ensembl"
    in c.lower()
]


if len(
    ensembl_candidates
) == 0:

    raise RuntimeError(
        "No Ensembl column found in STRING mapping.\n"
        f"Columns: {mapping.columns.tolist()}"
    )


ensembl_col = (
    ensembl_candidates[0]
)


mapping[
    "ensembl_base"
] = (
    mapping[
        ensembl_col
    ]
    .apply(
        strip_version
    )
)


# deterministic mapping:
# one STRING protein -> first unique Ensembl base ID

protein_to_gene = (
    mapping
    .dropna(
        subset=[
            "string_protein_id",
            "ensembl_base"
        ]
    )
    .drop_duplicates(
        subset="string_protein_id",
        keep="first"
    )
    .set_index(
        "string_protein_id"
    )[
        "ensembl_base"
    ]
    .to_dict()
)


stage_b_nodes[
    "ensembl_id"
] = (
    stage_b_nodes[
        "string_protein_id"
    ]
    .map(
        protein_to_gene
    )
)


stage_b_all_genes = sorted(
    set(
        stage_b_nodes[
            "ensembl_id"
        ]
        .dropna()
    )
    &
    set(
        calibration_genes
    )
)


anchor_mask = (
    stage_b_nodes[
        "Is_DEG_Anchor"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            "true",
            "1",
            "yes"
        ]
    )
)


stage_b_deg_anchors = sorted(
    set(
        stage_b_nodes.loc[
            anchor_mask,
            "ensembl_id"
        ]
        .dropna()
    )
    &
    set(
        calibration_genes
    )
)


print(
    "Stage-B network genes available:",
    len(
        stage_b_all_genes
    )
)


print(
    "Stage-B DEG anchors available:",
    len(
        stage_b_deg_anchors
    )
)


checkpoint(
    len(
        stage_b_all_genes
    )
    >
    5000,
    "substantial Stage-B network gene set available"
)


checkpoint(
    len(
        stage_b_deg_anchors
    )
    >
    1000,
    "substantial DEG-anchor set available"
)


gene_subsets = {
    "all_non_MT":
        calibration_genes,

    "stage_B_network":
        stage_b_all_genes,

    "DEG_anchors":
        stage_b_deg_anchors,
}


# ======================================================================
# STEP 3 — REPEATED 5-FOLD CV
# ======================================================================

section(
    "STEP 3 — REPEATED HELD-OUT CALIBRATION"
)


cv = RepeatedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_SEED
)


patient_array = np.array(
    patients
)


raw_results = []


total_splits = (
    N_SPLITS
    *
    N_REPEATS
)


for split_number, (
    train_idx,
    test_idx
) in enumerate(
    cv.split(
        patient_array
    ),
    start=1
):


    repeat_number = (
        (
            split_number
            -
            1
        )
        //
        N_SPLITS
        +
        1
    )


    fold_number = (
        (
            split_number
            -
            1
        )
        %
        N_SPLITS
        +
        1
    )


    train_patients = list(
        patient_array[
            train_idx
        ]
    )

    test_patients = list(
        patient_array[
            test_idx
        ]
    )


    gdc_train = (
        gdc.loc[
            calibration_genes,
            train_patients
        ]
    )


    recount_train = (
        recount3.loc[
            calibration_genes,
            train_patients
        ]
    )


    params = fit_calibration(
        gdc_train,
        recount_train
    )


    gdc_test = (
        gdc.loc[
            calibration_genes,
            test_patients
        ]
    )


    actual_test = (
        recount3.loc[
            calibration_genes,
            test_patients
        ]
    )


    calibrated_test = (
        apply_calibration(
            gdc_test,
            params
        )
    )


    for patient_id in test_patients:

        for subset_name, genes in (
            gene_subsets.items()
        ):

            result = evaluate_patient(
                before=gdc_test,
                after=calibrated_test,
                actual=actual_test,
                genes=genes,
                patient_id=patient_id,
                subset_name=subset_name,
                repeat=repeat_number,
                fold=fold_number
            )

            raw_results.append(
                result
            )


    print(
        f"Split {split_number}/{total_splits} "
        f"complete "
        f"(repeat {repeat_number}, fold {fold_number})"
    )


raw_df = pd.DataFrame(
    raw_results
)


raw_df.to_csv(
    RAW_CV_OUT,
    index=False
)


print(
    "\nRaw held-out result rows:",
    len(
        raw_df
    )
)


# ======================================================================
# STEP 4 — VERIFY REPEATED TEST COVERAGE
# ======================================================================

section(
    "STEP 4 — VERIFY PATIENT TEST COVERAGE"
)


coverage = (
    raw_df
    .groupby(
        [
            "gene_subset",
            "patient_id"
        ]
    )
    .size()
)


print(
    coverage.value_counts()
)


checkpoint(
    (
        coverage
        ==
        N_REPEATS
    ).all(),
    (
        f"every patient appears in held-out evaluation "
        f"exactly {N_REPEATS} times per gene subset"
    )
)


# ======================================================================
# STEP 5 — COLLAPSE REPEATS TO 60 PATIENT-LEVEL RESULTS
# ======================================================================

section(
    "STEP 5 — AGGREGATE TO PATIENT-LEVEL RESULTS"
)


numeric_metrics = [
    "n_genes",
    "rmse_before",
    "rmse_after",
    "delta_rmse",
    "rmse_improvement_pct",
    "pearson_before",
    "pearson_after",
    "delta_pearson",
    "spearman_before",
    "spearman_after",
    "delta_spearman"
]


patient_df = (
    raw_df
    .groupby(
        [
            "gene_subset",
            "patient_id"
        ],
        as_index=False
    )[
        numeric_metrics
    ]
    .mean()
)


patient_df.to_csv(
    PATIENT_CV_OUT,
    index=False
)


for subset in gene_subsets:

    subset_n = (
        patient_df[
            patient_df[
                "gene_subset"
            ]
            ==
            subset
        ][
            "patient_id"
        ]
        .nunique()
    )


    checkpoint(
        subset_n == 60,
        f"{subset}: exactly 60 independent patient-level results"
    )


# ======================================================================
# STEP 6 — STATISTICAL SUMMARY
# ======================================================================

section(
    "STEP 6 — PATIENT-LEVEL VALIDATION SUMMARY"
)


summary_rows = []


for subset_name in gene_subsets:

    df = (
        patient_df[
            patient_df[
                "gene_subset"
            ]
            ==
            subset_name
        ]
        .copy()
    )


    # ----------------------------------------------------------
    # RMSE
    # ----------------------------------------------------------

    rmse_before_med = (
        df[
            "rmse_before"
        ]
        .median()
    )

    rmse_after_med = (
        df[
            "rmse_after"
        ]
        .median()
    )


    rmse_improvement_pct = (
        100
        *
        (
            rmse_before_med
            -
            rmse_after_med
        )
        /
        rmse_before_med
    )


    rmse_improved_fraction = (
        (
            df[
                "rmse_after"
            ]
            <
            df[
                "rmse_before"
            ]
        )
        .mean()
    )


    rmse_delta = (
        df[
            "rmse_after"
        ]
        -
        df[
            "rmse_before"
        ]
    )


    try:

        rmse_wilcoxon = (
            wilcoxon(
                df[
                    "rmse_before"
                ],
                df[
                    "rmse_after"
                ],
                alternative="greater"
            )
        )

        rmse_p = (
            rmse_wilcoxon.pvalue
        )

    except ValueError:

        rmse_p = np.nan


    rmse_ci_low, rmse_ci_high = (
        bootstrap_median_ci(
            -rmse_delta.values,
            n_bootstrap=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED
        )
    )


    # ----------------------------------------------------------
    # Pearson
    # ----------------------------------------------------------

    pearson_before_med = (
        df[
            "pearson_before"
        ]
        .median()
    )

    pearson_after_med = (
        df[
            "pearson_after"
        ]
        .median()
    )


    pearson_delta = (
        df[
            "pearson_after"
        ]
        -
        df[
            "pearson_before"
        ]
    )


    pearson_improved_fraction = (
        (
            pearson_delta
            >
            0
        )
        .mean()
    )


    try:

        pearson_wilcoxon = (
            wilcoxon(
                df[
                    "pearson_after"
                ],
                df[
                    "pearson_before"
                ],
                alternative="greater"
            )
        )

        pearson_p = (
            pearson_wilcoxon.pvalue
        )

    except ValueError:

        pearson_p = np.nan


    pearson_ci_low, pearson_ci_high = (
        bootstrap_median_ci(
            pearson_delta.values,
            n_bootstrap=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED
            +
            1
        )
    )


    # ----------------------------------------------------------
    # Spearman
    # ----------------------------------------------------------

    spearman_before_med = (
        df[
            "spearman_before"
        ]
        .median()
    )

    spearman_after_med = (
        df[
            "spearman_after"
        ]
        .median()
    )


    spearman_delta = (
        df[
            "spearman_after"
        ]
        -
        df[
            "spearman_before"
        ]
    )


    spearman_improved_fraction = (
        (
            spearman_delta
            >
            0
        )
        .mean()
    )


    try:

        spearman_wilcoxon = (
            wilcoxon(
                df[
                    "spearman_after"
                ],
                df[
                    "spearman_before"
                ],
                alternative="greater"
            )
        )

        spearman_p = (
            spearman_wilcoxon.pvalue
        )

    except ValueError:

        spearman_p = np.nan


    spearman_ci_low, spearman_ci_high = (
        bootstrap_median_ci(
            spearman_delta.values,
            n_bootstrap=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED
            +
            2
        )
    )


    summary_rows.append(
        {
            "gene_subset":
                subset_name,

            "n_patients":
                len(df),

            "n_genes":
                int(
                    round(
                        df[
                            "n_genes"
                        ].median()
                    )
                ),

            "rmse_before_median":
                rmse_before_med,

            "rmse_after_median":
                rmse_after_med,

            "rmse_improvement_pct":
                rmse_improvement_pct,

            "fraction_patients_rmse_improved":
                rmse_improved_fraction,

            "rmse_improvement_median_absolute":
                (
                    -rmse_delta
                ).median(),

            "rmse_improvement_CI_low":
                rmse_ci_low,

            "rmse_improvement_CI_high":
                rmse_ci_high,

            "rmse_wilcoxon_p":
                rmse_p,

            "pearson_before_median":
                pearson_before_med,

            "pearson_after_median":
                pearson_after_med,

            "pearson_delta_median":
                pearson_delta.median(),

            "fraction_patients_pearson_improved":
                pearson_improved_fraction,

            "pearson_delta_CI_low":
                pearson_ci_low,

            "pearson_delta_CI_high":
                pearson_ci_high,

            "pearson_wilcoxon_p":
                pearson_p,

            "spearman_before_median":
                spearman_before_med,

            "spearman_after_median":
                spearman_after_med,

            "spearman_delta_median":
                spearman_delta.median(),

            "fraction_patients_spearman_improved":
                spearman_improved_fraction,

            "spearman_delta_CI_low":
                spearman_ci_low,

            "spearman_delta_CI_high":
                spearman_ci_high,

            "spearman_wilcoxon_p":
                spearman_p
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


summary_df.to_csv(
    SUMMARY_OUT,
    index=False
)


pd.set_option(
    "display.max_columns",
    None
)


print(
    summary_df.to_string(
        index=False
    )
)


# ======================================================================
# STEP 7 — DECISION
# ======================================================================

section(
    "STEP 7 — CALIBRATION DECISION"
)


network_row = (
    summary_df[
        summary_df[
            "gene_subset"
        ]
        ==
        "stage_B_network"
    ]
    .iloc[0]
)


anchor_row = (
    summary_df[
        summary_df[
            "gene_subset"
        ]
        ==
        "DEG_anchors"
    ]
    .iloc[0]
)


# ------------------------------------------------------------
# Primary conditions
# ------------------------------------------------------------

network_rmse_good = (
    network_row[
        "rmse_improvement_pct"
    ]
    >=
    MIN_RMSE_IMPROVEMENT_PCT
)


anchor_rmse_good = (
    anchor_row[
        "rmse_improvement_pct"
    ]
    >=
    MIN_RMSE_IMPROVEMENT_PCT
)


network_majority = (
    network_row[
        "fraction_patients_rmse_improved"
    ]
    >=
    MIN_PATIENT_IMPROVEMENT_FRACTION
)


anchor_majority = (
    anchor_row[
        "fraction_patients_rmse_improved"
    ]
    >=
    MIN_PATIENT_IMPROVEMENT_FRACTION
)


network_rmse_significant = (
    network_row[
        "rmse_wilcoxon_p"
    ]
    <
    0.05
)


anchor_rmse_significant = (
    anchor_row[
        "rmse_wilcoxon_p"
    ]
    <
    0.05
)


network_pearson_not_worse = (
    network_row[
        "pearson_delta_median"
    ]
    >=
    -0.005
)


anchor_pearson_not_worse = (
    anchor_row[
        "pearson_delta_median"
    ]
    >=
    -0.005
)


print(
    "Stage-B network:"
)

print(
    f"  RMSE improvement = "
    f"{network_row['rmse_improvement_pct']:.2f}%"
)

print(
    f"  Patients with RMSE improvement = "
    f"{100*network_row['fraction_patients_rmse_improved']:.1f}%"
)

print(
    f"  RMSE Wilcoxon p = "
    f"{network_row['rmse_wilcoxon_p']:.4g}"
)

print(
    f"  Pearson median delta = "
    f"{network_row['pearson_delta_median']:+.4f}"
)


print(
    "\nDEG anchors:"
)

print(
    f"  RMSE improvement = "
    f"{anchor_row['rmse_improvement_pct']:.2f}%"
)

print(
    f"  Patients with RMSE improvement = "
    f"{100*anchor_row['fraction_patients_rmse_improved']:.1f}%"
)

print(
    f"  RMSE Wilcoxon p = "
    f"{anchor_row['rmse_wilcoxon_p']:.4g}"
)

print(
    f"  Pearson median delta = "
    f"{anchor_row['pearson_delta_median']:+.4f}"
)


accepted = all(
    [
        network_rmse_good,
        anchor_rmse_good,

        network_majority,
        anchor_majority,

        network_rmse_significant,
        anchor_rmse_significant,

        network_pearson_not_worse,
        anchor_pearson_not_worse
    ]
)


# ======================================================================
# STEP 8 — IF ACCEPTED, FIT FINAL FROZEN CALIBRATION
# ======================================================================

if accepted:

    print(
        "\n"
        "**************************************************"
    )

    print(
        "*** CALIBRATION ACCEPTED ***"
    )

    print(
        "**************************************************"
    )


    print(
        "\nHeld-out validation supports calibration."
    )


    print(
        "\nFitting final parameters using all 60 exact bridge pairs..."
    )


    final_params = fit_calibration(
        gdc.loc[
            calibration_genes,
            patients
        ],
        recount3.loc[
            calibration_genes,
            patients
        ]
    )


    final_params[
        "is_MT_gene"
    ] = False


    final_params[
        "is_stage_B_network_gene"
    ] = (
        final_params.index.isin(
            stage_b_all_genes
        )
    )


    final_params[
        "is_DEG_anchor"
    ] = (
        final_params.index.isin(
            stage_b_deg_anchors
        )
    )


    final_params.to_csv(
        FINAL_PARAMS_OUT
    )


    print(
        "\nFrozen parameters saved:"
    )

    print(
        FINAL_PARAMS_OUT
    )


    print(
        "\nGenes in frozen calibration:",
        len(
            final_params
        )
    )


    print(
        "\nNEXT STEP:"
    )

    print(
        "Apply these frozen parameters to the "
        "144 GDC survival patients."
    )


    print(
        "Then construct the GTEx healthy reference "
        "and personalized rewiring matrix."
    )


else:

    print(
        "\n"
        "**************************************************"
    )

    print(
        "*** CALIBRATION REJECTED ***"
    )

    print(
        "**************************************************"
    )


    print(
        "\nThe current GDC -> recount3 transformation "
        "did not satisfy the pre-defined held-out criteria."
    )


    print(
        "\nDo NOT apply it to the 144 survival patients."
    )


    print(
        "\nFailed criteria:"
    )


    criteria = {
        "Stage-B RMSE >= 5% improvement":
            network_rmse_good,

        "DEG-anchor RMSE >= 5% improvement":
            anchor_rmse_good,

        "Stage-B >=60% patients RMSE improved":
            network_majority,

        "DEG-anchor >=60% patients RMSE improved":
            anchor_majority,

        "Stage-B RMSE Wilcoxon p<0.05":
            network_rmse_significant,

        "DEG-anchor RMSE Wilcoxon p<0.05":
            anchor_rmse_significant,

        "Stage-B Pearson not materially worse":
            network_pearson_not_worse,

        "DEG-anchor Pearson not materially worse":
            anchor_pearson_not_worse,
    }


    for criterion, passed in (
        criteria.items()
    ):

        print(
            f"  "
            f"{'PASS' if passed else 'FAIL'}"
            f" — {criterion}"
        )


# ======================================================================
# FINAL FILES
# ======================================================================

section(
    "FILES CREATED"
)


print(
    RAW_CV_OUT
)

print(
    PATIENT_CV_OUT
)

print(
    SUMMARY_OUT
)


if accepted:

    print(
        FINAL_PARAMS_OUT
    )


section(
    "STAGE C0 PART 2 COMPLETE"
)

### STAGE C0 PART 3

In [ ]:
"""
STAGE C0 PART 3
APPLY FROZEN GDC -> recount3 CALIBRATION TO THE 144 SURVIVAL PATIENTS
===================================================================

Purpose
-------
Take the FINAL 144 TCGA-GBM GDC STAR-Counts samples and transform their
expression onto the scale learned from the independent 60-patient
GDC <-> recount3 bridge.

IMPORTANT
---------
- The 144 survival labels are NOT used.
- Calibration is NOT refit.
- The frozen parameters learned from the independent 60 bridge samples
  are used exactly as saved.
- The same normalization used during bridge validation is reproduced:
      raw GDC unstranded counts
      -> mitochondrial genes excluded from CPM denominator
      -> log2(CPM + 1)
      -> frozen GDC -> recount3 calibration

Outputs
-------
1. 144-patient pre-calibration log2(CPM+1) matrix
2. 144-patient calibrated expression matrix
3. Stage-B network-only calibrated matrix
4. DEG-anchor-only calibrated matrix
5. sample/file audit table

The calibrated 144-patient matrix is the input for personalized rewiring.
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path

import gzip
import io
import re
import tarfile
import shutil
import time

import numpy as np
import pandas as pd
import requests


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE


# ----------------------------------------------------------------------
# Main 144-patient inventory
# ----------------------------------------------------------------------

INVENTORY_FILE = (
    DOWNLOADS
    / "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv"
)


# ----------------------------------------------------------------------
# Calibration directory
# ----------------------------------------------------------------------

CAL_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_A_DISCOVERY"
    / "STAGE_C0_BRIDGE_CALIBRATION"
)


FROZEN_PARAMS_FILE = (
    CAL_DIR
    / "STAGE_C0_FROZEN_CALIBRATION_PARAMS.csv"
)


# ----------------------------------------------------------------------
# Stage-B data
# ----------------------------------------------------------------------

STAGE_B_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
)


STAGE_B_NODES_FILE = (
    STAGE_B_DIR
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)


# ----------------------------------------------------------------------
# Output directory
# ----------------------------------------------------------------------

OUTDIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_144_CALIBRATED"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True
)


GDC_DOWNLOAD_DIR = (
    OUTDIR
    / "gdc_144_star_files"
)

GDC_DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# OUTPUT FILES
# ======================================================================

PRECAL_OUT = (
    OUTDIR
    / "STAGE_C_144_GDC_LOG_CPM_MTFIXED.csv"
)


CALIBRATED_OUT = (
    OUTDIR
    / "STAGE_C_144_CALIBRATED_RECOUNT3_SCALE.csv"
)


NETWORK_OUT = (
    OUTDIR
    / "STAGE_C_144_CALIBRATED_STAGE_B_NETWORK.csv"
)


ANCHOR_OUT = (
    OUTDIR
    / "STAGE_C_144_CALIBRATED_DEG_ANCHORS.csv"
)


AUDIT_OUT = (
    OUTDIR
    / "STAGE_C_144_CALIBRATION_AUDIT.csv"
)


SUMMARY_OUT = (
    OUTDIR
    / "STAGE_C_144_CALIBRATION_SUMMARY.csv"
)


# ======================================================================
# SETTINGS
# ======================================================================

BATCH_SIZE = 15

MAX_RETRIES = 4

RETRY_SECONDS = 5


# ======================================================================
# MITOCHONDRIAL GENES
# Same 15 genes excluded during bridge calibration
# ======================================================================

MT_GENES = {
    "ENSG00000198888",
    "ENSG00000198763",
    "ENSG00000198804",
    "ENSG00000198712",
    "ENSG00000228253",
    "ENSG00000198899",
    "ENSG00000198938",
    "ENSG00000198840",
    "ENSG00000212907",
    "ENSG00000198886",
    "ENSG00000198786",
    "ENSG00000198695",
    "ENSG00000198727",
    "ENSG00000211459",
    "ENSG00000210082",
}


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(condition, message):

    status = "PASS" if condition else "FAIL"

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: {message}"
        )


def strip_version(value):

    return (
        str(value)
        .split(".")[0]
        .strip()
    )


def chunk_list(values, size):

    for start in range(
        0,
        len(values),
        size
    ):

        yield values[
            start:
            start + size
        ]


# ======================================================================
# GDC STAR PARSER
# ======================================================================

def parse_star_count_file(path):
    """
    Parse a GDC STAR augmented gene-count file.

    Finds the first ENSG row automatically so summary rows are ignored.
    """

    opener = (
        gzip.open
        if str(path).endswith(".gz")
        else open
    )


    with opener(
        path,
        "rt"
    ) as handle:

        lines = handle.readlines()


    data_start = None


    for i, line in enumerate(lines):

        first_value = (
            line
            .split("\t")[0]
            .strip()
        )

        if re.match(
            r"^ENSG\d+",
            first_value
        ):

            data_start = i
            break


    if data_start is None:

        raise RuntimeError(
            f"No ENSG gene rows found in:\n{path}"
        )


    df = pd.read_csv(
        io.StringIO(
            "".join(
                lines[data_start:]
            )
        ),
        sep="\t",
        header=None,
        names=[
            "gene_id",
            "gene_name",
            "gene_type",
            "unstranded",
            "stranded_first",
            "stranded_second",
            "tpm_unstranded",
            "fpkm_unstranded",
            "fpkm_uq_unstranded"
        ]
    )


    df[
        "unstranded"
    ] = pd.to_numeric(
        df[
            "unstranded"
        ],
        errors="coerce"
    )


    return df


# ======================================================================
# TAR VALIDATION
# ======================================================================

def tar_is_valid(path):

    try:

        with tarfile.open(
            path,
            "r:gz"
        ) as archive:

            members = archive.getmembers()

            return len(members) > 0

    except (
        EOFError,
        OSError,
        tarfile.ReadError
    ):

        return False


# ======================================================================
# MT-EXCLUDED CPM NORMALIZATION
# ======================================================================

def log2_cpm_excluding_mt(
    matrix,
    mt_genes
):
    """
    IMPORTANT:
    MT genes are excluded from the library-size denominator.

    The MT rows themselves may remain temporarily in the matrix, but they
    are subsequently excluded because the frozen calibration contains only
    non-MT genes.
    """

    matrix = matrix.apply(
        pd.to_numeric,
        errors="coerce"
    )


    non_mt = [
        gene
        for gene
        in matrix.index
        if gene not in mt_genes
    ]


    library_size = (
        matrix.loc[
            non_mt
        ]
        .sum(
            axis=0
        )
    )


    checkpoint(
        (
            library_size
            >
            0
        ).all(),
        "all 144 samples have positive non-MT library size"
    )


    cpm = (
        matrix
        .div(
            library_size,
            axis=1
        )
        *
        1e6
    )


    return np.log2(
        cpm + 1
    )


# ======================================================================
# FROZEN CALIBRATION APPLICATION
# ======================================================================

def apply_frozen_calibration(
    gdc_log_cpm,
    params
):
    """
    Apply EXACTLY the transformation validated in Stage C0 Part 2.

    z =
       (GDC - mu_GDC) / sd_GDC_used

    calibrated =
       z *
       (sd_GDC_used * shrunk_scale_ratio)
       +
       mu_recount3
    """

    genes = (
        gdc_log_cpm.index
        .intersection(
            params.index
        )
    )


    x = (
        gdc_log_cpm.loc[
            genes
        ]
        .copy()
    )


    p = (
        params.loc[
            genes
        ]
    )


    z = (
        x
        .sub(
            p[
                "mu_gdc"
            ],
            axis=0
        )
        .div(
            p[
                "sd_gdc_used"
            ],
            axis=0
        )
    )


    calibrated = (
        z
        .mul(
            (
                p[
                    "sd_gdc_used"
                ]
                *
                p[
                    "shrunk_scale_ratio"
                ]
            ),
            axis=0
        )
        .add(
            p[
                "mu_recount3"
            ],
            axis=0
        )
    )


    return calibrated


# ======================================================================
# STEP 1 — LOAD FINAL 144-PATIENT INVENTORY
# ======================================================================

section(
    "STEP 1 — LOAD FINAL 144-PATIENT SURVIVAL COHORT"
)


checkpoint(
    INVENTORY_FILE.exists(),
    "144-patient inventory file exists"
)


inventory = pd.read_csv(
    INVENTORY_FILE,
    sep="\t",
    low_memory=False
)


print(
    "Inventory shape:",
    inventory.shape
)


required_columns = [
    "Patient ID",
    "Selected File ID",
    "Selected File Name"
]


missing_columns = [
    c
    for c
    in required_columns
    if c not in inventory.columns
]


if missing_columns:

    raise RuntimeError(
        "Missing required inventory columns:\n"
        f"{missing_columns}\n\n"
        f"Available:\n"
        f"{inventory.columns.tolist()}"
    )


checkpoint(
    len(inventory) == 144,
    f"inventory contains exactly 144 patients ({len(inventory)})"
)


checkpoint(
    inventory[
        "Patient ID"
    ].nunique()
    ==
    144,
    "144 unique patient IDs"
)


checkpoint(
    inventory[
        "Selected File ID"
    ].notna().all(),
    "all 144 patients have a selected RNA file ID"
)


checkpoint(
    inventory[
        "Selected File ID"
    ].nunique()
    ==
    144,
    "one unique selected GDC file per patient"
)


patient_order = (
    inventory[
        "Patient ID"
    ]
    .astype(str)
    .tolist()
)


file_ids = (
    inventory[
        "Selected File ID"
    ]
    .astype(str)
    .tolist()
)


print(
    "Patients:",
    len(
        patient_order
    )
)


print(
    "Selected GDC files:",
    len(
        file_ids
    )
)


# ======================================================================
# STEP 2 — LOAD FROZEN CALIBRATION
# ======================================================================

section(
    "STEP 2 — LOAD FROZEN BRIDGE CALIBRATION"
)


checkpoint(
    FROZEN_PARAMS_FILE.exists(),
    "frozen calibration parameter file exists"
)


params = pd.read_csv(
    FROZEN_PARAMS_FILE,
    index_col=0
)


required_param_columns = [
    "mu_gdc",
    "sd_gdc_used",
    "mu_recount3",
    "shrunk_scale_ratio"
]


missing_param_columns = [
    c
    for c
    in required_param_columns
    if c not in params.columns
]


if missing_param_columns:

    raise RuntimeError(
        "Frozen calibration parameters are missing columns:\n"
        f"{missing_param_columns}\n\n"
        f"Available:\n"
        f"{params.columns.tolist()}"
    )


print(
    "Frozen calibration genes:",
    len(
        params
    )
)


checkpoint(
    len(
        params
    )
    ==
    57547,
    (
        "frozen calibration contains the expected "
        f"57,547 non-MT genes ({len(params)})"
    )
)


mt_in_params = (
    set(
        params.index
    )
    &
    MT_GENES
)


checkpoint(
    len(
        mt_in_params
    )
    ==
    0,
    "no mitochondrial calibration genes are present"
)


# ======================================================================
# STEP 3 — FIND ALREADY-CACHED GDC FILES
# ======================================================================

section(
    "STEP 3 — FIND / DOWNLOAD 144 SELECTED GDC STAR FILES"
)


# Search Downloads broadly first in case RNA files already exist locally.
search_roots = [
    GDC_DOWNLOAD_DIR,
    DOWNLOADS
]


file_id_to_path = {}


print(
    "Searching existing files first..."
)


for root in search_roots:

    if not root.exists():
        continue


    candidates = (
        list(
            root.rglob(
                "*.tsv"
            )
        )
        +
        list(
            root.rglob(
                "*.tsv.gz"
            )
        )
    )


    for path in candidates:

        path_text = str(
            path
        )


        for file_id in file_ids:

            if (
                file_id not in file_id_to_path
                and
                file_id in path_text
            ):

                file_id_to_path[
                    file_id
                ] = path


print(
    "Already found locally:",
    len(
        file_id_to_path
    ),
    "/144"
)


missing_file_ids = [
    file_id
    for file_id
    in file_ids
    if file_id
    not in file_id_to_path
]


print(
    "Need download:",
    len(
        missing_file_ids
    )
)


# ======================================================================
# STEP 4 — ROBUST DOWNLOAD OF ONLY MISSING FILES
# ======================================================================

if missing_file_ids:

    batches = list(
        chunk_list(
            missing_file_ids,
            BATCH_SIZE
        )
    )


    print(
        f"\nDownloading {len(missing_file_ids)} files "
        f"in {len(batches)} batches."
    )


    for batch_number, batch_ids in enumerate(
        batches,
        start=1
    ):


        batch_tar = (
            GDC_DOWNLOAD_DIR
            /
            f"batch_{batch_number:03d}.tar.gz"
        )


        batch_folder = (
            GDC_DOWNLOAD_DIR
            /
            f"batch_{batch_number:03d}"
        )


        valid_existing = (
            batch_tar.exists()
            and
            tar_is_valid(
                batch_tar
            )
        )


        if not valid_existing:

            if batch_tar.exists():

                batch_tar.unlink()


            success = False


            for attempt in range(
                1,
                MAX_RETRIES + 1
            ):


                print(
                    f"Batch {batch_number}/{len(batches)}, "
                    f"attempt {attempt}/{MAX_RETRIES}..."
                )


                try:

                    response = requests.post(
                        "https://api.gdc.cancer.gov/data",
                        json={
                            "ids":
                                batch_ids
                        },
                        headers={
                            "Content-Type":
                                "application/json"
                        },
                        stream=True,
                        timeout=600
                    )


                    response.raise_for_status()


                    with open(
                        batch_tar,
                        "wb"
                    ) as handle:


                        for chunk in response.iter_content(
                            chunk_size=1024 * 1024
                        ):

                            if chunk:

                                handle.write(
                                    chunk
                                )


                    if tar_is_valid(
                        batch_tar
                    ):

                        print(
                            "  valid archive — "
                            f"{batch_tar.stat().st_size/1e6:.2f} MB"
                        )

                        success = True
                        break


                    print(
                        "  archive invalid/truncated"
                    )

                    batch_tar.unlink(
                        missing_ok=True
                    )


                except Exception as error:

                    print(
                        "  failed:",
                        repr(
                            error
                        )
                    )


                    batch_tar.unlink(
                        missing_ok=True
                    )


                if attempt < MAX_RETRIES:

                    time.sleep(
                        RETRY_SECONDS
                    )


            if not success:

                raise RuntimeError(
                    f"Could not obtain valid batch {batch_number}"
                )


        # --------------------------------------------------------------
        # Extract
        # --------------------------------------------------------------

        if batch_folder.exists():

            shutil.rmtree(
                batch_folder
            )


        batch_folder.mkdir(
            parents=True,
            exist_ok=True
        )


        with tarfile.open(
            batch_tar,
            "r:gz"
        ) as archive:

            archive.extractall(
                batch_folder
            )


# ======================================================================
# STEP 5 — MAP ALL 144 FILE IDS TO ACTUAL FILES
# ======================================================================

section(
    "STEP 5 — VERIFY ALL 144 GDC FILES"
)


all_candidates = (
    list(
        GDC_DOWNLOAD_DIR.rglob(
            "*.tsv"
        )
    )
    +
    list(
        GDC_DOWNLOAD_DIR.rglob(
            "*.tsv.gz"
        )
    )
)


# also include whatever was already found
for path in all_candidates:

    path_text = str(
        path
    )


    for file_id in file_ids:

        if (
            file_id not in file_id_to_path
            and
            file_id in path_text
        ):

            file_id_to_path[
                file_id
            ] = path


checkpoint(
    len(
        file_id_to_path
    )
    ==
    144,
    (
        "all 144 selected GDC STAR files are available "
        f"({len(file_id_to_path)}/144)"
    )
)


# ======================================================================
# STEP 6 — BUILD RAW GDC COUNT MATRIX
# ======================================================================

section(
    "STEP 6 — BUILD 144-PATIENT RAW COUNT MATRIX"
)


gdc_columns = {}

audit_rows = []


for _, row in inventory.iterrows():

    patient_id = str(
        row[
            "Patient ID"
        ]
    )


    file_id = str(
        row[
            "Selected File ID"
        ]
    )


    path = (
        file_id_to_path[
            file_id
        ]
    )


    df = parse_star_count_file(
        path
    )


    df[
        "ensembl_id"
    ] = (
        df[
            "gene_id"
        ]
        .apply(
            strip_version
        )
    )


    duplicate_count = int(
        df[
            "ensembl_id"
        ]
        .duplicated()
        .sum()
    )


    # deterministic duplicate resolution
    df = (
        df
        .sort_values(
            [
                "ensembl_id",
                "unstranded"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop_duplicates(
            subset="ensembl_id",
            keep="first"
        )
    )


    series = (
        df
        .set_index(
            "ensembl_id"
        )[
            "unstranded"
        ]
    )


    gdc_columns[
        patient_id
    ] = series


    protein_coding = (
        df[
            df[
                "gene_type"
            ]
            ==
            "protein_coding"
        ]
    )


    audit_rows.append(
        {
            "patient_id":
                patient_id,

            "selected_file_id":
                file_id,

            "selected_file_name":
                row[
                    "Selected File Name"
                ],

            "local_file":
                str(
                    path
                ),

            "genes_in_file":
                len(
                    df
                ),

            "duplicate_base_ensembl_rows_resolved":
                duplicate_count,

            "protein_coding_detected":
                int(
                    (
                        protein_coding[
                            "unstranded"
                        ]
                        >
                        0
                    )
                    .sum()
                ),

            "protein_coding_counts":
                float(
                    protein_coding[
                        "unstranded"
                    ]
                    .sum()
                )
        }
    )


gdc_raw = pd.DataFrame(
    gdc_columns
)


# force frozen patient order
gdc_raw = (
    gdc_raw[
        patient_order
    ]
)


print(
    "Raw matrix:",
    gdc_raw.shape[0],
    "genes x",
    gdc_raw.shape[1],
    "patients"
)


checkpoint(
    gdc_raw.shape[1]
    ==
    144,
    "raw expression matrix contains all 144 patients"
)


checkpoint(
    list(
        gdc_raw.columns
    )
    ==
    patient_order,
    "patient order matches frozen 144-patient inventory"
)


# ======================================================================
# STEP 7 — SAME NORMALIZATION AS BRIDGE
# ======================================================================

section(
    "STEP 7 — APPLY IDENTICAL MT-EXCLUDED CPM NORMALIZATION"
)


mt_present = (
    set(
        gdc_raw.index
    )
    &
    MT_GENES
)


print(
    "MT genes present:",
    len(
        mt_present
    ),
    "/15"
)


checkpoint(
    len(
        mt_present
    )
    ==
    15,
    "all 15 canonical MT genes are present"
)


gdc_log_cpm = (
    log2_cpm_excluding_mt(
        gdc_raw,
        MT_GENES
    )
)


print(
    "Normalized matrix:",
    gdc_log_cpm.shape
)


# ======================================================================
# STEP 8 — ALIGN TO FROZEN CALIBRATION GENE UNIVERSE
# ======================================================================

section(
    "STEP 8 — ALIGN 144 PATIENTS TO FROZEN CALIBRATION GENES"
)


available_calibration_genes = (
    params.index.intersection(
        gdc_log_cpm.index
    )
)


missing_calibration_genes = (
    params.index.difference(
        gdc_log_cpm.index
    )
)


print(
    "Frozen calibration genes:",
    len(
        params
    )
)


print(
    "Available in 144 GDC matrix:",
    len(
        available_calibration_genes
    )
)


print(
    "Missing:",
    len(
        missing_calibration_genes
    )
)


mapping_rate = (
    100
    *
    len(
        available_calibration_genes
    )
    /
    len(
        params
    )
)


print(
    f"Resolution rate: {mapping_rate:.2f}%"
)


checkpoint(
    mapping_rate
    >
    95,
    (
        "at least 95% of frozen calibration genes "
        f"resolve in the 144 cohort ({mapping_rate:.2f}%)"
    )
)


# Use frozen parameter order
genes_to_use = [
    gene
    for gene
    in params.index
    if gene
    in gdc_log_cpm.index
]


gdc_log_cpm_for_calibration = (
    gdc_log_cpm.loc[
        genes_to_use,
        patient_order
    ]
)


params_use = (
    params.loc[
        genes_to_use
    ]
)


# ======================================================================
# STEP 9 — APPLY FROZEN CALIBRATION
# ======================================================================

section(
    "STEP 9 — APPLY FROZEN GDC -> recount3 CALIBRATION"
)


calibrated_144 = (
    apply_frozen_calibration(
        gdc_log_cpm_for_calibration,
        params_use
    )
)


checkpoint(
    calibrated_144.shape[1]
    ==
    144,
    "calibrated matrix contains all 144 patients"
)


checkpoint(
    list(
        calibrated_144.columns
    )
    ==
    patient_order,
    "calibrated matrix preserves frozen patient order"
)


checkpoint(
    not calibrated_144.isna().any().any(),
    "calibrated expression matrix contains no missing values"
)


checkpoint(
    np.isfinite(
        calibrated_144.values
    ).all(),
    "calibrated matrix contains only finite values"
)


print(
    "Calibrated matrix:",
    calibrated_144.shape[0],
    "genes x",
    calibrated_144.shape[1],
    "patients"
)


# ======================================================================
# STEP 10 — BASIC CALIBRATION SANITY CHECK
# ======================================================================

section(
    "STEP 10 — CALIBRATION SANITY CHECK"
)


delta = (
    calibrated_144
    -
    gdc_log_cpm_for_calibration
)


print(
    "Absolute transformation magnitude:"
)


print(
    np.abs(
        delta.values
    )
    .reshape(-1)
    .mean()
)


print(
    "\nPer-patient median absolute transformation:"
)


patient_shift = (
    delta
    .abs()
    .median(
        axis=0
    )
)


print(
    patient_shift.describe()
)


checkpoint(
    patient_shift.notna().all(),
    "all 144 patients received a valid calibration transformation"
)


# ======================================================================
# STEP 11 — SAVE COMPLETE CALIBRATED MATRIX
# ======================================================================

section(
    "STEP 11 — SAVE 144-PATIENT CALIBRATED MATRICES"
)


gdc_log_cpm_for_calibration.to_csv(
    PRECAL_OUT
)


calibrated_144.to_csv(
    CALIBRATED_OUT
)


print(
    "Saved pre-calibration matrix:"
)

print(
    PRECAL_OUT
)


print(
    "\nSaved calibrated matrix:"
)

print(
    CALIBRATED_OUT
)


# ======================================================================
# STEP 12 — SAVE STAGE-B NETWORK / DEG-ANCHOR MATRICES
# ======================================================================

section(
    "STEP 12 — CREATE DOWNSTREAM STAGE-B GENE MATRICES"
)


if (
    "is_stage_B_network_gene"
    in params_use.columns
):

    network_genes = (
        params_use.index[
            params_use[
                "is_stage_B_network_gene"
            ]
            .astype(bool)
        ]
    )


else:

    print(
        "WARNING: network flag not found in frozen params."
    )

    network_genes = []


if (
    "is_DEG_anchor"
    in params_use.columns
):

    anchor_genes = (
        params_use.index[
            params_use[
                "is_DEG_anchor"
            ]
            .astype(bool)
        ]
    )


else:

    print(
        "WARNING: DEG-anchor flag not found in frozen params."
    )

    anchor_genes = []


print(
    "Stage-B network genes:",
    len(
        network_genes
    )
)


print(
    "DEG anchors:",
    len(
        anchor_genes
    )
)


checkpoint(
    len(
        network_genes
    )
    >
    10000,
    "large Stage-B network gene set available"
)


checkpoint(
    len(
        anchor_genes
    )
    >
    3000,
    "large DEG-anchor gene set available"
)


calibrated_network = (
    calibrated_144.loc[
        network_genes
    ]
)


calibrated_anchors = (
    calibrated_144.loc[
        anchor_genes
    ]
)


calibrated_network.to_csv(
    NETWORK_OUT
)


calibrated_anchors.to_csv(
    ANCHOR_OUT
)


print(
    "\nSaved Stage-B network matrix:"
)

print(
    NETWORK_OUT
)


print(
    "\nSaved DEG-anchor matrix:"
)

print(
    ANCHOR_OUT
)


# ======================================================================
# STEP 13 — AUDIT
# ======================================================================

section(
    "STEP 13 — SAVE AUDIT / SUMMARY"
)


audit_df = pd.DataFrame(
    audit_rows
)


audit_df[
    "median_abs_calibration_shift"
] = (
    audit_df[
        "patient_id"
    ]
    .map(
        patient_shift
    )
)


audit_df.to_csv(
    AUDIT_OUT,
    index=False
)


summary_df = pd.DataFrame(
    [
        {
            "Metric":
                "Survival patients",

            "Value":
                144
        },

        {
            "Metric":
                "Raw GDC genes",

            "Value":
                gdc_raw.shape[0]
        },

        {
            "Metric":
                "Frozen calibration genes",

            "Value":
                len(
                    params
                )
        },

        {
            "Metric":
                "Calibration genes resolved",

            "Value":
                len(
                    genes_to_use
                )
        },

        {
            "Metric":
                "Calibration gene resolution %",

            "Value":
                mapping_rate
        },

        {
            "Metric":
                "Final calibrated genes",

            "Value":
                calibrated_144.shape[0]
        },

        {
            "Metric":
                "Stage-B network genes",

            "Value":
                len(
                    network_genes
                )
        },

        {
            "Metric":
                "DEG anchors",

            "Value":
                len(
                    anchor_genes
                )
        }
    ]
)


summary_df.to_csv(
    SUMMARY_OUT,
    index=False
)


print(
    summary_df.to_string(
        index=False
    )
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "STAGE C0 PART 3 COMPLETE"
)


print(
    "The frozen GDC -> recount3 calibration has been "
    "applied to all 144 survival patients."
)


print(
    "\nNO survival labels were used."
)


print(
    "\nMain output:"
)

print(
    CALIBRATED_OUT
)


print(
    "\nStage-B network matrix:"
)

print(
    NETWORK_OUT
)


print(
    "\nDEG-anchor matrix:"
)

print(
    ANCHOR_OUT
)


print(
    "\nNEXT STEP:"
)


print(
    "Build the 334-GTEx healthy reference network R,"
)


print(
    "then calculate patient-specific networks for all 144 patients,"
)


print(
    "and finally compute the personalized rewiring / eNESH matrix."
)

### MEMORY-SAFE GTEx gene_sums LOADER

In [ ]:
# ======================================================================
# MEMORY-SAFE GTEx gene_sums LOADER
# Reads ONLY the 334 frozen GTEx columns
# ======================================================================

import gzip
import requests
import pandas as pd
from pathlib import Path


def load_gtex_gene_sums_selected_columns(
    url,
    required_expression_columns,
    cache_path=None
):
    """
    Memory-safe loader for recount3 GTEx gene_sums.

    Downloads the compressed file to disk first,
    then reads ONLY:
        - gene ID column
        - the required 334 expression columns

    It does NOT decompress the entire dataset into RAM.
    """

    if cache_path is None:
        cache_path = (
            BASE
            / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
            / "recount3_cache"
            / "gtex_brain_gene_sums.G026.gz"
        )

    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # ----------------------------------------------------------
    # Download compressed file once
    # ----------------------------------------------------------

    if not cache_path.exists():

        print("Downloading large GTEx gene_sums file to disk:")
        print(url)

        with requests.get(
            url,
            stream=True,
            timeout=1200
        ) as response:

            response.raise_for_status()

            with open(
                cache_path,
                "wb"
            ) as handle:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        handle.write(chunk)

        print(
            f"Saved compressed file: "
            f"{cache_path.stat().st_size / 1e6:.1f} MB"
        )

    else:

        print("Using cached compressed gene_sums:")
        print(cache_path)


    # ----------------------------------------------------------
    # Inspect header safely
    # ----------------------------------------------------------

    with gzip.open(
        cache_path,
        "rt"
    ) as handle:

        header_line = None

        for line in handle:

            if line.startswith("#"):
                continue

            header_line = line.rstrip("\n")
            break


    if header_line is None:

        raise RuntimeError(
            "Could not find gene_sums header."
        )


    header_columns = (
        header_line
        .split("\t")
    )


    gene_id_col = header_columns[0]


    print(
        "gene ID column:",
        gene_id_col
    )

    print(
        "Total expression columns in GTEx file:",
        len(header_columns) - 1
    )


    required_expression_columns = list(
        required_expression_columns
    )


    missing = [
        col
        for col in required_expression_columns
        if col not in header_columns
    ]


    if missing:

        raise RuntimeError(
            f"{len(missing)} required GTEx expression "
            "columns are absent from gene_sums.\n"
            f"First missing IDs: {missing[:10]}"
        )


    print(
        f"Required GTEx columns found: "
        f"{len(required_expression_columns)}/"
        f"{len(required_expression_columns)}"
    )


    # ----------------------------------------------------------
    # Read ONLY gene ID + 334 samples
    # ----------------------------------------------------------

    usecols = (
        [gene_id_col]
        +
        required_expression_columns
    )


    print(
        "Reading only selected columns..."
    )


    df = pd.read_csv(
        cache_path,
        compression="gzip",
        sep="\t",
        comment="#",
        usecols=usecols,
        low_memory=False
    )


    print(
        "Selected gene_sums shape:",
        df.shape
    )


    return df

### STAGE C PART 1 — MEMORY-SAFE HEALTHY GTEx REFERENCE NETWORK R

In [ ]:
"""
STAGE C PART 1 — MEMORY-SAFE HEALTHY GTEx REFERENCE NETWORK R
=============================================================

Purpose
-------
Construct the healthy expression-correlation reference network R using:

    334 frozen GTEx cortical controls
    +
    frozen Stage-B DEG-anchored STRING topology

For every retained STRING edge (gene_i, gene_j):

    reference_PCC(i,j)
        = Pearson correlation across the 334 GTEx controls

IMPORTANT
---------
This version fixes the previous MemoryError.

The previous script tried to load the FULL recount3 GTEx-BRAIN gene_sums:
    ~63,856 genes x 2,931 GTEx samples

This corrected version:

1. loads metadata first,
2. resolves only the 334 frozen GTEx sample columns,
3. downloads/caches the compressed gene_sums file on DISK,
4. reads ONLY the gene ID + those 334 required columns.

No full decompression into one giant Python string.

Normalization
-------------
Same recount3-side normalization used in Stage C0:

    estimated counts =
        gene_sums / star.average_mapped_length

then:

    mitochondrial genes excluded from CPM denominator
    log2(CPM + 1)

Outputs
-------
1. STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv
2. STAGE_C_HEALTHY_REFERENCE_NETWORK_R.csv
3. STAGE_C_HEALTHY_REFERENCE_NODE_SUMMARY.csv
4. sample mapping audit
5. final summary

NEXT
----
Stage C Part 2:
    144 calibrated patients
    -> patient-specific edge networks
    -> personalized eNESH rewiring matrix
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path

import gzip
import re

import numpy as np
import pandas as pd
import requests


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_A_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_A_DISCOVERY"
)

STAGE_B_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
)

STRING_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
)

STAGE_C_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

STAGE_C_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CACHE_DIR = (
    STAGE_C_DIR
    / "recount3_cache"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# INPUT FILES
# ======================================================================

GTEX_FINAL_SAMPLE_FILE = (
    STAGE_A_DIR
    / "STAGE_A_GTEX_CORTEX_FINAL_QC.csv"
)


FULL_STRING_MAPPING_FILE = (
    STRING_DIR
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)


STAGE_B_EDGE_CANDIDATES = [
    STAGE_B_DIR
    / "STAGE_B_DEG_ANCHORED_STRING_EDGES.csv",

    STAGE_B_DIR
    / "STAGE_B_DEG_STRING_EDGES.csv",
]


STAGE_B_NODE_CANDIDATES = [
    STAGE_B_DIR
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv",

    STAGE_B_DIR
    / "STAGE_B_ALL_DEG_NETWORK_METRICS.csv",
]


# ======================================================================
# RECOUNT3 URLS
# ======================================================================

RECOUNT3_BASE = (
    "http://duffel.rail.bio/"
    "recount3/human/data_sources"
)

GTEX_PROJECT = "BRAIN"
GTEX_SHARD = "IN"


GTEX_META_URL = (
    f"{RECOUNT3_BASE}/gtex/metadata/"
    f"{GTEX_SHARD}/{GTEX_PROJECT}/"
    f"gtex.gtex.{GTEX_PROJECT}.MD.gz"
)


GTEX_QC_URL = (
    f"{RECOUNT3_BASE}/gtex/metadata/"
    f"{GTEX_SHARD}/{GTEX_PROJECT}/"
    f"gtex.recount_qc.{GTEX_PROJECT}.MD.gz"
)


GTEX_GENE_SUMS_URL = (
    f"{RECOUNT3_BASE}/gtex/gene_sums/"
    f"{GTEX_SHARD}/{GTEX_PROJECT}/"
    f"gtex.gene_sums.{GTEX_PROJECT}.G026.gz"
)


# ======================================================================
# CACHE FILES
# ======================================================================

GTEX_META_CACHE = (
    CACHE_DIR
    / "gtex_brain_metadata.csv"
)


GTEX_QC_CACHE = (
    CACHE_DIR
    / "gtex_brain_qc.csv"
)


# IMPORTANT:
# compressed gene_sums stays compressed on disk
GTEX_GENE_SUMS_GZ_CACHE = (
    CACHE_DIR
    / "gtex_brain_gene_sums.G026.gz"
)


# Optional selected-column cache
GTEX_SELECTED_GENE_SUMS_CACHE = (
    CACHE_DIR
    / "gtex_brain_gene_sums_SELECTED_334.csv"
)


# ======================================================================
# OUTPUTS
# ======================================================================

GTEX_EXPR_OUT = (
    STAGE_C_DIR
    / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
)


REFERENCE_EDGE_OUT = (
    STAGE_C_DIR
    / "STAGE_C_HEALTHY_REFERENCE_NETWORK_R.csv"
)


REFERENCE_NODE_OUT = (
    STAGE_C_DIR
    / "STAGE_C_HEALTHY_REFERENCE_NODE_SUMMARY.csv"
)


SAMPLE_MAPPING_OUT = (
    STAGE_C_DIR
    / "STAGE_C_GTEX_334_EXPRESSION_MAPPING_AUDIT.csv"
)


SUMMARY_OUT = (
    STAGE_C_DIR
    / "STAGE_C_HEALTHY_REFERENCE_SUMMARY.csv"
)


# ======================================================================
# MITOCHONDRIAL GENES
# ======================================================================

MT_GENES = {
    "ENSG00000198888",
    "ENSG00000198763",
    "ENSG00000198804",
    "ENSG00000198712",
    "ENSG00000228253",
    "ENSG00000198899",
    "ENSG00000198938",
    "ENSG00000198840",
    "ENSG00000212907",
    "ENSG00000198886",
    "ENSG00000198786",
    "ENSG00000198695",
    "ENSG00000198727",
    "ENSG00000211459",
    "ENSG00000210082",
}


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(condition, message):

    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: {message}"
        )


def strip_version(value):

    return (
        str(value)
        .split(".")[0]
        .strip()
    )


def clean_string(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    if (
        value == ""
        or
        value.lower() == "nan"
    ):
        return None

    return value


def choose_existing_file(
    candidates,
    description
):

    found = [
        path
        for path in candidates
        if path.exists()
    ]

    if len(found) == 0:

        raise FileNotFoundError(
            f"No {description} found.\n"
            "Tried:\n"
            +
            "\n".join(
                str(path)
                for path in candidates
            )
        )

    print(
        f"{description}:"
    )

    print(
        found[0]
    )

    return found[0]


# ======================================================================
# RECOUNT3 SMALL METADATA PARSER
# ======================================================================

def parse_recount3_small_file(
    raw_bytes
):
    """
    Used only for metadata/QC, which are small.

    Safe to decompress fully.
    """

    text = gzip.decompress(
        raw_bytes
    ).decode(
        "utf-8"
    )

    lines = text.splitlines(
        keepends=True
    )

    start = None

    for i, line in enumerate(lines):

        if not line.startswith("#"):

            start = i
            break

    if start is None:

        raise RuntimeError(
            "Could not find recount3 table header."
        )

    from io import StringIO

    return pd.read_csv(
        StringIO(
            "".join(
                lines[start:]
            )
        ),
        sep="\t",
        low_memory=False
    )


def fetch_small_recount3_table(
    url
):

    print(
        "Downloading:"
    )

    print(
        url
    )

    response = requests.get(
        url,
        timeout=300
    )

    response.raise_for_status()

    return parse_recount3_small_file(
        response.content
    )


def load_or_fetch_small(
    cache_path,
    url
):

    if cache_path.exists():

        print(
            "Using cache:"
        )

        print(
            cache_path
        )

        return pd.read_csv(
            cache_path,
            low_memory=False
        )

    df = fetch_small_recount3_table(
        url
    )

    df.to_csv(
        cache_path,
        index=False
    )

    return df


# ======================================================================
# MEMORY-SAFE GENE_SUMS DOWNLOAD
# ======================================================================

def download_large_file_to_disk(
    url,
    output_path
):
    """
    Stream compressed gene_sums directly to disk.
    Never keeps the whole file in RAM.
    """

    if output_path.exists():

        print(
            "Using cached compressed gene_sums:"
        )

        print(
            output_path
        )

        print(
            f"Size: "
            f"{output_path.stat().st_size / 1e6:.1f} MB"
        )

        return


    print(
        "Downloading large gene_sums file directly to disk:"
    )

    print(
        url
    )


    with requests.get(
        url,
        stream=True,
        timeout=1200
    ) as response:

        response.raise_for_status()

        with open(
            output_path,
            "wb"
        ) as handle:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if chunk:

                    handle.write(
                        chunk
                    )


    print(
        "Compressed gene_sums saved:"
    )

    print(
        output_path
    )

    print(
        f"Size: "
        f"{output_path.stat().st_size / 1e6:.1f} MB"
    )


# ======================================================================
# GENE_SUMS HEADER INSPECTION
# ======================================================================

def inspect_gene_sums_header(
    gz_path
):

    with gzip.open(
        gz_path,
        "rt"
    ) as handle:

        for line in handle:

            if line.startswith("#"):
                continue

            columns = (
                line.rstrip(
                    "\n"
                )
                .split(
                    "\t"
                )
            )

            return columns


    raise RuntimeError(
        "Could not find gene_sums header."
    )


# ======================================================================
# READ ONLY SELECTED 334 COLUMNS
# ======================================================================

def load_selected_gene_sums(
    gz_path,
    required_expression_columns,
    selected_cache=None
):
    """
    Reads ONLY:
        gene ID column
        +
        the selected 334 expression columns

    Avoids loading all 2,931 GTEx samples.
    """

    if (
        selected_cache is not None
        and
        selected_cache.exists()
    ):

        print(
            "Using selected-column gene_sums cache:"
        )

        print(
            selected_cache
        )

        return pd.read_csv(
            selected_cache,
            low_memory=False
        )


    header_columns = (
        inspect_gene_sums_header(
            gz_path
        )
    )


    gene_id_col = (
        header_columns[0]
    )


    print(
        "gene_sums gene-ID column:",
        gene_id_col
    )


    print(
        "Total GTEx expression columns:",
        len(
            header_columns
        )
        -
        1
    )


    required_expression_columns = list(
        required_expression_columns
    )


    missing = [
        column
        for column
        in required_expression_columns
        if column
        not in header_columns
    ]


    checkpoint(
        len(
            missing
        )
        ==
        0,
        (
            "all 334 resolved GTEx expression columns "
            "exist in gene_sums"
        )
    )


    usecols = (
        [
            gene_id_col
        ]
        +
        required_expression_columns
    )


    print(
        "Reading only:"
    )

    print(
        f"  1 gene-ID column"
    )

    print(
        f"  {len(required_expression_columns)} GTEx samples"
    )


    df = pd.read_csv(
        gz_path,
        compression="gzip",
        sep="\t",
        comment="#",
        usecols=usecols,
        low_memory=False
    )


    print(
        "Selected gene_sums shape:",
        df.shape
    )


    if selected_cache is not None:

        df.to_csv(
            selected_cache,
            index=False
        )

        print(
            "Saved selected-column cache:"
        )

        print(
            selected_cache
        )


    return df


# ======================================================================
# READ LENGTH COLUMN
# ======================================================================

def find_average_mapped_length_column(
    df
):

    preferred = [
        "star.average_mapped_length",
        "recount_qc.star.average_mapped_length",
    ]


    for column in preferred:

        if column in df.columns:

            return column


    candidates = [
        column
        for column in df.columns
        if (
            "average_mapped_length"
            in column.lower()
            and
            "length2"
            not in column.lower()
        )
    ]


    if len(candidates) == 1:

        return candidates[0]


    raise RuntimeError(
        "Could not uniquely identify "
        "star.average_mapped_length.\n"
        f"Candidates: {candidates}"
    )


# ======================================================================
# MT-EXCLUDED CPM
# ======================================================================

def log2_cpm_excluding_mt(
    matrix,
    mt_genes
):

    matrix = matrix.apply(
        pd.to_numeric,
        errors="coerce"
    )


    non_mt = [
        gene
        for gene in matrix.index
        if gene not in mt_genes
    ]


    library_size = (
        matrix.loc[
            non_mt
        ]
        .sum(
            axis=0
        )
    )


    checkpoint(
        (
            library_size
            >
            0
        ).all(),
        (
            "all 334 GTEx samples have "
            "positive non-MT library size"
        )
    )


    cpm = (
        matrix
        .div(
            library_size,
            axis=1
        )
        *
        1e6
    )


    return np.log2(
        cpm
        +
        1
    )


# ======================================================================
# STEP 1 — LOAD 334 FROZEN GTEx CONTROLS
# ======================================================================

section(
    "STEP 1 — LOAD FROZEN 334 GTEx CORTICAL CONTROLS"
)


checkpoint(
    GTEX_FINAL_SAMPLE_FILE.exists(),
    "final GTEx control table exists"
)


gtex_final = pd.read_csv(
    GTEX_FINAL_SAMPLE_FILE,
    low_memory=False
)


print(
    "GTEx final table:",
    gtex_final.shape
)


checkpoint(
    len(
        gtex_final
    )
    ==
    334,
    (
        f"exactly 334 frozen GTEx controls "
        f"({len(gtex_final)})"
    )
)


sample_id_candidates = [
    "Recount sample ID",
    "external_id",
    "sample_id",
]


sample_id_col = next(
    (
        column
        for column
        in sample_id_candidates
        if column
        in gtex_final.columns
    ),
    None
)


if sample_id_col is None:

    raise RuntimeError(
        "Could not identify frozen GTEx sample-ID column.\n"
        f"Available:\n"
        f"{gtex_final.columns.tolist()}"
    )


frozen_samples = (
    gtex_final[
        sample_id_col
    ]
    .astype(str)
    .tolist()
)


checkpoint(
    len(
        set(
            frozen_samples
        )
    )
    ==
    334,
    "334 unique frozen GTEx sample IDs"
)


print(
    "Frozen sample-ID column:",
    sample_id_col
)


# ======================================================================
# STEP 2 — LOAD ONLY SMALL recount3 METADATA + QC
# ======================================================================

section(
    "STEP 2 — LOAD recount3 GTEx METADATA AND QC"
)


gtex_meta = load_or_fetch_small(
    GTEX_META_CACHE,
    GTEX_META_URL
)


gtex_qc = load_or_fetch_small(
    GTEX_QC_CACHE,
    GTEX_QC_URL
)


print(
    "Metadata:",
    gtex_meta.shape
)


print(
    "QC:",
    gtex_qc.shape
)


# ======================================================================
# STEP 3 — RESOLVE 334 FROZEN IDs BEFORE LOADING gene_sums
# ======================================================================

section(
    "STEP 3 — RESOLVE 334 GTEx EXPRESSION COLUMN IDs"
)


possible_meta_ids = [
    column
    for column in [
        "external_id",
        "rail_id",
        "run",
        "run_acc",
        "sample",
        "SAMPID",
    ]
    if column
    in gtex_meta.columns
]


print(
    "Metadata identifier columns:",
    possible_meta_ids
)


checkpoint(
    len(
        possible_meta_ids
    )
    >
    0,
    "GTEx metadata contains usable identifiers"
)


# We do not yet know gene_sums columns.
# First inspect metadata and build candidate identifiers.

sample_candidate_ids = {}


for frozen_id in frozen_samples:

    candidate_ids = {
        str(
            frozen_id
        ).strip()
    }


    mask = pd.Series(
        False,
        index=gtex_meta.index
    )


    for column in possible_meta_ids:

        values = (
            gtex_meta[
                column
            ]
            .astype(str)
            .str.strip()
        )


        mask |= (
            values
            ==
            str(
                frozen_id
            ).strip()
        )


    matched_rows = (
        gtex_meta[
            mask
        ]
    )


    for _, row in matched_rows.iterrows():

        for column in possible_meta_ids:

            value = clean_string(
                row.get(
                    column
                )
            )

            if value is not None:

                candidate_ids.add(
                    value
                )


    sample_candidate_ids[
        frozen_id
    ] = sorted(
        candidate_ids
    )


print(
    "Metadata candidate IDs prepared for 334 samples."
)


# ======================================================================
# STEP 4 — DOWNLOAD LARGE gene_sums COMPRESSED TO DISK
# ======================================================================

section(
    "STEP 4 — DOWNLOAD/CACHE GTEx gene_sums SAFELY"
)


download_large_file_to_disk(
    GTEX_GENE_SUMS_URL,
    GTEX_GENE_SUMS_GZ_CACHE
)


# ======================================================================
# STEP 5 — INSPECT gene_sums HEADER ONLY
# ======================================================================

section(
    "STEP 5 — INSPECT gene_sums HEADER"
)


gene_sums_header = (
    inspect_gene_sums_header(
        GTEX_GENE_SUMS_GZ_CACHE
    )
)


gene_id_col = (
    gene_sums_header[0]
)


expression_columns = set(
    gene_sums_header[
        1:
    ]
)


print(
    "Gene-ID column:",
    gene_id_col
)


print(
    "Expression columns in gene_sums:",
    len(
        expression_columns
    )
)


# ======================================================================
# STEP 6 — RESOLVE EXACTLY ONE gene_sums COLUMN PER GTEx SAMPLE
# ======================================================================

section(
    "STEP 6 — MAP 334 FROZEN GTEx SAMPLES TO gene_sums COLUMNS"
)


resolved_columns = {}

mapping_audit = []


for frozen_id in frozen_samples:

    candidates = [
        candidate
        for candidate
        in sample_candidate_ids[
            frozen_id
        ]
        if candidate
        in expression_columns
    ]


    candidates = list(
        dict.fromkeys(
            candidates
        )
    )


    if len(
        candidates
    ) != 1:

        raise RuntimeError(
            f"\nGTEx sample {frozen_id!r}: "
            f"expected exactly one gene_sums column, "
            f"found {len(candidates)}.\n"
            f"Candidates: {candidates}\n"
            f"Metadata IDs tried: "
            f"{sample_candidate_ids[frozen_id]}"
        )


    resolved_column = (
        candidates[0]
    )


    resolved_columns[
        frozen_id
    ] = (
        resolved_column
    )


    mapping_audit.append(
        {
            "frozen_sample_id":
                frozen_id,

            "expression_column":
                resolved_column
        }
    )


checkpoint(
    len(
        resolved_columns
    )
    ==
    334,
    "all 334 GTEx controls resolved"
)


checkpoint(
    len(
        set(
            resolved_columns.values()
        )
    )
    ==
    334,
    "334 unique gene_sums expression columns"
)


pd.DataFrame(
    mapping_audit
).to_csv(
    SAMPLE_MAPPING_OUT,
    index=False
)


print(
    "Sample mapping audit saved:"
)

print(
    SAMPLE_MAPPING_OUT
)


# ======================================================================
# STEP 7 — READ ONLY 334 gene_sums COLUMNS
# ======================================================================

section(
    "STEP 7 — MEMORY-SAFE LOAD OF 334 GTEx EXPRESSION COLUMNS"
)


ordered_expression_columns = [
    resolved_columns[
        sample_id
    ]
    for sample_id
    in frozen_samples
]


gene_sums_selected = (
    load_selected_gene_sums(
        GTEX_GENE_SUMS_GZ_CACHE,
        ordered_expression_columns,
        selected_cache=GTEX_SELECTED_GENE_SUMS_CACHE
    )
)


checkpoint(
    gene_sums_selected.shape[1]
    ==
    335,
    (
        "selected gene_sums contains "
        "1 gene-ID column + 334 samples"
    )
)


checkpoint(
    gene_sums_selected.shape[0]
    >
    50000,
    "selected gene_sums has plausible gene count"
)


# ======================================================================
# STEP 8 — RESOLVE GTEx AVERAGE MAPPED READ LENGTH
# ======================================================================

section(
    "STEP 8 — RESOLVE AVERAGE MAPPED READ LENGTHS"
)


length_col = (
    find_average_mapped_length_column(
        gtex_qc
    )
)


print(
    "Mapped-length column:",
    length_col
)


possible_qc_ids = [
    column
    for column in [
        "external_id",
        "rail_id",
        "run",
        "run_acc",
        "sample",
        "SAMPID",
    ]
    if column
    in gtex_qc.columns
]


print(
    "QC identifier columns:",
    possible_qc_ids
)


checkpoint(
    len(
        possible_qc_ids
    )
    >
    0,
    "GTEx QC contains usable identifiers"
)


sample_lengths = {}


for frozen_id, expression_col in (
    resolved_columns.items()
):

    valid_ids = set(
        sample_candidate_ids[
            frozen_id
        ]
    )

    valid_ids.add(
        str(
            expression_col
        )
    )


    mask = pd.Series(
        False,
        index=gtex_qc.index
    )


    for column in possible_qc_ids:

        values = (
            gtex_qc[
                column
            ]
            .astype(str)
            .str.strip()
        )


        mask |= values.isin(
            valid_ids
        )


    matches = (
        gtex_qc[
            mask
        ]
    )


    if len(
        matches
    ) == 0:

        raise RuntimeError(
            f"No QC row found for "
            f"{frozen_id} / {expression_col}"
        )


    lengths = (
        pd.to_numeric(
            matches[
                length_col
            ],
            errors="coerce"
        )
        .dropna()
        .unique()
    )


    if len(
        lengths
    ) != 1:

        raise RuntimeError(
            f"{frozen_id}: expected one mapped length, "
            f"found {lengths.tolist()}"
        )


    sample_lengths[
        expression_col
    ] = float(
        lengths[0]
    )


checkpoint(
    len(
        sample_lengths
    )
    ==
    334,
    "mapped read lengths obtained for all 334 controls"
)


# ======================================================================
# STEP 9 — BUILD 334-SAMPLE COUNT MATRIX
# ======================================================================

section(
    "STEP 9 — BUILD 334-SAMPLE GTEx COUNT MATRIX"
)


gene_sums_selected = (
    gene_sums_selected
    .set_index(
        gene_id_col
    )
)


sub = (
    gene_sums_selected[
        ordered_expression_columns
    ]
    .copy()
)


length_series = (
    pd.Series(
        sample_lengths
    )
    .loc[
        ordered_expression_columns
    ]
)


gtex_counts = (
    sub
    .div(
        length_series,
        axis=1
    )
    .round()
)


gtex_counts.columns = (
    frozen_samples
)


gtex_counts.index = [
    strip_version(
        gene
    )
    for gene
    in gtex_counts.index
]


duplicate_count = int(
    gtex_counts.index
    .duplicated()
    .sum()
)


print(
    "Duplicate base Ensembl IDs:",
    duplicate_count
)


if duplicate_count > 0:

    print(
        "Aggregating duplicate Ensembl IDs by sum..."
    )

    gtex_counts = (
        gtex_counts
        .groupby(
            level=0
        )
        .sum()
    )


print(
    "GTEx count matrix:",
    gtex_counts.shape
)


checkpoint(
    gtex_counts.shape[1]
    ==
    334,
    "count matrix contains 334 controls"
)


# ======================================================================
# STEP 10 — NORMALIZE IDENTICALLY TO STAGE C0
# ======================================================================

section(
    "STEP 10 — APPLY MT-EXCLUDED CPM + LOG2"
)


mt_present = (
    set(
        gtex_counts.index
    )
    &
    MT_GENES
)


print(
    "MT genes present:",
    len(
        mt_present
    ),
    "/15"
)


checkpoint(
    len(
        mt_present
    )
    ==
    15,
    "all 15 canonical MT genes present"
)


gtex_log = (
    log2_cpm_excluding_mt(
        gtex_counts,
        MT_GENES
    )
)


print(
    "Normalized GTEx matrix:",
    gtex_log.shape
)


# ======================================================================
# STEP 11 — LOAD FROZEN STAGE-B TOPOLOGY
# ======================================================================

section(
    "STEP 11 — LOAD FROZEN STAGE-B STRING TOPOLOGY"
)


stage_b_edge_file = (
    choose_existing_file(
        STAGE_B_EDGE_CANDIDATES,
        "Stage-B edge file"
    )
)


stage_b_node_file = (
    choose_existing_file(
        STAGE_B_NODE_CANDIDATES,
        "Stage-B node file"
    )
)


checkpoint(
    FULL_STRING_MAPPING_FILE.exists(),
    "full STRING mapping file exists"
)


edges = pd.read_csv(
    stage_b_edge_file,
    low_memory=False
)


nodes = pd.read_csv(
    stage_b_node_file,
    low_memory=False
)


mapping = pd.read_csv(
    FULL_STRING_MAPPING_FILE,
    low_memory=False
)


print(
    "Edges:",
    edges.shape
)


print(
    "Nodes:",
    nodes.shape
)


print(
    "STRING mapping:",
    mapping.shape
)


# ======================================================================
# STEP 12 — MAP STRING IDS TO ENSEMBL
# ======================================================================

section(
    "STEP 12 — MAP STAGE-B NETWORK TO ENSEMBL"
)


ensembl_candidates = [
    column
    for column
    in mapping.columns
    if "ensembl"
    in column.lower()
]


if len(
    ensembl_candidates
) == 0:

    raise RuntimeError(
        "No Ensembl column found in STRING mapping."
    )


ensembl_col = (
    ensembl_candidates[0]
)


mapping[
    "ensembl_base"
] = (
    mapping[
        ensembl_col
    ]
    .apply(
        strip_version
    )
)


# deliberate deterministic mapping
mapping_clean = (
    mapping
    .dropna(
        subset=[
            "string_protein_id",
            "ensembl_base"
        ]
    )
    .sort_values(
        [
            "string_protein_id",
            "ensembl_base"
        ]
    )
    .drop_duplicates(
        subset="string_protein_id",
        keep="first"
    )
)


protein_to_gene = (
    mapping_clean
    .set_index(
        "string_protein_id"
    )[
        "ensembl_base"
    ]
    .to_dict()
)


possible_edge_pairs = [
    (
        "protein1",
        "protein2"
    ),
    (
        "string_protein_id_1",
        "string_protein_id_2"
    ),
    (
        "node1",
        "node2"
    ),
    (
        "source",
        "target"
    ),
]


edge_cols = None


for col1, col2 in (
    possible_edge_pairs
):

    if (
        col1 in edges.columns
        and
        col2 in edges.columns
    ):

        edge_cols = (
            col1,
            col2
        )

        break


if edge_cols is None:

    raise RuntimeError(
        "Could not identify edge endpoint columns.\n"
        f"Available:\n"
        f"{edges.columns.tolist()}"
    )


protein1_col, protein2_col = (
    edge_cols
)


print(
    "Edge columns:",
    protein1_col,
    protein2_col
)


edges[
    "gene1"
] = (
    edges[
        protein1_col
    ]
    .map(
        protein_to_gene
    )
)


edges[
    "gene2"
] = (
    edges[
        protein2_col
    ]
    .map(
        protein_to_gene
    )
)


before_mapping = len(
    edges
)


edges = (
    edges
    .dropna(
        subset=[
            "gene1",
            "gene2"
        ]
    )
    .copy()
)


print(
    "Edges mapped to Ensembl:",
    len(
        edges
    ),
    "/",
    before_mapping
)


# ======================================================================
# STEP 13 — ALIGN NETWORK WITH GTEx
# ======================================================================

section(
    "STEP 13 — ALIGN STAGE-B NETWORK WITH GTEx"
)


available_genes = set(
    gtex_log.index
)


reference_edges = (
    edges[
        edges[
            "gene1"
        ].isin(
            available_genes
        )
        &
        edges[
            "gene2"
        ].isin(
            available_genes
        )
    ]
    .copy()
)


print(
    "Mapped Stage-B edges:",
    len(
        edges
    )
)


print(
    "Edges resolvable in GTEx:",
    len(
        reference_edges
    )
)


resolution_rate = (
    100
    *
    len(
        reference_edges
    )
    /
    len(
        edges
    )
)


print(
    f"Edge resolution rate: "
    f"{resolution_rate:.2f}%"
)


checkpoint(
    resolution_rate
    >
    95,
    (
        "at least 95% Stage-B edges resolve "
        f"in GTEx ({resolution_rate:.2f}%)"
    )
)


network_genes = sorted(
    set(
        reference_edges[
            "gene1"
        ]
    )
    |
    set(
        reference_edges[
            "gene2"
        ]
    )
)


print(
    "Reference network genes:",
    len(
        network_genes
    )
)


# ======================================================================
# STEP 14 — NETWORK-ONLY GTEx MATRIX
# ======================================================================

section(
    "STEP 14 — BUILD NETWORK-ONLY GTEx MATRIX"
)


gtex_network = (
    gtex_log.loc[
        network_genes,
        frozen_samples
    ]
)


gtex_network.to_csv(
    GTEX_EXPR_OUT
)


print(
    "GTEx network matrix:",
    gtex_network.shape
)


print(
    "Saved:"
)

print(
    GTEX_EXPR_OUT
)


# ======================================================================
# STEP 15 — CALCULATE HEALTHY PCC
# ======================================================================

section(
    "STEP 15 — CALCULATE HEALTHY REFERENCE PCC"
)


gene_to_row = {
    gene:
        index
    for index, gene
    in enumerate(
        gtex_network.index
    )
}


X = (
    gtex_network
    .values
    .astype(
        np.float64
    )
)


means = (
    X.mean(
        axis=1,
        keepdims=True
    )
)


sd = (
    X.std(
        axis=1,
        ddof=1,
        keepdims=True
    )
)


zero_variance_mask = (
    sd[
        :,
        0
    ]
    ==
    0
)


print(
    "Zero-variance genes:",
    int(
        zero_variance_mask.sum()
    )
)


sd_safe = (
    sd.copy()
)


sd_safe[
    sd_safe
    ==
    0
] = np.nan


Z = (
    X
    -
    means
) / sd_safe


gene1_idx = np.array(
    [
        gene_to_row[
            gene
        ]
        for gene
        in reference_edges[
            "gene1"
        ]
    ],
    dtype=int
)


gene2_idx = np.array(
    [
        gene_to_row[
            gene
        ]
        for gene
        in reference_edges[
            "gene2"
        ]
    ],
    dtype=int
)


N_CONTROLS = (
    gtex_network.shape[1]
)


reference_pcc = (
    np.nansum(
        Z[
            gene1_idx
        ]
        *
        Z[
            gene2_idx
        ],
        axis=1
    )
    /
    (
        N_CONTROLS
        -
        1
    )
)


reference_edges[
    "reference_PCC"
] = (
    reference_pcc
)


reference_edges[
    "reference_abs_PCC"
] = (
    np.abs(
        reference_edges[
            "reference_PCC"
        ]
    )
)


reference_edges[
    "reference_sign"
] = np.where(
    reference_edges[
        "reference_PCC"
    ]
    >
    0,
    1,
    np.where(
        reference_edges[
            "reference_PCC"
        ]
        <
        0,
        -1,
        0
    )
)


reference_edges[
    "n_GTEx_controls"
] = (
    N_CONTROLS
)


# ======================================================================
# STEP 16 — PCC SANITY CHECK
# ======================================================================

section(
    "STEP 16 — REFERENCE NETWORK SANITY CHECK"
)


print(
    reference_edges[
        "reference_PCC"
    ].describe()
)


valid_fraction = (
    reference_edges[
        "reference_PCC"
    ]
    .notna()
    .mean()
)


print(
    f"Valid PCC fraction: "
    f"{100*valid_fraction:.2f}%"
)


checkpoint(
    reference_edges[
        "reference_PCC"
    ]
    .dropna()
    .between(
        -1,
        1
    )
    .all(),
    "all valid PCC values are within [-1,1]"
)


checkpoint(
    valid_fraction
    >
    0.99,
    "at least 99% of edges have valid healthy PCC"
)


print(
    "\nReference edge signs:"
)


print(
    reference_edges[
        "reference_sign"
    ]
    .value_counts()
    .sort_index()
)


# ======================================================================
# STEP 17 — NODE SUMMARY
# ======================================================================

section(
    "STEP 17 — BUILD REFERENCE NODE SUMMARY"
)


left = (
    reference_edges[
        [
            "gene1",
            "gene2",
            "reference_PCC"
        ]
    ]
    .rename(
        columns={
            "gene1":
                "gene",

            "gene2":
                "neighbor"
        }
    )
)


right = (
    reference_edges[
        [
            "gene1",
            "gene2",
            "reference_PCC"
        ]
    ]
    .rename(
        columns={
            "gene2":
                "gene",

            "gene1":
                "neighbor"
        }
    )
)


long_edges = pd.concat(
    [
        left,
        right
    ],
    ignore_index=True
)


node_summary = (
    long_edges
    .groupby(
        "gene"
    )
    .agg(
        reference_degree=(
            "neighbor",
            "nunique"
        ),

        reference_mean_PCC=(
            "reference_PCC",
            "mean"
        ),

        reference_mean_abs_PCC=(
            "reference_PCC",
            lambda values:
                np.mean(
                    np.abs(
                        values
                    )
                )
        ),

        positive_neighbors=(
            "reference_PCC",
            lambda values:
                int(
                    (
                        values
                        >
                        0
                    )
                    .sum()
                )
        ),

        negative_neighbors=(
            "reference_PCC",
            lambda values:
                int(
                    (
                        values
                        <
                        0
                    )
                    .sum()
                )
        )
    )
    .reset_index()
)


node_summary.to_csv(
    REFERENCE_NODE_OUT,
    index=False
)


# ======================================================================
# STEP 18 — SAVE HEALTHY REFERENCE R
# ======================================================================

section(
    "STEP 18 — SAVE HEALTHY REFERENCE NETWORK R"
)


reference_edges.to_csv(
    REFERENCE_EDGE_OUT,
    index=False
)


print(
    "Reference network:"
)

print(
    REFERENCE_EDGE_OUT
)


print(
    "\nReference node summary:"
)

print(
    REFERENCE_NODE_OUT
)


# ======================================================================
# STEP 19 — FINAL SUMMARY
# ======================================================================

section(
    "STEP 19 — FINAL REFERENCE SUMMARY"
)


summary = pd.DataFrame(
    [
        {
            "Metric":
                "Frozen GTEx cortical controls",

            "Value":
                334
        },

        {
            "Metric":
                "GTEx count genes",

            "Value":
                gtex_counts.shape[0]
        },

        {
            "Metric":
                "Normalized GTEx genes",

            "Value":
                gtex_log.shape[0]
        },

        {
            "Metric":
                "Stage-B mapped edges",

            "Value":
                len(
                    edges
                )
        },

        {
            "Metric":
                "Healthy reference edges",

            "Value":
                len(
                    reference_edges
                )
        },

        {
            "Metric":
                "Edge resolution %",

            "Value":
                resolution_rate
        },

        {
            "Metric":
                "Reference network genes",

            "Value":
                len(
                    network_genes
                )
        },

        {
            "Metric":
                "Positive edges",

            "Value":
                int(
                    (
                        reference_edges[
                            "reference_PCC"
                        ]
                        >
                        0
                    )
                    .sum()
                )
        },

        {
            "Metric":
                "Negative edges",

            "Value":
                int(
                    (
                        reference_edges[
                            "reference_PCC"
                        ]
                        <
                        0
                    )
                    .sum()
                )
        },

        {
            "Metric":
                "Median abs PCC",

            "Value":
                float(
                    reference_edges[
                        "reference_abs_PCC"
                    ]
                    .median()
                )
        }
    ]
)


summary.to_csv(
    SUMMARY_OUT,
    index=False
)


print(
    summary.to_string(
        index=False
    )
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "STAGE C PART 1 COMPLETE"
)


print(
    "Healthy GTEx reference network R is ready."
)


print(
    "\n334-sample normalized GTEx network matrix:"
)

print(
    GTEX_EXPR_OUT
)


print(
    "\nHealthy reference network:"
)

print(
    REFERENCE_EDGE_OUT
)


print(
    "\nNEXT:"
)


print(
    "Stage C Part 2 — compute patient-specific networks "
    "and personalized eNESH rewiring scores for all 144 patients."
)

### STAGE C PART 2 — FINAL PERSONALIZED NETWORK REWIRING

In [ ]:
"""
STAGE C PART 2 — FINAL PERSONALIZED NETWORK REWIRING
====================================================

PURPOSE
-------
Compute personalized single-sample network rewiring for all 144
TCGA-GBM survival patients relative to the frozen 334-GTEx healthy
cortical reference.

INPUTS ALREADY CREATED
----------------------
1. Healthy GTEx expression matrix:
   STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv

2. Healthy STRING-constrained reference edge universe:
   STAGE_C_HEALTHY_REFERENCE_NETWORK_R.csv

3. 144 calibrated GDC patients:
   STAGE_C_144_CALIBRATED_RECOUNT3_SCALE.csv

4. Stage-B DEG-anchored network nodes:
   STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv

METHOD
------
A. Clean/freeze common STRING topology
B. Recompute healthy PCC exactly
C. Compute LIONESS/sPCC for every patient:

       sPCC_p = (N+1)*PCC_(N+1) - N*PCC_N

D. Pool all disease single-sample edge weights and standardize:

       z = (sPCC - global_mean) / global_sd

E. Patient network:
       keep edges with |z| >= zcrit

   Default:
       alpha = 0.05
       zcrit ≈ 1.96

F. Reference network:
       retain exactly K reference edges,
       where K = mean number of retained patient edges.

   Reference edges are ranked by absolute standardized reference
   correlation magnitude.

G. Split reference/patient networks into positive/negative subnetworks.

H. Compute eNESH for every DEG anchor and patient.

IMPORTANT
---------
This script produces PERSONALIZED eNESH FEATURES.

It does NOT yet claim "driver genes".

Full ssNetShift driver identification additionally requires:
    - background significance of eNESH
    - positive differential betweenness ΔB

Those can be performed later on the shortlisted genes if needed.

NO SURVIVAL LABELS ARE USED IN THIS SCRIPT.
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path

import json
import numpy as np
import pandas as pd

from scipy.stats import norm


# ======================================================================
# PATHS — THESE MATCH THE FILES YOU ACTUALLY CREATED
# ======================================================================

DOWNLOADS = BASE


STAGE_C_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

STAGE_C_DIR.mkdir(
    parents=True,
    exist_ok=True
)


STAGE_B_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
)


STRING_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
)


CALIBRATED_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_144_CALIBRATED"
)


# ----------------------------------------------------------------------
# REAL Stage-C Part-1 outputs
# ----------------------------------------------------------------------

REFERENCE_EDGES_FILE = (
    STAGE_C_DIR
    / "STAGE_C_HEALTHY_REFERENCE_NETWORK_R.csv"
)


GTEX_EXPR_FILE = (
    STAGE_C_DIR
    / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
)


# ----------------------------------------------------------------------
# REAL calibrated 144-patient matrix
# ----------------------------------------------------------------------

CALIBRATED_144_FILE = (
    CALIBRATED_DIR
    / "STAGE_C_144_CALIBRATED_RECOUNT3_SCALE.csv"
)


# ----------------------------------------------------------------------
# Stage-B node / mapping files
# ----------------------------------------------------------------------

STAGE_B_NODES_FILE = (
    STAGE_B_DIR
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)


FULL_STRING_MAPPING_FILE = (
    STRING_DIR
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)


# ======================================================================
# OUTPUTS
# ======================================================================

FINAL_EDGE_UNIVERSE_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
)


SPARSIFIED_REFERENCE_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_SPARSIFIED_REFERENCE_R.csv"
)


SPARSITY_PARAMETERS_OUT = (
    STAGE_C_DIR
    / "STAGE_C_SPARSITY_PARAMETERS.json"
)


PATIENT_NETWORK_SUMMARY_OUT = (
    STAGE_C_DIR
    / "STAGE_C_PATIENT_NETWORK_SPARSITY_SUMMARY.csv"
)


REWIRING_MATRIX_OUT = (
    STAGE_C_DIR
    / "STAGE_C_PERSONALIZED_ENESH_144xDEG_ANCHORS.csv"
)


REWIRING_LONG_OUT = (
    STAGE_C_DIR
    / "STAGE_C_PERSONALIZED_ENESH_LONG.csv"
)


ANCHOR_SUMMARY_OUT = (
    STAGE_C_DIR
    / "STAGE_C_ENESH_ANCHOR_SUMMARY.csv"
)


FINAL_SUMMARY_OUT = (
    STAGE_C_DIR
    / "STAGE_C_PART2_FINAL_SUMMARY.csv"
)


# Optional binary single-sample edge weights.
# NPZ is much smaller than CSV.
SPCC_NPZ_OUT = (
    STAGE_C_DIR
    / "STAGE_C_SINGLE_SAMPLE_SPCC_144.npz"
)


# ======================================================================
# SETTINGS
# ======================================================================

N_GTEX_EXPECTED = 334
N_PATIENTS_EXPECTED = 144

ALPHA_EDGE = 0.05

RANDOM_SEED = 42

# Validation:
N_BRUTE_FORCE_EDGES = 20

# Save sPCC array?
SAVE_SPCC_NPZ = True


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(condition, message):

    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: {message}"
        )


def strip_version(value):

    return (
        str(value)
        .split(".")[0]
        .strip()
    )


def robust_bool(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
                "y"
            ]
        )
    )


# ======================================================================
# STEP 1 — LOAD EVERYTHING
# ======================================================================

section(
    "STEP 1 — LOAD REFERENCE / GTEx / CALIBRATED 144"
)


for path, description in [

    (
        REFERENCE_EDGES_FILE,
        "healthy reference edge file"
    ),

    (
        GTEX_EXPR_FILE,
        "334-GTEx expression matrix"
    ),

    (
        CALIBRATED_144_FILE,
        "144 calibrated patient matrix"
    ),

    (
        STAGE_B_NODES_FILE,
        "Stage-B node table"
    ),

    (
        FULL_STRING_MAPPING_FILE,
        "full STRING mapping"
    ),
]:

    checkpoint(
        path.exists(),
        f"{description} exists"
    )


ref_edges = pd.read_csv(
    REFERENCE_EDGES_FILE,
    low_memory=False
)


gtex_expr = pd.read_csv(
    GTEX_EXPR_FILE,
    index_col=0
)


patients_expr = pd.read_csv(
    CALIBRATED_144_FILE,
    index_col=0
)


nodes = pd.read_csv(
    STAGE_B_NODES_FILE,
    low_memory=False
)


mapping = pd.read_csv(
    FULL_STRING_MAPPING_FILE,
    low_memory=False
)


# Clean index formats
gtex_expr.index = [
    strip_version(g)
    for g in gtex_expr.index
]


patients_expr.index = [
    strip_version(g)
    for g in patients_expr.index
]


print(
    "Reference edges:",
    ref_edges.shape
)

print(
    "GTEx expression:",
    gtex_expr.shape
)

print(
    "Calibrated patients:",
    patients_expr.shape
)

print(
    "Stage-B nodes:",
    nodes.shape
)


checkpoint(
    gtex_expr.shape[1]
    ==
    N_GTEX_EXPECTED,
    "GTEx matrix contains exactly 334 controls"
)


checkpoint(
    patients_expr.shape[1]
    ==
    N_PATIENTS_EXPECTED,
    "calibrated matrix contains exactly 144 patients"
)


checkpoint(
    {
        "gene1",
        "gene2"
    }.issubset(
        ref_edges.columns
    ),
    "reference edge table contains gene1/gene2"
)


# ======================================================================
# STEP 2 — REMOVE DUPLICATED EXPRESSION IDS IF ANY
# ======================================================================

section(
    "STEP 2 — EXPRESSION-ID SANITY"
)


gtex_duplicates = int(
    pd.Index(
        gtex_expr.index
    )
    .duplicated()
    .sum()
)


patient_duplicates = int(
    pd.Index(
        patients_expr.index
    )
    .duplicated()
    .sum()
)


print(
    "GTEx duplicated gene IDs:",
    gtex_duplicates
)

print(
    "Patient duplicated gene IDs:",
    patient_duplicates
)


checkpoint(
    gtex_duplicates == 0,
    "GTEx network matrix has unique gene IDs"
)


checkpoint(
    patient_duplicates == 0,
    "144-patient calibrated matrix has unique gene IDs"
)


# ======================================================================
# STEP 3 — REMOVE ZERO-VARIANCE GTEx GENES
# ======================================================================

section(
    "STEP 3 — REMOVE ZERO-VARIANCE REFERENCE GENES"
)


gtex_sd = (
    gtex_expr.std(
        axis=1,
        ddof=1
    )
)


zero_var_genes = set(
    gtex_sd.index[
        (
            ~np.isfinite(
                gtex_sd.values
            )
        )
        |
        (
            gtex_sd.values
            <
            1e-12
        )
    ]
)


print(
    "Zero-variance GTEx genes:",
    len(
        zero_var_genes
    )
)


before_zero = len(
    ref_edges
)


ref_edges = (
    ref_edges[
        ~ref_edges[
            "gene1"
        ].isin(
            zero_var_genes
        )
        &
        ~ref_edges[
            "gene2"
        ].isin(
            zero_var_genes
        )
    ]
    .copy()
)


print(
    "Edges removed because of zero variance:",
    before_zero
    -
    len(
        ref_edges
    )
)


checkpoint(
    len(
        ref_edges
    )
    >
    50000,
    "substantial network remains after zero-variance cleanup"
)


# ======================================================================
# STEP 4 — INTERSECT REFERENCE AND 144-PATIENT GENES
# ======================================================================

section(
    "STEP 4 — FREEZE COMMON GENE / EDGE UNIVERSE"
)


common_genes = (

    set(
        gtex_expr.index
    )

    &

    set(
        patients_expr.index
    )

) - zero_var_genes


print(
    "Common usable genes:",
    len(
        common_genes
    )
)


before_overlap = len(
    ref_edges
)


ref_edges = (
    ref_edges[
        ref_edges[
            "gene1"
        ].isin(
            common_genes
        )
        &
        ref_edges[
            "gene2"
        ].isin(
            common_genes
        )
    ]
    .copy()
)


# Remove self-loops
ref_edges = (
    ref_edges[
        ref_edges[
            "gene1"
        ]
        !=
        ref_edges[
            "gene2"
        ]
    ]
    .copy()
)


# Canonical undirected edge IDs
edge_low = np.minimum(
    ref_edges[
        "gene1"
    ].astype(str).values,
    ref_edges[
        "gene2"
    ].astype(str).values
)


edge_high = np.maximum(
    ref_edges[
        "gene1"
    ].astype(str).values,
    ref_edges[
        "gene2"
    ].astype(str).values
)


ref_edges[
    "edge_key"
] = (
    edge_low
    +
    "||"
    +
    edge_high
)


duplicate_edges = int(
    ref_edges[
        "edge_key"
    ]
    .duplicated()
    .sum()
)


print(
    "Duplicate undirected edges:",
    duplicate_edges
)


if duplicate_edges > 0:

    ref_edges = (
        ref_edges
        .sort_values(
            "edge_key"
        )
        .drop_duplicates(
            subset="edge_key",
            keep="first"
        )
        .copy()
    )


ref_edges = (
    ref_edges
    .reset_index(
        drop=True
    )
)


print(
    "Edges before patient intersection:",
    before_overlap
)


print(
    "FINAL FROZEN STRING EDGE UNIVERSE:",
    len(
        ref_edges
    )
)


checkpoint(
    len(
        ref_edges
    )
    >
    50000,
    "substantial frozen STRING topology remains"
)


ref_edges.to_csv(
    FINAL_EDGE_UNIVERSE_OUT,
    index=False
)


# ======================================================================
# STEP 5 — BUILD FINAL COMMON EXPRESSION MATRICES
# ======================================================================

section(
    "STEP 5 — ALIGN EXPRESSION MATRICES"
)


network_genes = sorted(

    set(
        ref_edges[
            "gene1"
        ]
    )

    |

    set(
        ref_edges[
            "gene2"
        ]
    )
)


gtex = (
    gtex_expr.loc[
        network_genes
    ]
    .astype(
        np.float64
    )
)


patients = (
    patients_expr.loc[
        network_genes
    ]
    .astype(
        np.float64
    )
)


print(
    "Final GTEx network matrix:",
    gtex.shape
)


print(
    "Final patient network matrix:",
    patients.shape
)


checkpoint(
    list(
        gtex.index
    )
    ==
    list(
        patients.index
    ),
    "GTEx and patient gene order is identical"
)


checkpoint(
    np.isfinite(
        gtex.values
    ).all(),
    "GTEx matrix contains only finite values"
)


checkpoint(
    np.isfinite(
        patients.values
    ).all(),
    "patient matrix contains only finite values"
)


# ======================================================================
# STEP 6 — RECOMPUTE HEALTHY REFERENCE PCC EXACTLY
# ======================================================================

section(
    "STEP 6 — RECOMPUTE HEALTHY REFERENCE PCC"
)


gene_to_idx = {
    gene:
        i
    for i, gene
    in enumerate(
        network_genes
    )
}


edge_i = np.array(
    [
        gene_to_idx[
            gene
        ]
        for gene in ref_edges[
            "gene1"
        ]
    ],
    dtype=np.int32
)


edge_j = np.array(
    [
        gene_to_idx[
            gene
        ]
        for gene in ref_edges[
            "gene2"
        ]
    ],
    dtype=np.int32
)


X = (
    gtex.values
)


N = (
    X.shape[1]
)


checkpoint(
    N == 334,
    "reference network uses N=334"
)


mean_ref = (
    X.mean(
        axis=1
    )
)


centered = (
    X
    -
    mean_ref[
        :,
        None
    ]
)


M2_ref = np.sum(
    centered
    *
    centered,
    axis=1
)


checkpoint(
    (
        M2_ref
        >
        0
    ).all(),
    "no zero-variance genes remain"
)


# covariance numerator for each STRING edge
C_ref = np.sum(

    centered[
        edge_i
    ]

    *

    centered[
        edge_j
    ],

    axis=1
)


R_corr = (

    C_ref

    /

    np.sqrt(
        M2_ref[
            edge_i
        ]
        *
        M2_ref[
            edge_j
        ]
    )
)


checkpoint(
    np.isfinite(
        R_corr
    ).all(),
    "all reference PCC values are finite"
)


checkpoint(
    np.max(
        np.abs(
            R_corr
        )
    )
    <=
    1.000001,
    "all reference PCC values are within [-1,1]"
)


print(
    pd.Series(
        R_corr,
        name="reference_PCC"
    ).describe()
)


# ======================================================================
# STEP 7 — COMPUTE LIONESS sPCC FOR ALL 144 PATIENTS
# ======================================================================

section(
    "STEP 7 — COMPUTE 144 SINGLE-SAMPLE NETWORKS"
)


patient_ids = list(
    patients.columns
)


Y = (
    patients.values
)


n_patients = (
    Y.shape[1]
)


n_edges = (
    len(
        ref_edges
    )
)


# float32 is sufficient for storage;
# computations are performed in float64.
S_all = np.empty(
    (
        n_patients,
        n_edges
    ),
    dtype=np.float32
)


for patient_number in range(
    n_patients
):

    y = (
        Y[
            :,
            patient_number
        ]
    )


    # ----------------------------------------------------------
    # Exact online update of means/variances/covariances after
    # adding one patient to the N=334 controls.
    # ----------------------------------------------------------

    delta = (
        y
        -
        mean_ref
    )


    mean_new = (
        mean_ref
        +
        delta
        /
        (
            N
            +
            1
        )
    )


    M2_new = (
        M2_ref
        +
        delta
        *
        (
            y
            -
            mean_new
        )
    )


    # Welford covariance update
    C_new = (

        C_ref

        +

        delta[
            edge_i
        ]

        *

        (
            y[
                edge_j
            ]
            -
            mean_new[
                edge_j
            ]
        )
    )


    denominator = np.sqrt(

        M2_new[
            edge_i
        ]

        *

        M2_new[
            edge_j
        ]
    )


    PCC_N1 = (
        C_new
        /
        denominator
    )


    # LIONESS / single-sample PCC
    S_patient = (

        (
            N
            +
            1
        )
        *
        PCC_N1

        -

        N
        *
        R_corr
    )


    checkpoint(
        np.isfinite(
            S_patient
        ).all(),
        (
            f"{patient_ids[patient_number]} "
            "sPCC values are finite"
        )
    )


    S_all[
        patient_number
    ] = (
        S_patient.astype(
            np.float32
        )
    )


    if (
        (
            patient_number
            +
            1
        )
        %
        20
        ==
        0
        or
        (
            patient_number
            +
            1
        )
        ==
        n_patients
    ):

        print(
            f"  "
            f"{patient_number + 1}/"
            f"{n_patients} patients"
        )


checkpoint(
    S_all.shape
    ==
    (
        144,
        n_edges
    ),
    "sPCC matrix has 144 patients × frozen edges"
)


# ======================================================================
# STEP 8 — NUMERICALLY VERIFY LIONESS IMPLEMENTATION
# ======================================================================

section(
    "STEP 8 — BRUTE-FORCE LIONESS VALIDATION"
)


rng = np.random.default_rng(
    RANDOM_SEED
)


test_patient_idx = 0


random_edge_indices = rng.choice(
    n_edges,
    size=min(
        N_BRUTE_FORCE_EDGES,
        n_edges
    ),
    replace=False
)


max_abs_difference = 0.0


for eidx in random_edge_indices:

    i = edge_i[
        eidx
    ]

    j = edge_j[
        eidx
    ]


    x1 = (
        X[
            i
        ]
    )

    x2 = (
        X[
            j
        ]
    )


    y1 = (
        Y[
            i,
            test_patient_idx
        ]
    )

    y2 = (
        Y[
            j,
            test_patient_idx
        ]
    )


    pcc_n = np.corrcoef(
        x1,
        x2
    )[
        0,
        1
    ]


    pcc_n1 = np.corrcoef(

        np.append(
            x1,
            y1
        ),

        np.append(
            x2,
            y2
        )

    )[
        0,
        1
    ]


    brute = (

        (
            N
            +
            1
        )
        *
        pcc_n1

        -

        N
        *
        pcc_n
    )


    fast = float(
        S_all[
            test_patient_idx,
            eidx
        ]
    )


    difference = abs(
        brute
        -
        fast
    )


    max_abs_difference = max(
        max_abs_difference,
        difference
    )


print(
    "Maximum absolute difference "
    "against brute force:",
    max_abs_difference
)


# float32 storage introduces tiny numerical rounding.
checkpoint(
    max_abs_difference
    <
    1e-4,
    "fast LIONESS matches brute-force calculation"
)


# ======================================================================
# STEP 9 — POOLED Z-SCORE THRESHOLD FOR PATIENT NETWORKS
# ======================================================================

section(
    "STEP 9 — FIX PATIENT-NETWORK SPARSITY"
)


# ----------------------------------------------------------------------
# Published ssNetShift logic:
#
# Standardize single-sample edge weights using the pooled/global
# disease single-sample edge-weight distribution.
# ----------------------------------------------------------------------

pooled_mean = float(
    S_all.mean(
        dtype=np.float64
    )
)


pooled_sd = float(
    S_all.std(
        dtype=np.float64,
        ddof=1
    )
)


checkpoint(
    np.isfinite(
        pooled_mean
    ),
    "pooled sPCC mean is finite"
)


checkpoint(
    np.isfinite(
        pooled_sd
    )
    and
    pooled_sd
    >
    0,
    "pooled sPCC SD is positive"
)


Z_CRIT = float(
    norm.ppf(
        1
        -
        ALPHA_EDGE
        /
        2
    )
)


print(
    f"Global sPCC mean = "
    f"{pooled_mean:.6f}"
)


print(
    f"Global sPCC SD   = "
    f"{pooled_sd:.6f}"
)


print(
    f"alpha            = "
    f"{ALPHA_EDGE}"
)


print(
    f"|z| threshold    = "
    f"{Z_CRIT:.6f}"
)


# Boolean active-edge matrix.
# ~13 MB for 144 × ~92k.
patient_active = (

    np.abs(
        (
            S_all.astype(
                np.float64
            )
            -
            pooled_mean
        )
        /
        pooled_sd
    )

    >=

    Z_CRIT
)


patient_edge_counts = (
    patient_active.sum(
        axis=1
    )
)


print(
    "\nPatient retained-edge counts:"
)


print(
    pd.Series(
        patient_edge_counts
    ).describe()
)


checkpoint(
    np.all(
        patient_edge_counts
        >
        0
    ),
    "every patient has retained network edges"
)


# Average sparsity used for reference network
K_REFERENCE = int(
    np.rint(
        patient_edge_counts.mean()
    )
)


print(
    "\nAverage patient edge count:",
    patient_edge_counts.mean()
)


print(
    "Reference target edge count K:",
    K_REFERENCE
)


checkpoint(
    K_REFERENCE
    >
    0
    and
    K_REFERENCE
    <=
    n_edges,
    "valid reference sparsity target"
)


# ======================================================================
# STEP 10 — SPARSIFY REFERENCE TO SAME AVERAGE DENSITY
# ======================================================================

section(
    "STEP 10 — BUILD DENSITY-MATCHED HEALTHY REFERENCE R"
)


# Standardize reference edge weights across the frozen STRING universe.
reference_mean = float(
    R_corr.mean()
)


reference_sd = float(
    R_corr.std(
        ddof=1
    )
)


checkpoint(
    reference_sd
    >
    0,
    "reference PCC distribution has positive SD"
)


reference_z = (

    R_corr
    -
    reference_mean

) / reference_sd


# Rank by absolute z-score.
rank_order = np.argsort(
    -np.abs(
        reference_z
    )
)


reference_active = np.zeros(
    n_edges,
    dtype=bool
)


reference_active[
    rank_order[
        :
        K_REFERENCE
    ]
] = True


checkpoint(
    int(
        reference_active.sum()
    )
    ==
    K_REFERENCE,
    "reference contains exactly K retained edges"
)


reference_density = (
    K_REFERENCE
    /
    n_edges
)


patient_density_mean = float(
    patient_edge_counts.mean()
    /
    n_edges
)


print(
    f"Reference density: "
    f"{reference_density:.6f}"
)


print(
    f"Mean patient density: "
    f"{patient_density_mean:.6f}"
)


print(
    "Density difference:",
    abs(
        reference_density
        -
        patient_density_mean
    )
)


# Save final sparsified reference network
reference_network = (
    ref_edges.loc[
        reference_active
    ]
    .copy()
)


reference_network[
    "reference_PCC"
] = (
    R_corr[
        reference_active
    ]
)


reference_network[
    "reference_z"
] = (
    reference_z[
        reference_active
    ]
)


reference_network[
    "reference_sign"
] = np.where(

    reference_network[
        "reference_PCC"
    ]
    >
    0,

    1,

    -1
)


reference_network.to_csv(
    SPARSIFIED_REFERENCE_OUT,
    index=False
)


# ======================================================================
# STEP 11 — BUILD DEG-ANCHOR LIST ROBUSTLY
# ======================================================================

section(
    "STEP 11 — RESOLVE DEG ANCHORS"
)


checkpoint(
    "Is_DEG_Anchor"
    in nodes.columns,
    "Stage-B node table contains Is_DEG_Anchor"
)


anchor_mask = robust_bool(
    nodes[
        "Is_DEG_Anchor"
    ]
)


# ------------------------------------------------------------
# Prefer direct Ensembl column if available.
# ------------------------------------------------------------

ensembl_candidates = [
    column
    for column
    in nodes.columns
    if "ensembl"
    in column.lower()
]


if len(
    ensembl_candidates
) > 0:

    anchor_gene_col = (
        ensembl_candidates[0]
    )


    node_gene_ids = (
        nodes[
            anchor_gene_col
        ]
        .apply(
            strip_version
        )
    )


    print(
        "Using node Ensembl column:",
        anchor_gene_col
    )


else:

    # ----------------------------------------------------------
    # Fallback:
    # map STRING proteins -> Ensembl
    # ----------------------------------------------------------

    checkpoint(
        "string_protein_id"
        in nodes.columns,
        (
            "node table has STRING IDs "
            "for fallback mapping"
        )
    )


    mapping_ensembl_candidates = [
        column
        for column
        in mapping.columns
        if "ensembl"
        in column.lower()
    ]


    checkpoint(
        len(
            mapping_ensembl_candidates
        )
        >
        0,
        "full STRING mapping contains Ensembl IDs"
    )


    mapping_ensembl_col = (
        mapping_ensembl_candidates[0]
    )


    mapping[
        "ensembl_base"
    ] = (
        mapping[
            mapping_ensembl_col
        ]
        .apply(
            strip_version
        )
    )


    protein_to_gene = (
        mapping
        .dropna(
            subset=[
                "string_protein_id",
                "ensembl_base"
            ]
        )
        .sort_values(
            [
                "string_protein_id",
                "ensembl_base"
            ]
        )
        .drop_duplicates(
            subset="string_protein_id",
            keep="first"
        )
        .set_index(
            "string_protein_id"
        )[
            "ensembl_base"
        ]
        .to_dict()
    )


    node_gene_ids = (
        nodes[
            "string_protein_id"
        ]
        .map(
            protein_to_gene
        )
    )


nodes = (
    nodes.copy()
)


nodes[
    "_resolved_ensembl"
] = (
    node_gene_ids
)


deg_anchors_all = set(

    nodes.loc[
        anchor_mask,
        "_resolved_ensembl"
    ]
    .dropna()

)


deg_anchors = sorted(

    deg_anchors_all

    &

    set(
        network_genes
    )
)


print(
    "Original Stage-B DEG anchors:",
    len(
        deg_anchors_all
    )
)


print(
    "DEG anchors usable in final rewiring network:",
    len(
        deg_anchors
    )
)


checkpoint(
    len(
        deg_anchors
    )
    >
    3000,
    "substantial DEG-anchor feature set retained"
)


# ======================================================================
# STEP 12 — BUILD REFERENCE SIGNED NEIGHBOR SETS
# ======================================================================

section(
    "STEP 12 — BUILD REFERENCE SIGNED NEIGHBORHOODS"
)


ref_pos = {
    gene:
        set()
    for gene
    in deg_anchors
}


ref_neg = {
    gene:
        set()
    for gene
    in deg_anchors
}


# Need degree for ALL nodes to calculate global max |N_i^R|
reference_degree = {
    gene:
        0
    for gene
    in network_genes
}


active_reference_indices = np.flatnonzero(
    reference_active
)


for edge_index in active_reference_indices:

    g1 = ref_edges.at[
        edge_index,
        "gene1"
    ]

    g2 = ref_edges.at[
        edge_index,
        "gene2"
    ]


    r = R_corr[
        edge_index
    ]


    reference_degree[
        g1
    ] += 1

    reference_degree[
        g2
    ] += 1


    if g1 in ref_pos:

        if r > 0:

            ref_pos[
                g1
            ].add(
                g2
            )

        else:

            ref_neg[
                g1
            ].add(
                g2
            )


    if g2 in ref_pos:

        if r > 0:

            ref_pos[
                g2
            ].add(
                g1
            )

        else:

            ref_neg[
                g2
            ].add(
                g1
            )


MAX_REF_NEIGHBORHOOD = max(
    reference_degree.values()
)


print(
    "Maximum reference neighborhood size:",
    MAX_REF_NEIGHBORHOOD
)


checkpoint(
    MAX_REF_NEIGHBORHOOD
    >
    0,
    "reference maximum neighborhood is positive"
)


anchors_with_reference_neighbors = sum(

    (
        len(
            ref_pos[
                gene
            ]
        )
        +
        len(
            ref_neg[
                gene
            ]
        )
    )
    >
    0

    for gene
    in deg_anchors
)


print(
    "DEG anchors with >=1 reference neighbor:",
    anchors_with_reference_neighbors,
    "/",
    len(
        deg_anchors
    )
)


# ======================================================================
# STEP 13 — VERIFIED eNESH FUNCTION
# ======================================================================

def compute_enesh(
    gene,
    patient_positive,
    patient_negative
):
    """
    Published eNESH structure:

        eNESH = 1 - [ f - (g + h) ]

    where:

        f =
        retained same-sign neighbors
        --------------------------------
        union(reference, patient neighbors)

        g =
        lost reference same-sign neighbors
        -----------------------------------
        GLOBAL maximum reference degree

        h =
        lost reference same-sign neighbors
        -----------------------------------
        local union(reference, patient)

    A sign flip counts as loss of the corresponding signed
    reference relationship.
    """

    reference_positive = (
        ref_pos[
            gene
        ]
    )

    reference_negative = (
        ref_neg[
            gene
        ]
    )


    N_R = (

        reference_positive

        |

        reference_negative
    )


    N_S = (

        patient_positive

        |

        patient_negative
    )


    union_neighbors = (

        N_R

        |

        N_S
    )


    # Completely isolated in both networks
    if len(
        union_neighbors
    ) == 0:

        return 0.0


    retained_same_sign = (

        (
            reference_positive
            &
            patient_positive
        )

        |

        (
            reference_negative
            &
            patient_negative
        )
    )


    lost_reference_neighbors = (

        (
            reference_positive
            -
            patient_positive
        )

        |

        (
            reference_negative
            -
            patient_negative
        )
    )


    f = (

        len(
            retained_same_sign
        )

        /

        len(
            union_neighbors
        )
    )


    g = (

        len(
            lost_reference_neighbors
        )

        /

        MAX_REF_NEIGHBORHOOD
    )


    h = (

        len(
            lost_reference_neighbors
        )

        /

        len(
            union_neighbors
        )
    )


    score = (

        1.0

        -

        (
            f

            -

            (
                g
                +
                h
            )
        )
    )


    return float(
        score
    )


# ======================================================================
# STEP 14 — INTERNAL eNESH UNIT CHECKS
# ======================================================================

section(
    "STEP 14 — eNESH FORMULA SANITY CHECKS"
)


# Find one anchor that has reference neighbors
test_anchor = next(

    gene

    for gene
    in deg_anchors

    if (
        len(
            ref_pos[
                gene
            ]
        )
        +
        len(
            ref_neg[
                gene
            ]
        )
    )
    >
    0
)


# Exact same signed neighborhoods -> eNESH must equal 0
score_identical = compute_enesh(

    test_anchor,

    set(
        ref_pos[
            test_anchor
        ]
    ),

    set(
        ref_neg[
            test_anchor
        ]
    )
)


print(
    "Identical-neighborhood eNESH:",
    score_identical
)


checkpoint(
    abs(
        score_identical
    )
    <
    1e-12,
    "zero rewiring produces eNESH=0"
)


# ======================================================================
# STEP 15 — CALCULATE PERSONALIZED eNESH FOR ALL 144 PATIENTS
# ======================================================================

section(
    "STEP 15 — COMPUTE PERSONALIZED eNESH FOR 144 PATIENTS"
)


# For rapid anchor lookup
anchor_set = set(
    deg_anchors
)


rewiring = np.zeros(
    (
        n_patients,
        len(
            deg_anchors
        )
    ),
    dtype=np.float32
)


network_summary_rows = []


for p_idx, patient_id in enumerate(
    patient_ids
):

    active_indices = np.flatnonzero(
        patient_active[
            p_idx
        ]
    )


    # Only create neighborhood dictionaries for anchors,
    # not all ~13k nodes.
    patient_pos = {
        gene:
            set()
        for gene
        in deg_anchors
    }


    patient_neg = {
        gene:
            set()
        for gene
        in deg_anchors
    }


    spcc_patient = (
        S_all[
            p_idx
        ]
    )


    n_positive = 0
    n_negative = 0


    for edge_index in active_indices:

        g1 = ref_edges.at[
            edge_index,
            "gene1"
        ]

        g2 = ref_edges.at[
            edge_index,
            "gene2"
        ]


        edge_weight = float(
            spcc_patient[
                edge_index
            ]
        )


        if edge_weight > 0:

            n_positive += 1

            if g1 in anchor_set:

                patient_pos[
                    g1
                ].add(
                    g2
                )


            if g2 in anchor_set:

                patient_pos[
                    g2
                ].add(
                    g1
                )


        elif edge_weight < 0:

            n_negative += 1

            if g1 in anchor_set:

                patient_neg[
                    g1
                ].add(
                    g2
                )


            if g2 in anchor_set:

                patient_neg[
                    g2
                ].add(
                    g1
                )


    for anchor_idx, anchor in enumerate(
        deg_anchors
    ):

        rewiring[
            p_idx,
            anchor_idx
        ] = compute_enesh(

            anchor,

            patient_pos[
                anchor
            ],

            patient_neg[
                anchor
            ]
        )


    network_summary_rows.append(
        {
            "patient_id":
                patient_id,

            "n_retained_edges":
                len(
                    active_indices
                ),

            "n_positive_edges":
                n_positive,

            "n_negative_edges":
                n_negative,

            "network_density":
                len(
                    active_indices
                )
                /
                n_edges
        }
    )


    if (
        (
            p_idx
            +
            1
        )
        %
        10
        ==
        0
        or
        (
            p_idx
            +
            1
        )
        ==
        n_patients
    ):

        print(
            f"  "
            f"{p_idx + 1}/"
            f"{n_patients} patients complete"
        )


# ======================================================================
# STEP 16 — BUILD FINAL REWIRING MATRIX
# ======================================================================

section(
    "STEP 16 — FINAL REWIRING MATRIX CHECKS"
)


rewiring_df = pd.DataFrame(

    rewiring,

    index=patient_ids,

    columns=deg_anchors
)


rewiring_df.index.name = (
    "patient_id"
)


print(
    "Final personalized rewiring matrix:",
    rewiring_df.shape
)


checkpoint(
    rewiring_df.shape[
        0
    ]
    ==
    144,
    "144 patient rows"
)


checkpoint(
    rewiring_df.shape[
        1
    ]
    ==
    len(
        deg_anchors
    ),
    "one column per usable DEG anchor"
)


checkpoint(
    np.isfinite(
        rewiring_df.values
    ).all(),
    "all eNESH values are finite"
)


checkpoint(
    (
        rewiring_df.values
        >=
        -1e-8
    ).all(),
    "eNESH values are non-negative"
)


print(
    "\neNESH global distribution:"
)


print(
    pd.Series(
        rewiring_df.values.ravel()
    ).describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


exact_zero_fraction = float(

    np.mean(
        np.isclose(
            rewiring_df.values,
            0.0,
            atol=1e-12
        )
    )
)


print(
    f"\nExact-zero eNESH fraction: "
    f"{100*exact_zero_fraction:.2f}%"
)


# ======================================================================
# STEP 17 — ANCHOR-LEVEL SUMMARY
# ======================================================================

section(
    "STEP 17 — ANCHOR REWIRING SUMMARY"
)


anchor_summary = pd.DataFrame(
    {
        "ensembl_id":
            deg_anchors,

        "mean_eNESH":
            rewiring_df.mean(
                axis=0
            ).values,

        "median_eNESH":
            rewiring_df.median(
                axis=0
            ).values,

        "std_eNESH":
            rewiring_df.std(
                axis=0
            ).values,

        "max_eNESH":
            rewiring_df.max(
                axis=0
            ).values,

        "fraction_nonzero":
            (
                rewiring_df
                >
                0
            )
            .mean(
                axis=0
            )
            .values
    }
)


anchor_summary = (
    anchor_summary
    .sort_values(
        "mean_eNESH",
        ascending=False
    )
)


print(
    "Top 30 anchors by mean personalized rewiring:"
)


print(
    anchor_summary
    .head(
        30
    )
    .to_string(
        index=False
    )
)


# ======================================================================
# STEP 18 — SAVE EVERYTHING
# ======================================================================

section(
    "STEP 18 — SAVE STAGE C RESULTS"
)


rewiring_df.to_csv(
    REWIRING_MATRIX_OUT
)


anchor_summary.to_csv(
    ANCHOR_SUMMARY_OUT,
    index=False
)


network_summary_df = pd.DataFrame(
    network_summary_rows
)


network_summary_df.to_csv(
    PATIENT_NETWORK_SUMMARY_OUT,
    index=False
)


# Long-format version is useful for plotting/statistics
rewiring_long = (

    rewiring_df
    .reset_index()
    .melt(
        id_vars="patient_id",
        var_name="ensembl_id",
        value_name="eNESH"
    )
)


rewiring_long.to_csv(
    REWIRING_LONG_OUT,
    index=False
)


# ======================================================================
# STEP 19 — SAVE sPCC BINARY MATRIX
# ======================================================================

if SAVE_SPCC_NPZ:

    np.savez_compressed(
        SPCC_NPZ_OUT,

        spcc=S_all,

        patient_ids=np.array(
            patient_ids,
            dtype=object
        ),

        gene1=ref_edges[
            "gene1"
        ].values,

        gene2=ref_edges[
            "gene2"
        ].values,

        reference_pcc=R_corr,

        reference_active=reference_active
    )


    print(
        "Compressed sPCC saved:"
    )

    print(
        SPCC_NPZ_OUT
    )


# ======================================================================
# STEP 20 — SAVE AUDITABLE SPARSITY PARAMETERS
# ======================================================================

section(
    "STEP 20 — SAVE SPARSITY PARAMETERS"
)


sparsity_parameters = {

    "N_GTEx":
        int(
            N
        ),

    "N_patients":
        int(
            n_patients
        ),

    "N_frozen_STRING_edges":
        int(
            n_edges
        ),

    "alpha_edge":
        float(
            ALPHA_EDGE
        ),

    "z_threshold":
        float(
            Z_CRIT
        ),

    "pooled_sPCC_mean":
        float(
            pooled_mean
        ),

    "pooled_sPCC_sd":
        float(
            pooled_sd
        ),

    "mean_patient_retained_edges":
        float(
            patient_edge_counts.mean()
        ),

    "median_patient_retained_edges":
        float(
            np.median(
                patient_edge_counts
            )
        ),

    "reference_retained_edges":
        int(
            K_REFERENCE
        ),

    "mean_patient_density":
        float(
            patient_density_mean
        ),

    "reference_density":
        float(
            reference_density
        ),

    "reference_global_max_degree":
        int(
            MAX_REF_NEIGHBORHOOD
        ),

    "N_DEG_anchors":
        int(
            len(
                deg_anchors
            )
        )
}


with open(
    SPARSITY_PARAMETERS_OUT,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        sparsity_parameters,
        handle,
        indent=4
    )


print(
    json.dumps(
        sparsity_parameters,
        indent=4
    )
)


# ======================================================================
# STEP 21 — FINAL SUMMARY
# ======================================================================

section(
    "STEP 21 — FINAL STAGE-C SUMMARY"
)


summary = pd.DataFrame(
    [
        {
            "Metric":
                "Healthy GTEx controls",

            "Value":
                N
        },

        {
            "Metric":
                "GBM patients",

            "Value":
                n_patients
        },

        {
            "Metric":
                "Final network genes",

            "Value":
                len(
                    network_genes
                )
        },

        {
            "Metric":
                "Frozen STRING edges",

            "Value":
                n_edges
        },

        {
            "Metric":
                "Mean patient retained edges",

            "Value":
                float(
                    patient_edge_counts.mean()
                )
        },

        {
            "Metric":
                "Reference retained edges",

            "Value":
                K_REFERENCE
        },

        {
            "Metric":
                "DEG anchors",

            "Value":
                len(
                    deg_anchors
                )
        },

        {
            "Metric":
                "Rewiring matrix rows",

            "Value":
                rewiring_df.shape[
                    0
                ]
        },

        {
            "Metric":
                "Rewiring matrix columns",

            "Value":
                rewiring_df.shape[
                    1
                ]
        },

        {
            "Metric":
                "Mean eNESH",

            "Value":
                float(
                    rewiring_df.values.mean()
                )
        },

        {
            "Metric":
                "Median eNESH",

            "Value":
                float(
                    np.median(
                        rewiring_df.values
                    )
                )
        },

        {
            "Metric":
                "Maximum eNESH",

            "Value":
                float(
                    rewiring_df.values.max()
                )
        }
    ]
)


summary.to_csv(
    FINAL_SUMMARY_OUT,
    index=False
)


print(
    summary.to_string(
        index=False
    )
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "STAGE C PART 2 COMPLETE"
)


print(
    "PERSONALIZED NETWORK REWIRING HAS NOW BEEN COMPUTED."
)


print(
    "\nMain AI feature matrix:"
)

print(
    REWIRING_MATRIX_OUT
)


print(
    "\nShape:"
)

print(
    f"{rewiring_df.shape[0]} patients "
    f"x {rewiring_df.shape[1]} DEG-anchor eNESH features"
)


print(
    "\nIMPORTANT:"
)

print(
    "These are personalized eNESH rewiring features."
)


print(
    "They are NOT yet labelled as ssNetShift 'driver genes',"
)


print(
    "because driver classification additionally requires"
)


print(
    "background eNESH significance and positive delta betweenness."
)


print(
    "\nNEXT STEP:"
)


print(
    "STOP network construction."
)


print(
    "Move directly to survival AI:"
)


print(
    "M1 = clinical baseline"
)


print(
    "M2 = expression baseline"
)


print(
    "M3 = rewiring-only"
)


print(
    "M4 = clinical + personalized rewiring"
)


print(
    "with leakage-safe feature selection inside training folds."
)

### FAST REFERENCE-SPARSITY SENSITIVITY CHECK

In [ ]:
"""
FAST REFERENCE-SPARSITY SENSITIVITY CHECK
=========================================

NO LIONESS recomputation.

Purpose:
Check how reference-network sparsity affects coverage of the actual
3,992 DEG anchors.

This does NOT change the final method.
It is a sensitivity diagnostic to determine whether the current
4.4% density is causing excessive anchor isolation.
"""

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_C_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

STAGE_B_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
)

SPCC_CACHE = (
    STAGE_C_DIR
    / "STAGE_C_SINGLE_SAMPLE_SPCC_144.npz"
)

NODE_FILE = (
    STAGE_B_DIR
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)


# ======================================================================
# HELPERS
# ======================================================================

def section(title):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)


def strip_version(x):
    return str(x).split(".")[0].strip()


# ======================================================================
# STEP 1 — LOAD CACHE
# ======================================================================

section("STEP 1 — LOAD CACHED NETWORK DATA")

npz = np.load(
    SPCC_CACHE,
    allow_pickle=True
)

print("Cache keys:")
for key in npz.keys():
    arr = npz[key]
    print(
        f"  {key}: "
        f"shape={getattr(arr, 'shape', None)}, "
        f"dtype={getattr(arr, 'dtype', None)}"
    )


required = {
    "reference_pcc",
    "gene1",
    "gene2",
    "spcc",
    "patient_ids",
}

missing = required - set(npz.keys())

if missing:
    raise RuntimeError(
        f"Missing cache keys: {missing}"
    )


R_corr = np.asarray(
    npz["reference_pcc"],
    dtype=float
)

gene1 = np.asarray(
    npz["gene1"]
).astype(str)

gene2 = np.asarray(
    npz["gene2"]
).astype(str)

S_all = np.asarray(
    npz["spcc"]
)


print(
    "\nReference edges:",
    len(R_corr)
)

print(
    "sPCC matrix:",
    S_all.shape
)


# ======================================================================
# STEP 2 — LOAD THE ACTUAL DEG ANCHORS
# ======================================================================

section("STEP 2 — LOAD DEG ANCHORS")

nodes = pd.read_csv(
    NODE_FILE,
    low_memory=False
)


if "Is_DEG_Anchor" not in nodes.columns:
    raise RuntimeError(
        "Is_DEG_Anchor column missing."
    )


ensembl_candidates = [
    c
    for c in nodes.columns
    if "ensembl" in c.lower()
]


if not ensembl_candidates:
    raise RuntimeError(
        "No Ensembl column found in Stage-B nodes."
    )


ensembl_col = ensembl_candidates[0]


anchor_mask = (
    nodes["Is_DEG_Anchor"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)


deg_anchors = set(
    nodes.loc[
        anchor_mask,
        ensembl_col
    ]
    .dropna()
    .map(strip_version)
)


network_genes = set(
    gene1
) | set(
    gene2
)


deg_anchors = (
    deg_anchors
    &
    network_genes
)


print(
    "Usable DEG anchors:",
    len(deg_anchors)
)


# ======================================================================
# FUNCTION — MEASURE DEG-ANCHOR COVERAGE
# ======================================================================

def evaluate_reference_mask(
    active_mask,
    label
):

    degree = {
        gene: 0
        for gene in deg_anchors
    }


    active_indices = np.flatnonzero(
        active_mask
    )


    for idx in active_indices:

        g1 = gene1[idx]
        g2 = gene2[idx]

        if g1 in degree:
            degree[g1] += 1

        if g2 in degree:
            degree[g2] += 1


    degree_series = pd.Series(
        degree,
        dtype=float
    )


    n_with_neighbor = int(
        (
            degree_series > 0
        ).sum()
    )


    n_isolated = int(
        (
            degree_series == 0
        ).sum()
    )


    return {
        "scheme":
            label,

        "retained_edges":
            int(
                active_mask.sum()
            ),

        "density_pct":
            100
            *
            active_mask.mean(),

        "anchors_total":
            len(
                degree_series
            ),

        "anchors_with_neighbor":
            n_with_neighbor,

        "anchors_isolated":
            n_isolated,

        "anchor_coverage_pct":
            100
            *
            n_with_neighbor
            /
            len(
                degree_series
            ),

        "median_anchor_degree":
            degree_series.median(),

        "mean_anchor_degree":
            degree_series.mean(),

        "max_anchor_degree":
            degree_series.max(),
    }


# ======================================================================
# STEP 3 — DENSITY SENSITIVITY
# ======================================================================

section("STEP 3 — DEG-ANCHOR COVERAGE vs REFERENCE DENSITY")


results = []


# ------------------------------------------------------------
# Current Stage-C density
# ------------------------------------------------------------

densities = [
    0.04413,
    0.075,
    0.10,
    0.15,
    0.20,
]


rank_order = np.argsort(
    -np.abs(
        R_corr
    )
)


for density in densities:

    k = int(
        round(
            density
            *
            len(
                R_corr
            )
        )
    )

    active = np.zeros(
        len(
            R_corr
        ),
        dtype=bool
    )

    active[
        rank_order[:k]
    ] = True


    results.append(
        evaluate_reference_mask(
            active,
            f"top_{100*density:.2f}%"
        )
    )


# ======================================================================
# STEP 4 — ORDINARY PCC p<0.05 REFERENCE ONLY
# ======================================================================

section("STEP 4 — PCC p<0.05 REFERENCE-ONLY COMPARISON")


N_GTEX = 334

df = (
    N_GTEX
    -
    2
)

tcrit = stats.t.ppf(
    1 - 0.05 / 2,
    df
)


rcrit = (
    tcrit
    /
    np.sqrt(
        tcrit**2
        +
        df
    )
)


print(
    f"Reference PCC threshold: "
    f"|r| > {rcrit:.4f}"
)


active_significance = (
    np.abs(
        R_corr
    )
    >
    rcrit
)


results.append(
    evaluate_reference_mask(
        active_significance,
        "reference_PCC_p<0.05"
    )
)


# ======================================================================
# STEP 5 — RESULTS
# ======================================================================

section("STEP 5 — SENSITIVITY RESULTS")


results_df = pd.DataFrame(
    results
)


print(
    results_df.to_string(
        index=False
    )
)


OUT = (
    STAGE_C_DIR
    /
    "STAGE_C_REFERENCE_DENSITY_SENSITIVITY.csv"
)


results_df.to_csv(
    OUT,
    index=False
)


print(
    "\nSaved:"
)

print(
    OUT
)

### FINAL CORRECTED STAGE-C NETWORK THRESHOLD SENSITIVITY ANALYSIS

In [ ]:
"""
FINAL CORRECTED STAGE-C NETWORK THRESHOLD SENSITIVITY ANALYSIS
==============================================================

PURPOSE
-------
We already computed the expensive personalized single-sample networks
(LIONESS / sPCC) for all 144 GBM patients.

This script does NOT recompute LIONESS.

It reuses:
    STAGE_C_SINGLE_SAMPLE_SPCC_144.npz

and compares several candidate network sparsification strategies
without touching survival outcomes:

    1. ORIGINAL Stage-C cached method
       - original healthy reference mask
       - original pooled-z patient-specific masks

    2. top 10%
       - healthy network = top 10% |reference PCC|
       - EACH patient = its own top 10% |sPCC|

    3. top 15%
       - healthy network = top 15% |reference PCC|
       - EACH patient = its own top 15% |sPCC|

    4. top 20%
       - healthy network = top 20% |reference PCC|
       - EACH patient = its own top 20% |sPCC|

    5. FDR reference density
       - healthy network = PCC edges surviving BH-FDR < 0.05
       - EACH patient = its own top-K |sPCC|
         where K = number of FDR-surviving healthy edges

For every candidate we measure:

    - number / density of retained reference edges
    - DEG-anchor coverage
    - number of isolated DEG anchors
    - eNESH zero rate
    - percentage of anchors showing patient-to-patient variance
    - mean / median SD among varying anchors
    - patient network sizes

IMPORTANT
---------
NO survival times
NO event labels
NO C-index
NO outcome-driven threshold selection

The point is to freeze the network definition BEFORE Stage D survival AI.
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy import stats
from statsmodels.stats.multitest import multipletests


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_C_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

STAGE_B_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
)


SPCC_CACHE = (
    STAGE_C_DIR
    / "STAGE_C_SINGLE_SAMPLE_SPCC_144.npz"
)


DEG_ANCHOR_FILE = (
    STAGE_B_DIR
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)


OUTPUT_FILE = (
    STAGE_C_DIR
    / "STAGE_C_CORRECTED_THRESHOLD_SENSITIVITY.csv"
)


# Optional detailed anchor-level results
ANCHOR_DETAIL_FILE = (
    STAGE_C_DIR
    / "STAGE_C_CORRECTED_THRESHOLD_ANCHOR_DETAIL.csv"
)


# ======================================================================
# CONSTANTS
# ======================================================================

N_GTEX = 334

FDR_ALPHA = 0.05

ORIGINAL_Z_THRESHOLD = (
    1.959963984540054
)


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print(
        "\n"
        +
        "=" * 80
    )

    print(
        title
    )

    print(
        "=" * 80
    )


def checkpoint(
    condition,
    message
):

    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"[CHECKPOINT {status}] "
        f"{message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: "
            f"{message}"
        )


def strip_version(value):

    return (
        str(value)
        .split(".")[0]
        .strip()
    )


def robust_bool(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
                "y"
            ]
        )
    )


# ======================================================================
# TOP-K MASK
# ======================================================================

def make_top_k_mask(
    values,
    k
):
    """
    Retain the K strongest edges by absolute edge weight.

    Uses argpartition for speed.

    NOTE:
    This is patient-specific whenever 'values' is a patient's sPCC vector.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    n = len(
        values
    )

    k = int(
        k
    )


    if k <= 0:

        return np.zeros(
            n,
            dtype=bool
        )


    if k >= n:

        return np.ones(
            n,
            dtype=bool
        )


    mask = np.zeros(
        n,
        dtype=bool
    )


    indices = np.argpartition(
        -np.abs(
            values
        ),
        k - 1
    )[
        :k
    ]


    mask[
        indices
    ] = True


    return mask


# ======================================================================
# SIGNED NEIGHBOR BUILDER
# ======================================================================

def build_signed_neighbors(
    active_mask,
    edge_values,
    gene1,
    gene2,
    genes_of_interest
):
    """
    Build positive / negative neighborhoods.

    Only stores neighborhoods for genes_of_interest,
    which makes this much more memory efficient.
    """

    positive = {
        gene:
            set()
        for gene
        in genes_of_interest
    }


    negative = {
        gene:
            set()
        for gene
        in genes_of_interest
    }


    active_indices = np.flatnonzero(
        active_mask
    )


    for edge_index in (
        active_indices
    ):

        g1 = gene1[
            edge_index
        ]

        g2 = gene2[
            edge_index
        ]

        value = edge_values[
            edge_index
        ]


        if value > 0:

            if g1 in positive:

                positive[
                    g1
                ].add(
                    g2
                )


            if g2 in positive:

                positive[
                    g2
                ].add(
                    g1
                )


        elif value < 0:

            if g1 in negative:

                negative[
                    g1
                ].add(
                    g2
                )


            if g2 in negative:

                negative[
                    g2
                ].add(
                    g1
                )


    return (
        positive,
        negative
    )


# ======================================================================
# eNESH
# ======================================================================

def compute_enesh(
    reference_positive,
    reference_negative,
    patient_positive,
    patient_negative,
    max_reference_neighborhood
):
    """
    eNESH = 1 - [f - (g + h)]

    f:
        retained same-sign neighbors /
        union(reference, patient neighbors)

    g:
        lost reference same-sign neighbors /
        GLOBAL maximum healthy-reference neighborhood size

    h:
        lost reference same-sign neighbors /
        local union(reference, patient neighbors)
    """

    reference_neighbors = (

        reference_positive

        |

        reference_negative
    )


    patient_neighbors = (

        patient_positive

        |

        patient_negative
    )


    union_neighbors = (

        reference_neighbors

        |

        patient_neighbors
    )


    # No neighborhood in either network
    if len(
        union_neighbors
    ) == 0:

        return 0.0


    retained_same_sign = (

        (
            reference_positive
            &
            patient_positive
        )

        |

        (
            reference_negative
            &
            patient_negative
        )
    )


    lost_reference = (

        (
            reference_positive
            -
            patient_positive
        )

        |

        (
            reference_negative
            -
            patient_negative
        )
    )


    f = (

        len(
            retained_same_sign
        )

        /

        len(
            union_neighbors
        )
    )


    g = (

        len(
            lost_reference
        )

        /

        max_reference_neighborhood
    )


    h = (

        len(
            lost_reference
        )

        /

        len(
            union_neighbors
        )
    )


    return float(

        1.0

        -

        (
            f

            -

            (
                g
                +
                h
            )
        )
    )


# ======================================================================
# STEP 1 — LOAD CACHE
# ======================================================================

section(
    "STEP 1 — LOAD CACHED SINGLE-SAMPLE NETWORKS"
)


checkpoint(
    SPCC_CACHE.exists(),
    "sPCC cache exists"
)


npz = np.load(
    SPCC_CACHE,
    allow_pickle=True
)


print(
    "Cache keys:"
)


for key in (
    npz.keys()
):

    array = (
        npz[
            key
        ]
    )

    print(
        f"  {key}: "
        f"shape="
        f"{getattr(array, 'shape', None)}, "
        f"dtype="
        f"{getattr(array, 'dtype', None)}"
    )


required_keys = {

    "reference_pcc",
    "reference_active",
    "spcc",
    "gene1",
    "gene2",
    "patient_ids"
}


missing_keys = (

    required_keys

    -

    set(
        npz.keys()
    )
)


checkpoint(
    len(
        missing_keys
    )
    ==
    0,
    (
        "all expected cache keys exist"
        if not missing_keys
        else
        f"missing keys: {missing_keys}"
    )
)


R_corr = np.asarray(
    npz[
        "reference_pcc"
    ],
    dtype=np.float64
)


original_reference_active = np.asarray(
    npz[
        "reference_active"
    ]
).astype(
    bool
)


S_all = np.asarray(
    npz[
        "spcc"
    ]
)


gene1 = np.asarray(
    npz[
        "gene1"
    ]
).astype(
    str
)


gene2 = np.asarray(
    npz[
        "gene2"
    ]
).astype(
    str
)


patient_ids = np.asarray(
    npz[
        "patient_ids"
    ]
).astype(
    str
)


n_edges = len(
    R_corr
)


# normalize orientation to:
# patients × edges
if (
    S_all.shape[0]
    ==
    n_edges
):

    S_all = (
        S_all.T
    )


print(
    "\nReference edges:",
    n_edges
)


print(
    "sPCC matrix:",
    S_all.shape
)


checkpoint(
    S_all.shape[0]
    ==
    144,
    "sPCC contains 144 patients"
)


checkpoint(
    S_all.shape[1]
    ==
    n_edges,
    "sPCC edge dimension matches reference"
)


checkpoint(
    len(
        gene1
    )
    ==
    n_edges,
    "gene1 edge count matches"
)


checkpoint(
    len(
        gene2
    )
    ==
    n_edges,
    "gene2 edge count matches"
)


checkpoint(
    np.isfinite(
        R_corr
    ).all(),
    "reference PCC values are finite"
)


checkpoint(
    np.isfinite(
        S_all
    ).all(),
    "single-sample PCC values are finite"
)


# ======================================================================
# STEP 2 — RECONCILE ORIGINAL REFERENCE MASK
# ======================================================================

section(
    "STEP 2 — RECONCILE ORIGINAL REFERENCE MASK"
)


original_k = int(
    original_reference_active.sum()
)


original_density = (

    original_k

    /

    n_edges
)


print(
    "Original retained reference edges:",
    original_k
)


print(
    f"Original density: "
    f"{100*original_density:.4f}%"
)


reconstructed_original = (
    make_top_k_mask(
        R_corr,
        original_k
    )
)


agreement = np.mean(

    original_reference_active

    ==

    reconstructed_original
)


only_original = int(

    np.sum(

        original_reference_active

        &

        ~reconstructed_original
    )
)


only_reconstructed = int(

    np.sum(

        ~original_reference_active

        &

        reconstructed_original
    )
)


print(
    f"Agreement with simple top-K |reference PCC|: "
    f"{100*agreement:.4f}%"
)


print(
    "Edges only in cached original:",
    only_original
)


print(
    "Edges only in reconstructed mask:",
    only_reconstructed
)


if (
    only_original
    >
    0
    or
    only_reconstructed
    >
    0
):

    print(
        "\nNOTE:"
    )

    print(
        "The cached original reference mask is not identical "
        "to simple top-K |PCC|."
    )

    print(
        "The ORIGINAL candidate below will therefore use "
        "the actual cached mask, not the reconstructed one."
    )


# ======================================================================
# STEP 3 — LOAD DEG ANCHORS
# ======================================================================

section(
    "STEP 3 — LOAD DEG ANCHORS"
)


checkpoint(
    DEG_ANCHOR_FILE.exists(),
    "Stage-B node file exists"
)


nodes = pd.read_csv(
    DEG_ANCHOR_FILE,
    low_memory=False
)


checkpoint(
    "Is_DEG_Anchor"
    in nodes.columns,
    "Stage-B nodes contain Is_DEG_Anchor"
)


ensembl_candidates = [
    column
    for column
    in nodes.columns
    if "ensembl"
    in column.lower()
]


checkpoint(
    len(
        ensembl_candidates
    )
    >
    0,
    "Stage-B node table contains an Ensembl column"
)


ensembl_col = (
    ensembl_candidates[0]
)


print(
    "Ensembl column:",
    ensembl_col
)


anchor_mask = (
    robust_bool(
        nodes[
            "Is_DEG_Anchor"
        ]
    )
)


deg_anchors_all = set(

    nodes.loc[
        anchor_mask,
        ensembl_col
    ]
    .dropna()
    .map(
        strip_version
    )
)


network_genes = (

    set(
        gene1
    )

    |

    set(
        gene2
    )
)


deg_anchors = sorted(

    deg_anchors_all

    &

    network_genes
)


print(
    "Stage-B DEG anchors:",
    len(
        deg_anchors_all
    )
)


print(
    "Usable DEG anchors in cached edge universe:",
    len(
        deg_anchors
    )
)


checkpoint(
    len(
        deg_anchors
    )
    >
    3000,
    "substantial DEG-anchor set retained"
)


anchor_set = set(
    deg_anchors
)


all_genes = sorted(
    network_genes
)


# ======================================================================
# STEP 4 — RECONSTRUCT ORIGINAL PATIENT MASKS
# ======================================================================

section(
    "STEP 4 — RECONSTRUCT ORIGINAL PATIENT-SPECIFIC MASKS"
)


# This reproduces the original Stage-C patient network rule:
#
# pooled single-sample edge distribution
# -> global mean / SD
# -> retain |z| >= 1.95996

pooled_mean = float(
    S_all.mean(
        dtype=np.float64
    )
)


pooled_sd = float(
    S_all.std(
        dtype=np.float64,
        ddof=1
    )
)


checkpoint(
    np.isfinite(
        pooled_mean
    ),
    "pooled sPCC mean is finite"
)


checkpoint(
    np.isfinite(
        pooled_sd
    )
    and
    pooled_sd
    >
    0,
    "pooled sPCC SD is positive"
)


print(
    f"Pooled sPCC mean: "
    f"{pooled_mean:.6f}"
)


print(
    f"Pooled sPCC SD:   "
    f"{pooled_sd:.6f}"
)


print(
    f"Original |z| threshold: "
    f"{ORIGINAL_Z_THRESHOLD:.6f}"
)


original_patient_masks = (

    np.abs(

        (
            S_all.astype(
                np.float64
            )

            -

            pooled_mean
        )

        /

        pooled_sd
    )

    >=

    ORIGINAL_Z_THRESHOLD
)


original_patient_edge_counts = (

    original_patient_masks.sum(
        axis=1
    )
)


print(
    "\nOriginal patient network sizes:"
)


print(
    pd.Series(
        original_patient_edge_counts
    ).describe()
)


print(
    "Mean:",
    float(
        original_patient_edge_counts.mean()
    )
)


# ======================================================================
# STEP 5 — CALCULATE HEALTHY PCC p-VALUES + BH FDR
# ======================================================================

section(
    "STEP 5 — CALCULATE HEALTHY PCC BH-FDR"
)


df_freedom = (
    N_GTEX
    -
    2
)


# Avoid division warning when r is extremely close to ±1.
r_for_test = np.clip(
    R_corr,
    -0.999999999,
    0.999999999
)


t_statistic = (

    r_for_test

    *

    np.sqrt(

        df_freedom

        /

        (
            1
            -
            r_for_test**2
        )
    )
)


p_values = (

    2

    *

    stats.t.sf(
        np.abs(
            t_statistic
        ),
        df_freedom
    )
)


checkpoint(
    np.isfinite(
        p_values
    ).all(),
    "all healthy reference PCC p-values are finite"
)


fdr_reject, fdr_q, _, _ = (
    multipletests(
        p_values,
        alpha=FDR_ALPHA,
        method="fdr_bh"
    )
)


fdr_k = int(
    fdr_reject.sum()
)


fdr_density = (

    fdr_k

    /

    n_edges
)


print(
    "BH-FDR < 0.05 reference edges:",
    fdr_k
)


print(
    f"FDR reference density: "
    f"{100*fdr_density:.2f}%"
)


checkpoint(
    fdr_k
    >
    0,
    "at least one reference edge survives BH-FDR"
)


# Optional diagnostic:
# weakest absolute correlation surviving FDR
if fdr_k > 0:

    minimum_fdr_abs_r = float(
        np.min(
            np.abs(
                R_corr[
                    fdr_reject
                ]
            )
        )
    )

    print(
        f"Minimum |r| among FDR-surviving edges: "
        f"{minimum_fdr_abs_r:.6f}"
    )


# ======================================================================
# STEP 6 — DEFINE CANDIDATE NETWORK RULES
# ======================================================================

section(
    "STEP 6 — DEFINE CANDIDATE NETWORKS"
)


candidate_specs = []


# ----------------------------------------------------------------------
# ORIGINAL
# ----------------------------------------------------------------------

candidate_specs.append(
    {
        "name":
            "original_cached",

        "reference_mask":
            original_reference_active,

        "patient_rule":
            "original_z",

        "k":
            original_k
    }
)


# ----------------------------------------------------------------------
# Fixed equal-density top-K candidates
# ----------------------------------------------------------------------

for percentage in [
    0.10,
    0.15,
    0.20
]:

    k = int(
        round(
            percentage
            *
            n_edges
        )
    )


    reference_mask = (
        make_top_k_mask(
            R_corr,
            k
        )
    )


    candidate_specs.append(
        {
            "name":
                f"top_{int(percentage*100)}pct",

            "reference_mask":
                reference_mask,

            "patient_rule":
                "top_k",

            "k":
                k
        }
    )


# ----------------------------------------------------------------------
# FDR reference
#
# Reference:
# exactly the edges that survive BH-FDR.
#
# Patient:
# each patient gets THEIR OWN top-K absolute sPCC edges,
# where K equals the number of FDR reference edges.
# ----------------------------------------------------------------------

candidate_specs.append(
    {
        "name":
            "FDR_reference_density",

        "reference_mask":
            fdr_reject.astype(
                bool
            ),

        "patient_rule":
            "top_k",

        "k":
            fdr_k
    }
)


for spec in (
    candidate_specs
):

    print(
        f"{spec['name']}:"
    )

    print(
        f"  reference edges = "
        f"{int(spec['reference_mask'].sum())}"
    )

    print(
        f"  reference density = "
        f"{100*spec['reference_mask'].mean():.2f}%"
    )

    print(
        f"  patient rule = "
        f"{spec['patient_rule']}"
    )


# ======================================================================
# STEP 7 — EVALUATE CANDIDATES
# ======================================================================

section(
    "STEP 7 — STRUCTURAL / eNESH SENSITIVITY ANALYSIS"
)


summary_rows = []

anchor_detail_rows = []


for candidate_index, spec in enumerate(
    candidate_specs,
    start=1
):

    candidate_name = (
        spec[
            "name"
        ]
    )


    reference_mask = (
        spec[
            "reference_mask"
        ]
    )


    k = int(
        spec[
            "k"
        ]
    )


    print(
        "\n"
        +
        "-" * 80
    )

    print(
        f"[{candidate_index}/"
        f"{len(candidate_specs)}] "
        f"{candidate_name}"
    )

    print(
        "-" * 80
    )


    reference_edge_count = int(
        reference_mask.sum()
    )


    print(
        "Reference edges:",
        reference_edge_count
    )


    print(
        f"Reference density: "
        f"{100*reference_mask.mean():.2f}%"
    )


    # ==================================================================
    # HEALTHY REFERENCE SIGNED NEIGHBORHOODS FOR DEG ANCHORS
    # ==================================================================

    reference_positive, reference_negative = (
        build_signed_neighbors(
            active_mask=
                reference_mask,

            edge_values=
                R_corr,

            gene1=
                gene1,

            gene2=
                gene2,

            genes_of_interest=
                anchor_set
        )
    )


    # ==================================================================
    # FULL HEALTHY DEGREE — needed for global g denominator
    # ==================================================================

    reference_degree = {
        gene:
            0
        for gene
        in all_genes
    }


    reference_active_indices = (
        np.flatnonzero(
            reference_mask
        )
    )


    for edge_index in (
        reference_active_indices
    ):

        g1 = (
            gene1[
                edge_index
            ]
        )

        g2 = (
            gene2[
                edge_index
            ]
        )


        reference_degree[
            g1
        ] += 1


        reference_degree[
            g2
        ] += 1


    max_reference_neighborhood = max(
        max(
            reference_degree.values()
        ),
        1
    )


    print(
        "Maximum healthy reference degree:",
        max_reference_neighborhood
    )


    # ==================================================================
    # DEG-ANCHOR COVERAGE
    # ==================================================================

    anchor_reference_degrees = np.array(
        [
            reference_degree.get(
                anchor,
                0
            )
            for anchor
            in deg_anchors
        ],
        dtype=int
    )


    anchors_with_neighbor = int(
        np.sum(
            anchor_reference_degrees
            >
            0
        )
    )


    isolated_anchors = int(
        np.sum(
            anchor_reference_degrees
            ==
            0
        )
    )


    anchor_coverage_pct = (

        100

        *

        anchors_with_neighbor

        /

        len(
            deg_anchors
        )
    )


    print(
        "DEG anchors with >=1 healthy neighbor:",
        anchors_with_neighbor,
        "/",
        len(
            deg_anchors
        )
    )


    print(
        f"Anchor coverage: "
        f"{anchor_coverage_pct:.2f}%"
    )


    # ==================================================================
    # PERSONALIZED eNESH
    # ==================================================================

    eNESH_matrix = np.zeros(
        (
            S_all.shape[0],
            len(
                deg_anchors
            )
        ),
        dtype=np.float32
    )


    patient_edge_counts = []


    for patient_index in range(
        S_all.shape[0]
    ):

        patient_spcc = (
            S_all[
                patient_index
            ]
        )


        # --------------------------------------------------------------
        # PATIENT-SPECIFIC EDGE MASK
        # --------------------------------------------------------------

        if (
            spec[
                "patient_rule"
            ]
            ==
            "original_z"
        ):

            patient_mask = (
                original_patient_masks[
                    patient_index
                ]
            )


        elif (
            spec[
                "patient_rule"
            ]
            ==
            "top_k"
        ):

            patient_mask = (
                make_top_k_mask(
                    patient_spcc,
                    k
                )
            )


        else:

            raise RuntimeError(
                "Unknown patient rule: "
                f"{spec['patient_rule']}"
            )


        patient_edge_count = int(
            patient_mask.sum()
        )


        patient_edge_counts.append(
            patient_edge_count
        )


        patient_positive, patient_negative = (
            build_signed_neighbors(
                active_mask=
                    patient_mask,

                edge_values=
                    patient_spcc,

                gene1=
                    gene1,

                gene2=
                    gene2,

                genes_of_interest=
                    anchor_set
            )
        )


        for anchor_index, anchor in enumerate(
            deg_anchors
        ):

            eNESH_matrix[
                patient_index,
                anchor_index
            ] = compute_enesh(

                reference_positive[
                    anchor
                ],

                reference_negative[
                    anchor
                ],

                patient_positive[
                    anchor
                ],

                patient_negative[
                    anchor
                ],

                max_reference_neighborhood
            )


    # ==================================================================
    # eNESH QUALITY / INFORMATION METRICS
    # ==================================================================

    checkpoint(
        np.isfinite(
            eNESH_matrix
        ).all(),
        (
            f"{candidate_name}: "
            "all eNESH values finite"
        )
    )


    checkpoint(
        (
            eNESH_matrix
            >=
            -1e-8
        ).all(),
        (
            f"{candidate_name}: "
            "eNESH non-negative"
        )
    )


    exact_zero_rate = float(

        100

        *

        np.mean(

            np.isclose(
                eNESH_matrix,
                0.0,
                atol=1e-12
            )
        )
    )


    # Patient-to-patient feature variance
    per_anchor_variance = np.var(
        eNESH_matrix,
        axis=0,
        ddof=1
    )


    varying_mask = (
        per_anchor_variance
        >
        1e-10
    )


    n_varying_anchors = int(
        varying_mask.sum()
    )


    pct_anchors_varying = (

        100

        *

        n_varying_anchors

        /

        len(
            deg_anchors
        )
    )


    per_anchor_sd = np.sqrt(
        per_anchor_variance
    )


    if (
        varying_mask.any()
    ):

        mean_sd_varying = float(
            np.mean(
                per_anchor_sd[
                    varying_mask
                ]
            )
        )


        median_sd_varying = float(
            np.median(
                per_anchor_sd[
                    varying_mask
                ]
            )
        )


    else:

        mean_sd_varying = 0.0
        median_sd_varying = 0.0


    # ==================================================================
    # NONZERO RATE BY ANCHOR
    # ==================================================================

    anchor_nonzero_fraction = np.mean(
        eNESH_matrix
        >
        0,
        axis=0
    )


    # ==================================================================
    # PATIENT EDGE COUNTS
    # ==================================================================

    patient_edge_counts = np.asarray(
        patient_edge_counts,
        dtype=int
    )


    # ==================================================================
    # PRINT
    # ==================================================================

    print(
        f"eNESH zero rate: "
        f"{exact_zero_rate:.2f}%"
    )


    print(
        "Anchors with patient-to-patient variance:",
        n_varying_anchors,
        "/",
        len(
            deg_anchors
        )
    )


    print(
        f"Percent anchors varying: "
        f"{pct_anchors_varying:.2f}%"
    )


    print(
        f"Mean SD among varying anchors: "
        f"{mean_sd_varying:.4f}"
    )


    print(
        f"Median SD among varying anchors: "
        f"{median_sd_varying:.4f}"
    )


    print(
        f"Patient retained edges: "
        f"mean={patient_edge_counts.mean():.1f}, "
        f"median={np.median(patient_edge_counts):.1f}, "
        f"min={patient_edge_counts.min()}, "
        f"max={patient_edge_counts.max()}"
    )


    # ==================================================================
    # SUMMARY
    # ==================================================================

    summary_rows.append(
        {
            "candidate":
                candidate_name,

            "reference_active_edges":
                reference_edge_count,

            "reference_density_pct":
                100
                *
                reference_mask.mean(),

            "patient_rule":
                spec[
                    "patient_rule"
                ],

            "mean_patient_active_edges":
                float(
                    patient_edge_counts.mean()
                ),

            "median_patient_active_edges":
                float(
                    np.median(
                        patient_edge_counts
                    )
                ),

            "min_patient_active_edges":
                int(
                    patient_edge_counts.min()
                ),

            "max_patient_active_edges":
                int(
                    patient_edge_counts.max()
                ),

            "anchors_total":
                len(
                    deg_anchors
                ),

            "anchors_with_reference_neighbor":
                anchors_with_neighbor,

            "anchors_isolated":
                isolated_anchors,

            "anchor_coverage_pct":
                anchor_coverage_pct,

            "median_reference_anchor_degree":
                float(
                    np.median(
                        anchor_reference_degrees
                    )
                ),

            "mean_reference_anchor_degree":
                float(
                    anchor_reference_degrees.mean()
                ),

            "max_reference_degree":
                int(
                    max_reference_neighborhood
                ),

            "eNESH_zero_rate_pct":
                exact_zero_rate,

            "n_anchors_with_variance":
                n_varying_anchors,

            "pct_anchors_with_variance":
                pct_anchors_varying,

            "mean_SD_varying_anchors":
                mean_sd_varying,

            "median_SD_varying_anchors":
                median_sd_varying,

            "mean_eNESH":
                float(
                    eNESH_matrix.mean()
                ),

            "median_eNESH":
                float(
                    np.median(
                        eNESH_matrix
                    )
                ),

            "max_eNESH":
                float(
                    eNESH_matrix.max()
                )
        }
    )


    # ==================================================================
    # ANCHOR-LEVEL DETAIL
    # ==================================================================

    for anchor_index, anchor in enumerate(
        deg_anchors
    ):

        anchor_detail_rows.append(
            {
                "candidate":
                    candidate_name,

                "ensembl_id":
                    anchor,

                "reference_degree":
                    int(
                        anchor_reference_degrees[
                            anchor_index
                        ]
                    ),

                "mean_eNESH":
                    float(
                        np.mean(
                            eNESH_matrix[
                                :,
                                anchor_index
                            ]
                        )
                    ),

                "median_eNESH":
                    float(
                        np.median(
                            eNESH_matrix[
                                :,
                                anchor_index
                            ]
                        )
                    ),

                "sd_eNESH":
                    float(
                        per_anchor_sd[
                            anchor_index
                        ]
                    ),

                "fraction_nonzero":
                    float(
                        anchor_nonzero_fraction[
                            anchor_index
                        ]
                    )
            }
        )


# ======================================================================
# STEP 8 — FINAL COMPARISON
# ======================================================================

section(
    "STEP 8 — FINAL CORRECTED COMPARISON"
)


summary_df = pd.DataFrame(
    summary_rows
)


pd.set_option(
    "display.max_columns",
    None
)


print(
    summary_df.to_string(
        index=False
    )
)


# ======================================================================
# STEP 9 — SAVE RESULTS
# ======================================================================

section(
    "STEP 9 — SAVE RESULTS"
)


summary_df.to_csv(
    OUTPUT_FILE,
    index=False
)


anchor_detail_df = pd.DataFrame(
    anchor_detail_rows
)


anchor_detail_df.to_csv(
    ANCHOR_DETAIL_FILE,
    index=False
)


print(
    "Summary saved:"
)

print(
    OUTPUT_FILE
)


print(
    "\nAnchor-level details saved:"
)

print(
    ANCHOR_DETAIL_FILE
)


# ======================================================================
# STEP 10 — INTERPRETATION GUIDE
# ======================================================================

section(
    "STEP 10 — HOW TO CHOOSE THE FINAL NETWORK"
)


print(
    """
DO NOT choose using survival performance.

Look at these columns:

1. anchor_coverage_pct
   Higher is better, but 100% is NOT required.

2. eNESH_zero_rate_pct
   Extremely high values mean most patient×gene features contain
   no rewiring information.

3. pct_anchors_with_variance
   This is crucial for survival ML.
   A feature that never varies across patients cannot predict survival.

4. median_SD_varying_anchors
   We want meaningful patient-to-patient variation, not merely many
   technically non-zero features.

5. network density
   Avoid choosing an unnecessarily huge network merely to maximize
   anchor coverage.

Recommended logic:

- Original 4.4% is the baseline.
- 10–20% are conservative sensitivity alternatives.
- FDR_reference_density is scientifically attractive because the
  reference density is defined from statistically supported healthy
  correlations rather than an arbitrary percentage.

BUT:
The final choice must balance biological/statistical support against
network informativeness.

Freeze the chosen rule BEFORE Stage D.
Never revisit it after seeing survival C-index results.
"""
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "SENSITIVITY ANALYSIS COMPLETE"
)


print(
    "No survival outcomes were used."
)


print(
    "No LIONESS recomputation was performed."
)


print(
    "The cached 144 × 90,959 personalized sPCC matrix was reused."
)


print(
    "\nNext:"
)


print(
    "Inspect STEP 8 only."
)


print(
    "Then freeze one network definition, regenerate the final eNESH "
    "matrix from the cache, and move permanently to Stage D survival AI."
)

### STAGE C — FINAL FROZEN PERSONALIZED REWIRING MATRIX

In [ ]:
"""
STAGE C — FINAL FROZEN PERSONALIZED REWIRING MATRIX
===================================================

FINAL NETWORK RULE — FROZEN BEFORE SURVIVAL ANALYSIS
-----------------------------------------------------

Reference network:
    Top 20% of the 90,959 frozen STRING-constrained edges
    ranked by |reference PCC|.

Patient network p:
    EACH patient's own top 20% of the same frozen edge universe,
    ranked independently by |sPCC_p|.

Therefore:
    - number of edges is identical across patients
    - edge IDENTITIES are patient-specific
    - topology can gain/lose/change relative to healthy reference
    - personalized rewiring is preserved

The expensive LIONESS / sPCC calculation is NOT repeated.
The validated cache is reused.

eNESH:
    Same already-verified formula.

NO survival labels are loaded or used.

EXPECTED values from the already-completed sensitivity analysis:
    Frozen STRING edges             = 90,959
    Top-20% retained edges          = 18,192
    Usable DEG anchors              = 3,992
    Anchor reference coverage       ≈ 65.93%
    eNESH zero rate                 ≈ 19.13%
    Anchors with cross-patient var. ≈ 92.51%

If these are not reproduced approximately, STOP before Stage D.
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_C_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

STAGE_B_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
)

SPCC_CACHE = (
    STAGE_C_DIR
    / "STAGE_C_SINGLE_SAMPLE_SPCC_144.npz"
)

DEG_ANCHOR_FILE = (
    STAGE_B_DIR
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)


# ======================================================================
# OUTPUTS
# ======================================================================

FINAL_REWIRING_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv"
)

FINAL_REFERENCE_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_REFERENCE_EDGES_20PCT.csv"
)

FINAL_ANCHOR_SUMMARY_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_REWIRING_ANCHOR_SUMMARY_20PCT.csv"
)

FINAL_PATIENT_SUMMARY_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_PATIENT_NETWORK_SUMMARY_20PCT.csv"
)

FINAL_RULE_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_FROZEN_NETWORK_RULE_20PCT.json"
)

FINAL_SUMMARY_OUT = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_REWIRING_SUMMARY_20PCT.csv"
)


# ======================================================================
# FROZEN SETTINGS
# ======================================================================

FROZEN_DENSITY = 0.20

EXPECTED_PATIENTS = 144

EXPECTED_EDGES = 90959

EXPECTED_ANCHORS = 3992

EXPECTED_RETAINED_EDGES = 18192


# Expected values from the already-run sensitivity analysis.
# These are VALIDATION CHECKPOINTS, not newly chosen thresholds.

EXPECTED_COVERAGE_PCT = 65.931864
EXPECTED_ZERO_RATE_PCT = 19.129057
EXPECTED_VARYING_PCT = 92.510020


# Allow tiny differences from floating-point/tie handling.
COVERAGE_TOLERANCE = 0.10
ZERO_RATE_TOLERANCE = 0.20
VARYING_TOLERANCE = 0.20


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(condition, message):

    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: {message}"
        )


def strip_version(value):

    return (
        str(value)
        .split(".")[0]
        .strip()
    )


def robust_bool(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
                "y"
            ]
        )
    )


# ======================================================================
# FAST EXACT TOP-K MASK
# ======================================================================

def top_k_mask(values, k):
    """
    Retain exactly K strongest edges by |edge weight|.

    argpartition is much faster than sorting all ~91k edges.

    The previous sensitivity code effectively used the same top-K
    definition. No outcome information enters this operation.
    """

    values = np.asarray(
        values,
        dtype=np.float64
    )

    n = len(values)

    k = int(k)

    if k <= 0:

        return np.zeros(
            n,
            dtype=bool
        )

    if k >= n:

        return np.ones(
            n,
            dtype=bool
        )

    mask = np.zeros(
        n,
        dtype=bool
    )

    selected = np.argpartition(
        -np.abs(values),
        k - 1
    )[:k]

    mask[selected] = True

    return mask


# ======================================================================
# SIGNED NEIGHBORHOODS
# ======================================================================

def build_anchor_signed_neighbors(
    active_mask,
    edge_values,
    gene1,
    gene2,
    anchor_set
):
    """
    Build positive/negative neighborhoods only for DEG anchors.

    Non-anchor genes can still be neighbors; we simply avoid allocating
    dictionaries for thousands of genes whose eNESH is never calculated.
    """

    positive = {
        gene: set()
        for gene in anchor_set
    }

    negative = {
        gene: set()
        for gene in anchor_set
    }

    active_indices = np.flatnonzero(
        active_mask
    )

    for edge_index in active_indices:

        g1 = gene1[edge_index]
        g2 = gene2[edge_index]

        value = edge_values[edge_index]

        if value > 0:

            if g1 in anchor_set:
                positive[g1].add(g2)

            if g2 in anchor_set:
                positive[g2].add(g1)

        elif value < 0:

            if g1 in anchor_set:
                negative[g1].add(g2)

            if g2 in anchor_set:
                negative[g2].add(g1)

    return (
        positive,
        negative
    )


# ======================================================================
# eNESH
# ======================================================================

def compute_enesh(
    ref_pos,
    ref_neg,
    pat_pos,
    pat_neg,
    max_ref_neighborhood
):
    """
    Verified eNESH structure:

        eNESH = 1 - [f - (g + h)]

    f:
        same-sign reference relationships retained in the patient
        ----------------------------------------------------------
        union of reference + patient neighborhoods

    g:
        reference signed relationships lost in patient
        ----------------------------------------------
        GLOBAL maximum reference neighborhood size

    h:
        reference signed relationships lost in patient
        ----------------------------------------------
        local union of reference + patient neighborhoods
    """

    N_R = (
        ref_pos
        |
        ref_neg
    )

    N_S = (
        pat_pos
        |
        pat_neg
    )

    union = (
        N_R
        |
        N_S
    )

    if len(union) == 0:

        return 0.0

    retained = (
        (ref_pos & pat_pos)
        |
        (ref_neg & pat_neg)
    )

    lost = (
        (ref_pos - pat_pos)
        |
        (ref_neg - pat_neg)
    )

    f = (
        len(retained)
        /
        len(union)
    )

    g = (
        len(lost)
        /
        max_ref_neighborhood
    )

    h = (
        len(lost)
        /
        len(union)
    )

    score = (
        1.0
        -
        (
            f
            -
            (
                g
                +
                h
            )
        )
    )

    return float(score)


# ======================================================================
# STEP 1 — LOAD VALIDATED sPCC CACHE
# ======================================================================

section(
    "STEP 1 — LOAD VALIDATED 144-PATIENT sPCC CACHE"
)

checkpoint(
    SPCC_CACHE.exists(),
    "sPCC cache exists"
)

npz = np.load(
    SPCC_CACHE,
    allow_pickle=True
)

print(
    "Cache keys:"
)

for key in npz.keys():

    array = npz[key]

    print(
        f"  {key}: "
        f"shape={getattr(array, 'shape', None)}, "
        f"dtype={getattr(array, 'dtype', None)}"
    )


required_keys = {
    "spcc",
    "patient_ids",
    "gene1",
    "gene2",
    "reference_pcc"
}

missing_keys = (
    required_keys
    -
    set(npz.keys())
)

checkpoint(
    len(missing_keys) == 0,
    (
        "all required cache keys exist"
        if not missing_keys
        else f"missing cache keys: {missing_keys}"
    )
)


S_all = np.asarray(
    npz["spcc"]
)

patient_ids = np.asarray(
    npz["patient_ids"]
).astype(str)

gene1 = np.asarray(
    npz["gene1"]
).astype(str)

gene2 = np.asarray(
    npz["gene2"]
).astype(str)

R_corr = np.asarray(
    npz["reference_pcc"],
    dtype=np.float64
)


n_edges = len(
    R_corr
)


# Robust orientation handling:
# final required shape = patients × edges

if (
    S_all.shape[0] == n_edges
    and
    S_all.shape[1] == EXPECTED_PATIENTS
):

    print(
        "Transposing cached sPCC matrix "
        "to patients × edges..."
    )

    S_all = S_all.T


print(
    "\nFinal sPCC shape:",
    S_all.shape
)


checkpoint(
    S_all.shape
    ==
    (
        EXPECTED_PATIENTS,
        n_edges
    ),
    (
        f"sPCC shape is "
        f"{EXPECTED_PATIENTS} patients × {n_edges} edges"
    )
)


checkpoint(
    n_edges
    ==
    EXPECTED_EDGES,
    (
        f"frozen edge universe contains expected "
        f"{EXPECTED_EDGES:,} edges"
    )
)


checkpoint(
    len(patient_ids)
    ==
    EXPECTED_PATIENTS,
    "144 patient IDs available"
)


checkpoint(
    len(set(patient_ids))
    ==
    EXPECTED_PATIENTS,
    "144 patient IDs are unique"
)


checkpoint(
    len(gene1)
    ==
    n_edges
    and
    len(gene2)
    ==
    n_edges,
    "edge endpoint arrays match edge universe"
)


checkpoint(
    np.isfinite(
        R_corr
    ).all(),
    "reference PCC values are finite"
)


checkpoint(
    np.isfinite(
        S_all
    ).all(),
    "all cached patient sPCC values are finite"
)


# ======================================================================
# STEP 2 — FREEZE TOP-20% REFERENCE NETWORK
# ======================================================================

section(
    "STEP 2 — FREEZE HEALTHY REFERENCE NETWORK AT TOP 20%"
)


k_reference = int(
    round(
        n_edges
        *
        FROZEN_DENSITY
    )
)


print(
    "Calculated retained edge count:",
    k_reference
)


checkpoint(
    k_reference
    ==
    EXPECTED_RETAINED_EDGES,
    (
        f"top-20% corresponds to expected "
        f"{EXPECTED_RETAINED_EDGES:,} edges"
    )
)


reference_mask = top_k_mask(
    R_corr,
    k_reference
)


checkpoint(
    int(
        reference_mask.sum()
    )
    ==
    k_reference,
    "reference mask contains exactly target K edges"
)


print(
    f"Reference edges retained: "
    f"{reference_mask.sum():,}/"
    f"{n_edges:,}"
)


print(
    f"Reference density: "
    f"{100*reference_mask.mean():.4f}%"
)


# ======================================================================
# STEP 3 — LOAD DEG ANCHORS ROBUSTLY
# ======================================================================

section(
    "STEP 3 — LOAD FINAL DEG ANCHORS"
)


checkpoint(
    DEG_ANCHOR_FILE.exists(),
    "Stage-B node file exists"
)


nodes = pd.read_csv(
    DEG_ANCHOR_FILE,
    low_memory=False
)


checkpoint(
    "Is_DEG_Anchor"
    in nodes.columns,
    "Is_DEG_Anchor column exists"
)


ensembl_candidates = [
    column
    for column
    in nodes.columns
    if "ensembl"
    in column.lower()
]


checkpoint(
    len(ensembl_candidates)
    >
    0,
    "Stage-B node table contains an Ensembl column"
)


anchor_col = (
    ensembl_candidates[0]
)


print(
    "Anchor Ensembl column:",
    anchor_col
)


anchor_mask = robust_bool(
    nodes[
        "Is_DEG_Anchor"
    ]
)


all_network_genes = (
    set(gene1)
    |
    set(gene2)
)


deg_anchors_all = set(
    nodes.loc[
        anchor_mask,
        anchor_col
    ]
    .dropna()
    .map(
        strip_version
    )
)


deg_anchors = sorted(
    deg_anchors_all
    &
    all_network_genes
)


print(
    "Original Stage-B DEG anchors:",
    len(
        deg_anchors_all
    )
)


print(
    "Usable final DEG anchors:",
    len(
        deg_anchors
    )
)


checkpoint(
    len(deg_anchors)
    ==
    EXPECTED_ANCHORS,
    (
        f"expected {EXPECTED_ANCHORS:,} usable DEG anchors"
    )
)


anchor_set = set(
    deg_anchors
)


# ======================================================================
# STEP 4 — BUILD FINAL HEALTHY REFERENCE NEIGHBORHOODS
# ======================================================================

section(
    "STEP 4 — BUILD FINAL HEALTHY SIGNED NEIGHBORHOODS"
)


ref_pos, ref_neg = (
    build_anchor_signed_neighbors(
        active_mask=
            reference_mask,

        edge_values=
            R_corr,

        gene1=
            gene1,

        gene2=
            gene2,

        anchor_set=
            anchor_set
    )
)


# Need degree for every network gene because g uses
# GLOBAL maximum reference neighborhood size.

reference_degree = {
    gene: 0
    for gene
    in all_network_genes
}


for edge_index in np.flatnonzero(
    reference_mask
):

    g1 = gene1[
        edge_index
    ]

    g2 = gene2[
        edge_index
    ]

    reference_degree[
        g1
    ] += 1

    reference_degree[
        g2
    ] += 1


MAX_REF_NEIGHBORHOOD = max(
    reference_degree.values()
)


checkpoint(
    MAX_REF_NEIGHBORHOOD
    >
    0,
    "global maximum reference degree is positive"
)


print(
    "Maximum reference degree:",
    MAX_REF_NEIGHBORHOOD
)


anchor_reference_degrees = np.array(
    [
        reference_degree.get(
            anchor,
            0
        )
        for anchor
        in deg_anchors
    ]
)


anchors_with_neighbor = int(
    np.sum(
        anchor_reference_degrees
        >
        0
    )
)


anchors_isolated = (
    len(deg_anchors)
    -
    anchors_with_neighbor
)


coverage_pct = (
    100
    *
    anchors_with_neighbor
    /
    len(deg_anchors)
)


print(
    f"DEG anchors with >=1 reference neighbor: "
    f"{anchors_with_neighbor:,}/"
    f"{len(deg_anchors):,}"
)


print(
    f"Reference anchor coverage: "
    f"{coverage_pct:.4f}%"
)


print(
    "Isolated DEG anchors:",
    anchors_isolated
)


checkpoint(
    abs(
        coverage_pct
        -
        EXPECTED_COVERAGE_PCT
    )
    <=
    COVERAGE_TOLERANCE,
    (
        "top-20% anchor coverage reproduces "
        "the validated sensitivity analysis"
    )
)


# ======================================================================
# STEP 5 — SAVE FINAL FROZEN REFERENCE EDGES
# ======================================================================

section(
    "STEP 5 — SAVE FINAL FROZEN REFERENCE NETWORK"
)


reference_df = pd.DataFrame(
    {
        "gene1":
            gene1[
                reference_mask
            ],

        "gene2":
            gene2[
                reference_mask
            ],

        "reference_PCC":
            R_corr[
                reference_mask
            ],

        "reference_abs_PCC":
            np.abs(
                R_corr[
                    reference_mask
                ]
            ),

        "reference_sign":
            np.where(
                R_corr[
                    reference_mask
                ]
                >
                0,
                1,
                -1
            )
    }
)


reference_df.to_csv(
    FINAL_REFERENCE_OUT,
    index=False
)


print(
    "Saved final reference network:"
)

print(
    FINAL_REFERENCE_OUT
)


print(
    "Reference edges saved:",
    len(reference_df)
)


# ======================================================================
# STEP 6 — COMPUTE EACH PATIENT'S OWN TOP-20% NETWORK + eNESH
# ======================================================================

section(
    "STEP 6 — COMPUTE FINAL 144 PERSONALIZED NETWORKS + eNESH"
)


k_patient = (
    k_reference
)


rewiring = np.zeros(
    (
        EXPECTED_PATIENTS,
        len(deg_anchors)
    ),
    dtype=np.float32
)


patient_summary_rows = []


for patient_index, patient_id in enumerate(
    patient_ids
):

    patient_spcc = (
        S_all[
            patient_index
        ]
    )


    # CRITICAL:
    # each patient independently ranks THEIR OWN sPCC vector.

    patient_mask = top_k_mask(
        patient_spcc,
        k_patient
    )


    checkpoint(
        int(
            patient_mask.sum()
        )
        ==
        k_patient,
        (
            f"{patient_id}: "
            "exactly 18,192 personalized edges"
        )
    )


    pat_pos, pat_neg = (
        build_anchor_signed_neighbors(
            active_mask=
                patient_mask,

            edge_values=
                patient_spcc,

            gene1=
                gene1,

            gene2=
                gene2,

            anchor_set=
                anchor_set
        )
    )


    n_positive = int(
        np.sum(
            patient_spcc[
                patient_mask
            ]
            >
            0
        )
    )


    n_negative = int(
        np.sum(
            patient_spcc[
                patient_mask
            ]
            <
            0
        )
    )


    # Reference vs patient edge-overlap diagnostic.
    edge_overlap = int(
        np.sum(
            reference_mask
            &
            patient_mask
        )
    )


    edge_jaccard = (
        edge_overlap
        /
        (
            int(
                reference_mask.sum()
            )
            +
            int(
                patient_mask.sum()
            )
            -
            edge_overlap
        )
    )


    for anchor_index, anchor in enumerate(
        deg_anchors
    ):

        rewiring[
            patient_index,
            anchor_index
        ] = compute_enesh(

            ref_pos[
                anchor
            ],

            ref_neg[
                anchor
            ],

            pat_pos[
                anchor
            ],

            pat_neg[
                anchor
            ],

            MAX_REF_NEIGHBORHOOD
        )


    patient_summary_rows.append(
        {
            "patient_id":
                patient_id,

            "retained_edges":
                int(
                    patient_mask.sum()
                ),

            "positive_edges":
                n_positive,

            "negative_edges":
                n_negative,

            "reference_edge_overlap":
                edge_overlap,

            "reference_patient_edge_jaccard":
                edge_jaccard
        }
    )


    if (
        (patient_index + 1) % 20 == 0
        or
        (patient_index + 1)
        ==
        EXPECTED_PATIENTS
    ):

        print(
            f"  "
            f"{patient_index + 1}/"
            f"{EXPECTED_PATIENTS} patients complete"
        )


# ======================================================================
# STEP 7 — BUILD FINAL MATRIX
# ======================================================================

section(
    "STEP 7 — VALIDATE FINAL eNESH MATRIX"
)


rewiring_matrix = pd.DataFrame(
    rewiring,
    index=patient_ids,
    columns=deg_anchors
)


rewiring_matrix.index.name = (
    "patient_id"
)


print(
    "Final rewiring matrix:",
    rewiring_matrix.shape
)


checkpoint(
    rewiring_matrix.shape
    ==
    (
        EXPECTED_PATIENTS,
        EXPECTED_ANCHORS
    ),
    (
        f"final matrix = "
        f"{EXPECTED_PATIENTS} × {EXPECTED_ANCHORS}"
    )
)


checkpoint(
    not rewiring_matrix.isna().any().any(),
    "final matrix contains no NaN values"
)


checkpoint(
    np.isfinite(
        rewiring_matrix.values
    ).all(),
    "all final eNESH values are finite"
)


checkpoint(
    (
        rewiring_matrix.values
        >=
        -1e-8
    ).all(),
    "all final eNESH values are non-negative"
)


# ======================================================================
# STEP 8 — REPRODUCE VALIDATED TOP-20% RESULTS
# ======================================================================

section(
    "STEP 8 — REPRODUCE TOP-20% SENSITIVITY RESULTS"
)


zero_rate = float(
    100
    *
    np.mean(
        np.isclose(
            rewiring_matrix.values,
            0.0,
            atol=1e-12
        )
    )
)


feature_variance = (
    rewiring_matrix.var(
        axis=0,
        ddof=1
    )
)


varying_mask = (
    feature_variance
    >
    1e-10
)


n_varying = int(
    varying_mask.sum()
)


pct_varying = float(
    100
    *
    n_varying
    /
    len(deg_anchors)
)


varying_sd = np.sqrt(
    feature_variance[
        varying_mask
    ]
)


mean_sd_varying = float(
    varying_sd.mean()
)


median_sd_varying = float(
    varying_sd.median()
)


print(
    f"eNESH zero rate: "
    f"{zero_rate:.4f}%"
)


print(
    f"Anchors with cross-patient variance: "
    f"{n_varying:,}/"
    f"{len(deg_anchors):,} "
    f"({pct_varying:.4f}%)"
)


print(
    f"Mean SD among varying anchors: "
    f"{mean_sd_varying:.6f}"
)


print(
    f"Median SD among varying anchors: "
    f"{median_sd_varying:.6f}"
)


checkpoint(
    abs(
        zero_rate
        -
        EXPECTED_ZERO_RATE_PCT
    )
    <=
    ZERO_RATE_TOLERANCE,
    (
        "zero rate reproduces validated "
        "top-20% sensitivity result"
    )
)


checkpoint(
    abs(
        pct_varying
        -
        EXPECTED_VARYING_PCT
    )
    <=
    VARYING_TOLERANCE,
    (
        "varying-feature percentage reproduces "
        "validated top-20% result"
    )
)


# ======================================================================
# STEP 9 — DISTRIBUTION
# ======================================================================

section(
    "STEP 9 — FINAL eNESH DISTRIBUTION"
)


global_values = (
    rewiring_matrix
    .values
    .ravel()
)


print(
    pd.Series(
        global_values,
        name="eNESH"
    ).describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# ======================================================================
# STEP 10 — ANCHOR SUMMARY
# ======================================================================

section(
    "STEP 10 — BUILD FINAL DEG-ANCHOR SUMMARY"
)


anchor_summary = pd.DataFrame(
    {
        "ensembl_id":
            deg_anchors,

        "reference_degree":
            anchor_reference_degrees,

        "has_reference_neighbor":
            anchor_reference_degrees
            >
            0,

        "mean_eNESH":
            rewiring_matrix.mean(
                axis=0
            ).values,

        "median_eNESH":
            rewiring_matrix.median(
                axis=0
            ).values,

        "sd_eNESH":
            rewiring_matrix.std(
                axis=0,
                ddof=1
            ).values,

        "min_eNESH":
            rewiring_matrix.min(
                axis=0
            ).values,

        "max_eNESH":
            rewiring_matrix.max(
                axis=0
            ).values,

        "fraction_nonzero":
            (
                rewiring_matrix
                >
                0
            )
            .mean(
                axis=0
            )
            .values
    }
)


anchor_summary = (
    anchor_summary
    .sort_values(
        [
            "sd_eNESH",
            "mean_eNESH"
        ],
        ascending=[
            False,
            False
        ]
    )
)


print(
    "Top 20 anchors by cross-patient eNESH SD:"
)


print(
    anchor_summary
    .head(20)
    .to_string(
        index=False
    )
)


# ======================================================================
# STEP 11 — SAVE FINAL FILES
# ======================================================================

section(
    "STEP 11 — SAVE FINAL FROZEN STAGE-C FILES"
)


rewiring_matrix.to_csv(
    FINAL_REWIRING_OUT
)


anchor_summary.to_csv(
    FINAL_ANCHOR_SUMMARY_OUT,
    index=False
)


patient_summary_df = pd.DataFrame(
    patient_summary_rows
)


patient_summary_df.to_csv(
    FINAL_PATIENT_SUMMARY_OUT,
    index=False
)


print(
    "FINAL AI FEATURE MATRIX:"
)

print(
    FINAL_REWIRING_OUT
)


print(
    "\nAnchor summary:"
)

print(
    FINAL_ANCHOR_SUMMARY_OUT
)


print(
    "\nPatient network summary:"
)

print(
    FINAL_PATIENT_SUMMARY_OUT
)


# ======================================================================
# STEP 12 — SAVE FROZEN METHOD RULE
# ======================================================================

section(
    "STEP 12 — SAVE FROZEN NETWORK DEFINITION"
)


frozen_rule = {
    "edge_universe":
        int(
            n_edges
        ),

    "reference_rule":
        "top 20% of frozen STRING-constrained edges by absolute reference PCC",

    "patient_rule":
        "each patient independently selects top 20% of frozen edge universe by absolute sPCC",

    "frozen_density":
        float(
            FROZEN_DENSITY
        ),

    "edges_per_network":
        int(
            k_reference
        ),

    "n_patients":
        int(
            EXPECTED_PATIENTS
        ),

    "n_DEG_anchors":
        int(
            len(deg_anchors)
        ),

    "anchor_reference_coverage_pct":
        float(
            coverage_pct
        ),

    "eNESH_zero_rate_pct":
        float(
            zero_rate
        ),

    "anchors_with_variance_pct":
        float(
            pct_varying
        ),

    "survival_outcomes_used_for_network_selection":
        False,

    "threshold_selection_status":
        "FROZEN BEFORE SURVIVAL MODELLING"
}


with open(
    FINAL_RULE_OUT,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        frozen_rule,
        handle,
        indent=4
    )


print(
    json.dumps(
        frozen_rule,
        indent=4
    )
)


# ======================================================================
# STEP 13 — FINAL SUMMARY
# ======================================================================

section(
    "STEP 13 — FINAL STAGE-C SUMMARY"
)


summary_df = pd.DataFrame(
    [
        {
            "Metric":
                "Patients",

            "Value":
                EXPECTED_PATIENTS
        },

        {
            "Metric":
                "Frozen STRING edge universe",

            "Value":
                n_edges
        },

        {
            "Metric":
                "Frozen density %",

            "Value":
                100
                *
                FROZEN_DENSITY
        },

        {
            "Metric":
                "Reference retained edges",

            "Value":
                k_reference
        },

        {
            "Metric":
                "Edges per patient network",

            "Value":
                k_patient
        },

        {
            "Metric":
                "Usable DEG anchors",

            "Value":
                len(
                    deg_anchors
                )
        },

        {
            "Metric":
                "Reference anchor coverage %",

            "Value":
                coverage_pct
        },

        {
            "Metric":
                "eNESH zero rate %",

            "Value":
                zero_rate
        },

        {
            "Metric":
                "Anchors with variance %",

            "Value":
                pct_varying
        },

        {
            "Metric":
                "Mean SD varying anchors",

            "Value":
                mean_sd_varying
        },

        {
            "Metric":
                "Median SD varying anchors",

            "Value":
                median_sd_varying
        }
    ]
)


summary_df.to_csv(
    FINAL_SUMMARY_OUT,
    index=False
)


print(
    summary_df.to_string(
        index=False
    )
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "STAGE C PERMANENTLY FINALIZED"
)


print(
    "FINAL PERSONALIZED REWIRING MATRIX CREATED."
)


print(
    "\nShape:"
)

print(
    f"{rewiring_matrix.shape[0]} patients "
    f"x {rewiring_matrix.shape[1]} DEG-anchor features"
)


print(
    "\nFrozen network rule:"
)

print(
    "Reference = top 20% |healthy PCC|"
)

print(
    "Patient p = that patient's own top 20% |sPCC_p|"
)


print(
    "\nNo survival outcome was used to select this rule."
)


print(
    "\nMAIN FILE FOR STAGE D:"
)

print(
    FINAL_REWIRING_OUT
)


print(
    "\nNEXT STEP:"
)

print(
    "Stage D — survival AI."
)

print(
    "No further STRING / GTEx / LIONESS / threshold optimization."
)